# DataMate Agent ? Kaggle notebook t? source hi?n t?i
Notebook n?y nh?ng to?n b? `src/` v? `prompts/` t?i th?i ?i?m t?o notebook, sau ?? ch?y Agent b?ng vLLM TP=2. ??y l? notebook m?i, ??c l?p v?i notebook Kaggle c?.


In [ ]:
%pip install -q vllm==0.32.0


In [ ]:
import torch
print("CUDA:", torch.version.cuda, "GPUs:", torch.cuda.device_count())
assert torch.cuda.device_count() >= 2, "Kaggle session must provide at least 2 GPUs"


In [ ]:
from pathlib import Path
import base64
import os
import sys

SOURCE_ROOT = Path('/kaggle/working/datamate_from_src')
source_files = {}
source_files['src/__init__.py'] = 'IyBEYXRhTWF0ZSAtIFZpZXRuYW1lc2UgTmF0dXJhbCBMYW5ndWFnZSBRdWVzdGlvbiBBbnN3ZXJpbmcgb3ZlciBUYWJ1bGFyIERhdGENCg=='
source_files['src/evaluate.py'] = 'IiIiDQpldmFsdWF0ZS5weSDigJQgTG9jYWwgUEEgU2NvcmVyIGZvciBEYXRhTWF0ZQ0KDQpDb21wdXRlcyBQZW5hbGl6ZWQgQWNjdXJhY3kgKFBBKSBvbiB0aGUgdHJhaW5pbmcgc2V0IGJ5IGNvbXBhcmluZw0Kc3VibWlzc2lvbi5qc29ubCBhZ2FpbnN0IGFuc3dlcnMuanNvbmwuDQoNClVzYWdlOg0KICAgIHB5dGhvbiBldmFsdWF0ZS5weSAtLXN1Ym1pc3Npb24gc3VibWlzc2lvbi5qc29ubCAtLWFuc3dlcnMgZGF0YS90cmFpbi9hbnN3ZXJzLmpzb25sDQoiIiINCg0KaW1wb3J0IGFyZ3BhcnNlDQppbXBvcnQganNvbg0KaW1wb3J0IG1hdGgNCmltcG9ydCByZQ0KaW1wb3J0IHN5cw0KaW1wb3J0IHVuaWNvZGVkYXRhDQpmcm9tIHR5cGluZyBpbXBvcnQgQW55LCBEaWN0LCBMaXN0LCBPcHRpb25hbCwgVHVwbGUNCg0KDQojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQ0KIyBOb3JtYWxpemF0aW9uIChtYXRjaGVzIGNvbXBldGl0aW9uIHJ1bGVzKQ0KIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0NCg0KZGVmIG5vcm1hbGl6ZV9zdHJpbmcoczogc3RyKSAtPiBzdHI6DQogICAgIiIiTm9ybWFsaXplIGEgc3RyaW5nIGZvciBjb21wYXJpc29uIHBlciBjb21wZXRpdGlvbiBydWxlcy4iIiINCiAgICBzID0gdW5pY29kZWRhdGEubm9ybWFsaXplKCJORkMiLCBzdHIocykpDQogICAgcyA9IHMubG93ZXIoKQ0KICAgIHMgPSByZS5zdWIociJbXlx3XHNdIiwgIiAiLCBzLCBmbGFncz1yZS5VTklDT0RFKQ0KICAgIHMgPSBzLnN0cmlwKCkNCiAgICBzID0gcmUuc3ViKHIiXHMrIiwgIiAiLCBzKQ0KICAgIHJldHVybiBzDQoNCg0KIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0NCiMgUGVyLXR5cGUgbWF0Y2hpbmcNCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tDQoNCmRlZiBtYXRjaF9udW1iZXIocHJlZDogQW55LCByZWY6IEFueSkgLT4gYm9vbDoNCiAgICAiIiJNYXRjaCBudW1lcmljIGFuc3dlcnMuIFJlZmVyZW5jZSBpcyBpbnRlZ2VyIG9yIHJvdW5kZWQgdG8gMiBkZWNpbWFscy4iIiINCiAgICB0cnk6DQogICAgICAgIHAgPSBmbG9hdChwcmVkKQ0KICAgICAgICByID0gZmxvYXQocmVmKQ0KICAgICAgICAjIEFsbG93IHNtYWxsIGZsb2F0aW5nLXBvaW50IHRvbGVyYW5jZQ0KICAgICAgICByZXR1cm4gYWJzKHAgLSByKSA8IDAuMDA1DQogICAgZXhjZXB0IChWYWx1ZUVycm9yLCBUeXBlRXJyb3IpOg0KICAgICAgICByZXR1cm4gRmFsc2UNCg0KDQpkZWYgbWF0Y2hfcGVyY2VudChwcmVkOiBBbnksIHJlZjogQW55KSAtPiBib29sOg0KICAgICIiIk1hdGNoIHBlcmNlbnRhZ2UgYW5zd2Vycy4gRXhhY3QgYWZ0ZXIgcm91bmRpbmcgdG8gMiBkZWNpbWFscy4iIiINCiAgICB0cnk6DQogICAgICAgIHAgPSByb3VuZChmbG9hdChwcmVkKSwgMikNCiAgICAgICAgciA9IHJvdW5kKGZsb2F0KHJlZiksIDIpDQogICAgICAgIHJldHVybiBhYnMocCAtIHIpIDwgMC4wMDUNCiAgICBleGNlcHQgKFZhbHVlRXJyb3IsIFR5cGVFcnJvcik6DQogICAgICAgIHJldHVybiBGYWxzZQ0KDQoNCmRlZiBtYXRjaF9zdHJpbmcocHJlZDogQW55LCByZWY6IEFueSwgYWxpYXNlczogT3B0aW9uYWxbTGlzdFtzdHJdXSA9IE5vbmUpIC0+IGJvb2w6DQogICAgIiIiDQogICAgTWF0Y2ggc3RyaW5nIGFuc3dlcnMgYWZ0ZXIgbm9ybWFsaXphdGlvbi4NCiAgICBDaGVja3MgYWdhaW5zdCB0aGUgcmVmZXJlbmNlIGFuZCBhbnkgYWxpYXNlcy4NCiAgICAiIiINCiAgICBwcmVkX25vcm0gPSBub3JtYWxpemVfc3RyaW5nKHN0cihwcmVkKSkNCg0KICAgIGFjY2VwdGVkID0gW25vcm1hbGl6ZV9zdHJpbmcoc3RyKHJlZikpXQ0KICAgIGlmIGFsaWFzZXM6DQogICAgICAgIGZvciBhbGlhcyBpbiBhbGlhc2VzOg0KICAgICAgICAgICAgYWNjZXB0ZWQuYXBwZW5kKG5vcm1hbGl6ZV9zdHJpbmcoc3RyKGFsaWFzKSkpDQoNCiAgICByZXR1cm4gcHJlZF9ub3JtIGluIGFjY2VwdGVkDQoNCg0KZGVmIG1hdGNoX2RhdGUocHJlZDogQW55LCByZWY6IEFueSkgLT4gYm9vbDoNCiAgICAiIiJNYXRjaCBkYXRlIGFuc3dlcnMuIEV4YWN0IG1hdGNoIGluIElTTyBmb3JtYXQuIiIiDQogICAgcmV0dXJuIHN0cihwcmVkKS5zdHJpcCgpID09IHN0cihyZWYpLnN0cmlwKCkNCg0KDQpkZWYgbWF0Y2hfYm9vbGVhbihwcmVkOiBBbnksIHJlZjogQW55KSAtPiBib29sOg0KICAgICIiIk1hdGNoIGJvb2xlYW4gYW5zd2Vycy4gRXhhY3QgbWF0Y2guIiIiDQogICAgZGVmIHRvX2Jvb2wodik6DQogICAgICAgIGlmIGlzaW5zdGFuY2UodiwgYm9vbCk6DQogICAgICAgICAgICByZXR1cm4gdg0KICAgICAgICBzID0gc3RyKHYpLnN0cmlwKCkubG93ZXIoKQ0KICAgICAgICBpZiBzIGluICgidHJ1ZSIsICIxIik6DQogICAgICAgICAgICByZXR1cm4gVHJ1ZQ0KICAgICAgICBpZiBzIGluICgiZmFsc2UiLCAiMCIpOg0KICAgICAgICAgICAgcmV0dXJuIEZhbHNlDQogICAgICAgIHJldHVybiBOb25lDQoNCiAgICByZXR1cm4gdG9fYm9vbChwcmVkKSA9PSB0b19ib29sKHJlZikNCg0KDQpkZWYgbWF0Y2hfbGlzdChwcmVkOiBBbnksIHJlZjogQW55KSAtPiBib29sOg0KICAgICIiIk1hdGNoIGxpc3QgYW5zd2VycyBwb3NpdGlvbiBieSBwb3NpdGlvbi4iIiINCiAgICBpZiBub3QgaXNpbnN0YW5jZShwcmVkLCBsaXN0KSBvciBub3QgaXNpbnN0YW5jZShyZWYsIGxpc3QpOg0KICAgICAgICByZXR1cm4gRmFsc2UNCiAgICBpZiBsZW4ocHJlZCkgIT0gbGVuKHJlZik6DQogICAgICAgIHJldHVybiBGYWxzZQ0KICAgIGZvciBwLCByIGluIHppcChwcmVkLCByZWYpOg0KICAgICAgICBpZiBpc2luc3RhbmNlKHIsIChpbnQsIGZsb2F0KSk6DQogICAgICAgICAgICBpZiBub3QgbWF0Y2hfbnVtYmVyKHAsIHIpOg0KICAgICAgICAgICAgICAgIHJldHVybiBGYWxzZQ0KICAgICAgICBlbHNlOg0KICAgICAgICAgICAgaWYgbm90IG1hdGNoX3N0cmluZyhwLCByKToNCiAgICAgICAgICAgICAgICByZXR1cm4gRmFsc2UNCiAgICByZXR1cm4gVHJ1ZQ0KDQoNCmRlZiBtYXRjaF90YWJsZShwcmVkOiBBbnksIHJlZjogQW55KSAtPiBib29sOg0KICAgICIiIk1hdGNoIHRhYmxlIGFuc3dlcnM6IGNvbHVtbnMgYnkgbmFtZSwgcm93cyBhcyBzZXRzLiIiIg0KICAgIGlmIG5vdCBpc2luc3RhbmNlKHByZWQsIGRpY3QpIG9yIG5vdCBpc2luc3RhbmNlKHJlZiwgZGljdCk6DQogICAgICAgIHJldHVybiBGYWxzZQ0KDQogICAgcHJlZF9jb2xzID0gcHJlZC5nZXQoImNvbHVtbnMiLCBbXSkNCiAgICByZWZfY29scyA9IHJlZi5nZXQoImNvbHVtbnMiLCBbXSkNCiAgICBwcmVkX3Jvd3MgPSBwcmVkLmdldCgicm93cyIsIFtdKQ0KICAgIHJlZl9yb3dzID0gcmVmLmdldCgicm93cyIsIFtdKQ0KDQogICAgIyBDb2x1bW4gbWF0Y2hpbmcNCiAgICBwcmVkX2NvbHNfbm9ybSA9IFtub3JtYWxpemVfc3RyaW5nKGMpIGZvciBjIGluIHByZWRfY29sc10NCiAgICByZWZfY29sc19ub3JtID0gW25vcm1hbGl6ZV9zdHJpbmcoYykgZm9yIGMgaW4gcmVmX2NvbHNdDQoNCiAgICBpZiBwcmVkX2NvbHNfbm9ybSA9PSByZWZfY29sc19ub3JtOg0KICAgICAgICBjb2xfb3JkZXIgPSBsaXN0KHJhbmdlKGxlbihwcmVkX2NvbHMpKSkNCiAgICBlbGlmIHNldChwcmVkX2NvbHNfbm9ybSkgPT0gc2V0KHJlZl9jb2xzX25vcm0pOg0KICAgICAgICAjIFJlb3JkZXIgcHJlZCBjb2x1bW5zIHRvIG1hdGNoIHJlZg0KICAgICAgICBjb2xfb3JkZXIgPSBbcHJlZF9jb2xzX25vcm0uaW5kZXgoYykgZm9yIGMgaW4gcmVmX2NvbHNfbm9ybV0NCiAgICBlbHNlOg0KICAgICAgICAjIENvbHVtbnMgZG9uJ3QgbWF0Y2ggYnkgbmFtZSwgdHJ5IHBvc2l0aW9uYWwNCiAgICAgICAgaWYgbGVuKHByZWRfY29scykgIT0gbGVuKHJlZl9jb2xzKToNCiAgICAgICAgICAgIHJldHVybiBGYWxzZQ0KICAgICAgICBjb2xfb3JkZXIgPSBsaXN0KHJhbmdlKGxlbihwcmVkX2NvbHMpKSkNCg0KICAgICMgUm93IG1hdGNoaW5nIChhcyBzZXRzKQ0KICAgIGlmIGxlbihwcmVkX3Jvd3MpICE9IGxlbihyZWZfcm93cyk6DQogICAgICAgIHJldHVybiBGYWxzZQ0KDQogICAgIyBSZW9yZGVyIHByZWQgcm93IGNvbHVtbnMNCiAgICByZW9yZGVyZWRfcHJlZCA9IFtdDQogICAgZm9yIHJvdyBpbiBwcmVkX3Jvd3M6DQogICAgICAgIHJlb3JkZXJlZF9wcmVkLmFwcGVuZCh0dXBsZShyb3dbaV0gZm9yIGkgaW4gY29sX29yZGVyKSkNCg0KICAgIHJlZl90dXBsZXMgPSBbdHVwbGUocm93KSBmb3Igcm93IGluIHJlZl9yb3dzXQ0KDQogICAgIyBNYXRjaCByb3dzIGFzIHNldHMNCiAgICBtYXRjaGVkID0gW0ZhbHNlXSAqIGxlbihyZWZfdHVwbGVzKQ0KICAgIGZvciBwcmVkX3JvdyBpbiByZW9yZGVyZWRfcHJlZDoNCiAgICAgICAgZm91bmQgPSBGYWxzZQ0KICAgICAgICBmb3IgaiwgcmVmX3JvdyBpbiBlbnVtZXJhdGUocmVmX3R1cGxlcyk6DQogICAgICAgICAgICBpZiBtYXRjaGVkW2pdOg0KICAgICAgICAgICAgICAgIGNvbnRpbnVlDQogICAgICAgICAgICBpZiBfcm93c19tYXRjaChwcmVkX3JvdywgcmVmX3Jvdyk6DQogICAgICAgICAgICAgICAgbWF0Y2hlZFtqXSA9IFRydWUNCiAgICAgICAgICAgICAgICBmb3VuZCA9IFRydWUNCiAgICAgICAgICAgICAgICBicmVhaw0KICAgICAgICBpZiBub3QgZm91bmQ6DQogICAgICAgICAgICByZXR1cm4gRmFsc2UNCg0KICAgIHJldHVybiBhbGwobWF0Y2hlZCkNCg0KDQpkZWYgX3Jvd3NfbWF0Y2gocHJlZF9yb3c6IHR1cGxlLCByZWZfcm93OiB0dXBsZSkgLT4gYm9vbDoNCiAgICAiIiJDaGVjayBpZiB0d28gdGFibGUgcm93cyBtYXRjaCBjZWxsIGJ5IGNlbGwuIiIiDQogICAgaWYgbGVuKHByZWRfcm93KSAhPSBsZW4ocmVmX3Jvdyk6DQogICAgICAgIHJldHVybiBGYWxzZQ0KICAgIGZvciBwLCByIGluIHppcChwcmVkX3JvdywgcmVmX3Jvdyk6DQogICAgICAgIGlmIGlzaW5zdGFuY2UociwgKGludCwgZmxvYXQpKToNCiAgICAgICAgICAgIGlmIG5vdCBtYXRjaF9udW1iZXIocCwgcik6DQogICAgICAgICAgICAgICAgcmV0dXJuIEZhbHNlDQogICAgICAgIGVsc2U6DQogICAgICAgICAgICBpZiBub3QgbWF0Y2hfc3RyaW5nKHAsIHIpOg0KICAgICAgICAgICAgICAgIHJldHVybiBGYWxzZQ0KICAgIHJldHVybiBUcnVlDQoNCg0KIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0NCiMgTWFpbiBzY29yaW5nDQojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQ0KDQpfTUFUQ0hFUlMgPSB7DQogICAgIm51bWJlciI6IG1hdGNoX251bWJlciwNCiAgICAicGVyY2VudCI6IG1hdGNoX3BlcmNlbnQsDQogICAgInN0cmluZyI6IG1hdGNoX3N0cmluZywNCiAgICAiZGF0ZSI6IG1hdGNoX2RhdGUsDQogICAgImJvb2xlYW4iOiBtYXRjaF9ib29sZWFuLA0KICAgICJsaXN0IjogbWF0Y2hfbGlzdCwNCiAgICAidGFibGUiOiBtYXRjaF90YWJsZSwNCn0NCg0KDQpkZWYgc2NvcmVfYW5zd2VyKHByZWQ6IERpY3QsIHJlZjogRGljdCkgLT4gaW50Og0KICAgICIiIg0KICAgIFNjb3JlIGEgc2luZ2xlIGFuc3dlci4NCg0KICAgIFJldHVybnM6DQogICAgICAgICsxIGlmIGNvcnJlY3QNCiAgICAgICAgIDAgaWYgYWJzdGFpbg0KICAgICAgICAtMSBpZiB3cm9uZw0KICAgICIiIg0KICAgIHByZWRfdHlwZSA9IHByZWQuZ2V0KCJ0eXBlIiwgImFic3RhaW4iKQ0KICAgIHJlZl90eXBlID0gcmVmLmdldCgidHlwZSIpDQoNCiAgICAjIEFic3RhaW4gKG9yIG1pc3NpbmcgYW5zd2VyKQ0KICAgIGlmIHByZWRfdHlwZSA9PSAiYWJzdGFpbiI6DQogICAgICAgIHJldHVybiAwDQoNCiAgICAjIFVuYW5zd2VyYWJsZQ0KICAgIGlmIHByZWRfdHlwZSA9PSAidW5hbnN3ZXJhYmxlIjoNCiAgICAgICAgaWYgcmVmX3R5cGUgPT0gInVuYW5zd2VyYWJsZSI6DQogICAgICAgICAgICByZXR1cm4gMQ0KICAgICAgICBlbHNlOg0KICAgICAgICAgICAgcmV0dXJuIC0xDQoNCiAgICAjIFR5cGUgbWlzbWF0Y2gNCiAgICBpZiBwcmVkX3R5cGUgIT0gcmVmX3R5cGU6DQogICAgICAgIHJldHVybiAtMQ0KDQogICAgIyBGb3IgdW5hbnN3ZXJhYmxlIHJlZmVyZW5jZSwgYW55IG5vbi1hYnN0YWluIGFuc3dlciBpcyB3cm9uZw0KICAgIGlmIHJlZl90eXBlID09ICJ1bmFuc3dlcmFibGUiOg0KICAgICAgICByZXR1cm4gLTENCg0KICAgICMgTWF0Y2ggdmFsdWVzDQogICAgbWF0Y2hlciA9IF9NQVRDSEVSUy5nZXQocHJlZF90eXBlKQ0KICAgIGlmIG1hdGNoZXIgaXMgTm9uZToNCiAgICAgICAgcmV0dXJuIC0xDQoNCiAgICBwcmVkX2Fuc3dlciA9IHByZWQuZ2V0KCJhbnN3ZXIiKQ0KICAgIHJlZl9hbnN3ZXIgPSByZWYuZ2V0KCJhbnN3ZXIiKQ0KICAgIGFsaWFzZXMgPSByZWYuZ2V0KCJhbGlhcyIsIFtdKQ0KDQogICAgaWYgcHJlZF90eXBlID09ICJzdHJpbmciOg0KICAgICAgICBpZiBtYXRjaGVyKHByZWRfYW5zd2VyLCByZWZfYW5zd2VyLCBhbGlhc2VzKToNCiAgICAgICAgICAgIHJldHVybiAxDQogICAgZWxzZToNCiAgICAgICAgaWYgbWF0Y2hlcihwcmVkX2Fuc3dlciwgcmVmX2Fuc3dlcik6DQogICAgICAgICAgICByZXR1cm4gMQ0KDQogICAgcmV0dXJuIC0xDQoNCg0KZGVmIGNvbXB1dGVfcGEocHJlZGljdGlvbnM6IERpY3Rbc3RyLCBEaWN0XSwgcmVmZXJlbmNlczogRGljdFtzdHIsIERpY3RdKSAtPiBEaWN0Og0KICAgICIiIg0KICAgIENvbXB1dGUgUGVuYWxpemVkIEFjY3VyYWN5IChQQSkgb3ZlciBhIHNldCBvZiBxdWVzdGlvbnMuDQoNCiAgICBQYXJhbWV0ZXJzDQogICAgLS0tLS0tLS0tLQ0KICAgIHByZWRpY3Rpb25zIDogRGljdFtzdHIsIERpY3RdDQogICAgICAgIE1hcHBpbmcgZnJvbSBxdWVzdGlvbl9pZCB0byBwcmVkaWN0aW9uIGRpY3QuDQogICAgcmVmZXJlbmNlcyA6IERpY3Rbc3RyLCBEaWN0XQ0KICAgICAgICBNYXBwaW5nIGZyb20gcXVlc3Rpb25faWQgdG8gcmVmZXJlbmNlIGRpY3QuDQoNCiAgICBSZXR1cm5zDQogICAgLS0tLS0tLQ0KICAgIERpY3QNCiAgICAgICAgUmVzdWx0cyB3aXRoIFBBIHNjb3JlIGFuZCBkZXRhaWxlZCBicmVha2Rvd24uDQogICAgIiIiDQogICAgY29ycmVjdCA9IDANCiAgICB3cm9uZyA9IDANCiAgICBhYnN0YWluID0gMA0KICAgIG1pc3NpbmcgPSAwDQogICAgdG90YWwgPSBsZW4ocmVmZXJlbmNlcykNCg0KICAgIHBlcl90eXBlX3Njb3JlcyA9IHt9DQogICAgZGV0YWlscyA9IFtdDQoNCiAgICBmb3IgcV9pZCwgcmVmIGluIHJlZmVyZW5jZXMuaXRlbXMoKToNCiAgICAgICAgcHJlZCA9IHByZWRpY3Rpb25zLmdldChxX2lkKQ0KDQogICAgICAgIGlmIHByZWQgaXMgTm9uZToNCiAgICAgICAgICAgICMgTWlzc2luZyA9IGFic3RhaW4NCiAgICAgICAgICAgIHNjb3JlID0gMA0KICAgICAgICAgICAgbWlzc2luZyArPSAxDQogICAgICAgIGVsc2U6DQogICAgICAgICAgICBzY29yZSA9IHNjb3JlX2Fuc3dlcihwcmVkLCByZWYpDQoNCiAgICAgICAgcmVmX3R5cGUgPSByZWYuZ2V0KCJ0eXBlIiwgInVua25vd24iKQ0KICAgICAgICBpZiByZWZfdHlwZSBub3QgaW4gcGVyX3R5cGVfc2NvcmVzOg0KICAgICAgICAgICAgcGVyX3R5cGVfc2NvcmVzW3JlZl90eXBlXSA9IHsiY29ycmVjdCI6IDAsICJ3cm9uZyI6IDAsICJhYnN0YWluIjogMCwgInRvdGFsIjogMH0NCg0KICAgICAgICBwZXJfdHlwZV9zY29yZXNbcmVmX3R5cGVdWyJ0b3RhbCJdICs9IDENCg0KICAgICAgICBpZiBzY29yZSA9PSAxOg0KICAgICAgICAgICAgY29ycmVjdCArPSAxDQogICAgICAgICAgICBwZXJfdHlwZV9zY29yZXNbcmVmX3R5cGVdWyJjb3JyZWN0Il0gKz0gMQ0KICAgICAgICBlbGlmIHNjb3JlID09IC0xOg0KICAgICAgICAgICAgd3JvbmcgKz0gMQ0KICAgICAgICAgICAgcGVyX3R5cGVfc2NvcmVzW3JlZl90eXBlXVsid3JvbmciXSArPSAxDQogICAgICAgIGVsc2U6DQogICAgICAgICAgICBhYnN0YWluICs9IDENCiAgICAgICAgICAgIHBlcl90eXBlX3Njb3Jlc1tyZWZfdHlwZV1bImFic3RhaW4iXSArPSAxDQoNCiAgICAgICAgZGV0YWlscy5hcHBlbmQoew0KICAgICAgICAgICAgImlkIjogcV9pZCwNCiAgICAgICAgICAgICJzY29yZSI6IHNjb3JlLA0KICAgICAgICAgICAgInByZWRfdHlwZSI6IHByZWQuZ2V0KCJ0eXBlIikgaWYgcHJlZCBlbHNlICJtaXNzaW5nIiwNCiAgICAgICAgICAgICJyZWZfdHlwZSI6IHJlZl90eXBlLA0KICAgICAgICB9KQ0KDQogICAgcGEgPSAoY29ycmVjdCAtIHdyb25nKSAvIHRvdGFsIGlmIHRvdGFsID4gMCBlbHNlIDAuMA0KDQogICAgcmV0dXJuIHsNCiAgICAgICAgInBhIjogcm91bmQocGEsIDQpLA0KICAgICAgICAicGVuYWxpemVkX2FjY3VyYWN5Ijogcm91bmQocGEsIDQpLA0KICAgICAgICAidG90YWwiOiB0b3RhbCwNCiAgICAgICAgImNvcnJlY3QiOiBjb3JyZWN0LA0KICAgICAgICAid3JvbmciOiB3cm9uZywNCiAgICAgICAgImFic3RhaW4iOiBhYnN0YWluLA0KICAgICAgICAibWlzc2luZyI6IG1pc3NpbmcsDQogICAgICAgICJhY2N1cmFjeSI6IHJvdW5kKGNvcnJlY3QgLyB0b3RhbCwgNCkgaWYgdG90YWwgPiAwIGVsc2UgMC4wLA0KICAgICAgICAicGVyX3R5cGUiOiBwZXJfdHlwZV9zY29yZXMsDQogICAgICAgICJkZXRhaWxzIjogZGV0YWlscywNCiAgICB9DQoNCg0KIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0NCiMgQ0xJDQojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQ0KDQpkZWYgbWFpbigpOg0KICAgIHBhcnNlciA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPSJEYXRhTWF0ZSBMb2NhbCBFdmFsdWF0b3IiKQ0KICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tc3VibWlzc2lvbiIsIHR5cGU9c3RyLCByZXF1aXJlZD1UcnVlLA0KICAgICAgICAgICAgICAgICAgICAgICAgaGVscD0iUGF0aCB0byBzdWJtaXNzaW9uLmpzb25sIikNCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWFuc3dlcnMiLCB0eXBlPXN0ciwgcmVxdWlyZWQ9VHJ1ZSwNCiAgICAgICAgICAgICAgICAgICAgICAgIGhlbHA9IlBhdGggdG8gYW5zd2Vycy5qc29ubCAocmVmZXJlbmNlKSIpDQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1vdXRwdXQiLCB0eXBlPXN0ciwgZGVmYXVsdD1Ob25lLA0KICAgICAgICAgICAgICAgICAgICAgICAgaGVscD0iUGF0aCB0byB3cml0ZSBkZXRhaWxlZCByZXN1bHRzIEpTT04iKQ0KDQogICAgYXJncyA9IHBhcnNlci5wYXJzZV9hcmdzKCkNCg0KICAgICMgTG9hZCBwcmVkaWN0aW9ucw0KICAgIHByZWRpY3Rpb25zID0ge30NCiAgICB3aXRoIG9wZW4oYXJncy5zdWJtaXNzaW9uLCAiciIsIGVuY29kaW5nPSJ1dGYtOCIpIGFzIGY6DQogICAgICAgIGZvciBsaW5lIGluIGY6DQogICAgICAgICAgICBsaW5lID0gbGluZS5zdHJpcCgpDQogICAgICAgICAgICBpZiBsaW5lOg0KICAgICAgICAgICAgICAgIGVudHJ5ID0ganNvbi5sb2FkcyhsaW5lKQ0KICAgICAgICAgICAgICAgIHByZWRpY3Rpb25zW2VudHJ5WyJpZCJdXSA9IGVudHJ5DQoNCiAgICAjIExvYWQgcmVmZXJlbmNlcw0KICAgIHJlZmVyZW5jZXMgPSB7fQ0KICAgIHdpdGggb3BlbihhcmdzLmFuc3dlcnMsICJyIiwgZW5jb2Rpbmc9InV0Zi04IikgYXMgZjoNCiAgICAgICAgZm9yIGxpbmUgaW4gZjoNCiAgICAgICAgICAgIGxpbmUgPSBsaW5lLnN0cmlwKCkNCiAgICAgICAgICAgIGlmIGxpbmU6DQogICAgICAgICAgICAgICAgZW50cnkgPSBqc29uLmxvYWRzKGxpbmUpDQogICAgICAgICAgICAgICAgcmVmZXJlbmNlc1tlbnRyeVsiaWQiXV0gPSBlbnRyeQ0KDQogICAgIyBDb21wdXRlIFBBDQogICAgcmVzdWx0cyA9IGNvbXB1dGVfcGEocHJlZGljdGlvbnMsIHJlZmVyZW5jZXMpDQoNCiAgICAjIFByaW50IHJlc3VsdHMNCiAgICBwcmludChmIlxueyc9Jyo1MH0iKQ0KICAgIHByaW50KGYiRGF0YU1hdGUgRXZhbHVhdGlvbiBSZXN1bHRzIikNCiAgICBwcmludChmInsnPScqNTB9IikNCiAgICBwcmludChmIiAgUEEgU2NvcmU6ICB7cmVzdWx0c1sncGEnXTouNGZ9IikNCiAgICBwcmludChmIiAgQWNjdXJhY3k6ICB7cmVzdWx0c1snYWNjdXJhY3knXTouNGZ9IikNCiAgICBwcmludChmIiAgQ29ycmVjdDogICB7cmVzdWx0c1snY29ycmVjdCddfS97cmVzdWx0c1sndG90YWwnXX0iKQ0KICAgIHByaW50KGYiICBXcm9uZzogICAgIHtyZXN1bHRzWyd3cm9uZyddfS97cmVzdWx0c1sndG90YWwnXX0iKQ0KICAgIHByaW50KGYiICBBYnN0YWluOiAgIHtyZXN1bHRzWydhYnN0YWluJ119L3tyZXN1bHRzWyd0b3RhbCddfSIpDQogICAgcHJpbnQoZiIgIE1pc3Npbmc6ICAge3Jlc3VsdHNbJ21pc3NpbmcnXX0ve3Jlc3VsdHNbJ3RvdGFsJ119IikNCiAgICBwcmludCgpDQogICAgcHJpbnQoIiAgUGVyLXR5cGUgYnJlYWtkb3duOiIpDQogICAgZm9yIHQsIHNjb3JlcyBpbiByZXN1bHRzWyJwZXJfdHlwZSJdLml0ZW1zKCk6DQogICAgICAgIHBhX3QgPSAoc2NvcmVzWyJjb3JyZWN0Il0gLSBzY29yZXNbIndyb25nIl0pIC8gc2NvcmVzWyJ0b3RhbCJdIGlmIHNjb3Jlc1sidG90YWwiXSA+IDAgZWxzZSAwDQogICAgICAgIHByaW50KGYiICAgIHt0OjE1c306IFBBPXtwYV90Oi4zZn0gICINCiAgICAgICAgICAgICAgZiIo4pyTe3Njb3Jlc1snY29ycmVjdCddfSDinJd7c2NvcmVzWyd3cm9uZyddfSDil4t7c2NvcmVzWydhYnN0YWluJ119ICINCiAgICAgICAgICAgICAgZiIvIHtzY29yZXNbJ3RvdGFsJ119KSIpDQogICAgcHJpbnQoZiJ7Jz0nKjUwfVxuIikNCg0KICAgICMgV3JpdGUgZGV0YWlsZWQgcmVzdWx0cw0KICAgIGlmIGFyZ3Mub3V0cHV0Og0KICAgICAgICB3aXRoIG9wZW4oYXJncy5vdXRwdXQsICJ3IiwgZW5jb2Rpbmc9InV0Zi04IikgYXMgZjoNCiAgICAgICAgICAgIGpzb24uZHVtcChyZXN1bHRzLCBmLCBlbnN1cmVfYXNjaWk9RmFsc2UsIGluZGVudD0yKQ0KICAgICAgICBwcmludChmIkRldGFpbGVkIHJlc3VsdHMgd3JpdHRlbiB0byB7YXJncy5vdXRwdXR9IikNCg0KDQppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOg0KICAgIG1haW4oKQ0K'
source_files['src/modules/__init__.py'] = 'IyBEYXRhTWF0ZSBtb2R1bGVzDQo='
source_files['src/modules/answer_formatter.py'] = 'IiIiDQphbnN3ZXJfZm9ybWF0dGVyLnB5IOKAlCBBbnN3ZXIgRm9ybWF0dGluZyAmIFZhbGlkYXRpb24gZm9yIERhdGFNYXRlDQoNCkVuc3VyZXMgZXZlcnkgYW5zd2VyIG1hdGNoZXMgdGhlIGV4YWN0IHN1Ym1pc3Npb24gZm9ybWF0IHJlcXVpcmVkIGJ5DQp0aGUgY29tcGV0aXRpb24uIEhhbmRsZXMgYWxsIGFuc3dlciB0eXBlczogbnVtYmVyLCBwZXJjZW50LCBzdHJpbmcsDQpkYXRlLCBib29sZWFuLCBsaXN0LCB0YWJsZSwgdW5hbnN3ZXJhYmxlLCBhYnN0YWluLg0KIiIiDQoNCmltcG9ydCBqc29uDQppbXBvcnQgbG9nZ2luZw0KaW1wb3J0IG1hdGgNCmltcG9ydCByZQ0KaW1wb3J0IHVuaWNvZGVkYXRhDQpmcm9tIGRhdGV0aW1lIGltcG9ydCBkYXRlLCBkYXRldGltZQ0KZnJvbSB0eXBpbmcgaW1wb3J0IEFueSwgRGljdCwgTGlzdCwgT3B0aW9uYWwsIFR1cGxlLCBVbmlvbg0KDQppbXBvcnQgcGFuZGFzIGFzIHBkDQppbXBvcnQgbnVtcHkgYXMgbnANCg0KbG9nZ2VyID0gbG9nZ2luZy5nZXRMb2dnZXIoX19uYW1lX18pDQoNCg0KIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0NCiMgU3RyaW5nIG5vcm1hbGl6YXRpb24gKG1hdGNoZXMgY29tcGV0aXRpb24gZXZhbHVhdGlvbikNCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tDQoNCmRlZiBub3JtYWxpemVfc3RyaW5nKHM6IHN0cikgLT4gc3RyOg0KICAgICIiIg0KICAgIE5vcm1hbGl6ZSBhIHN0cmluZyBmb3IgY29tcGFyaXNvbiwgZm9sbG93aW5nIHRoZSBjb21wZXRpdGlvbiBydWxlczoNCiAgICAxLiBVbmljb2RlIE5GQw0KICAgIDIuIExvd2VyY2FzZQ0KICAgIDMuIEV2ZXJ5IGNoYXJhY3RlciB0aGF0IGlzIG5vdCBhIGxldHRlciwgZGlnaXQsIG9yIHdoaXRlc3BhY2Ug4oaSIHNwYWNlDQogICAgNC4gTGVhZGluZy90cmFpbGluZyB3aGl0ZXNwYWNlIHJlbW92ZWQNCiAgICA1LiBSdW5zIG9mIHdoaXRlc3BhY2UgY29sbGFwc2VkIHRvIG9uZSBzcGFjZQ0KICAgIE5vdGU6IERpYWNyaXRpY3MgYXJlIE5PVCByZW1vdmVkLg0KICAgICIiIg0KICAgIHMgPSB1bmljb2RlZGF0YS5ub3JtYWxpemUoIk5GQyIsIHN0cihzKSkNCiAgICBzID0gcy5sb3dlcigpDQogICAgIyBSZXBsYWNlIG5vbi1sZXR0ZXIsIG5vbi1kaWdpdCwgbm9uLXdoaXRlc3BhY2Ugd2l0aCBzcGFjZQ0KICAgIHMgPSByZS5zdWIociJbXlx3XHNdIiwgIiAiLCBzLCBmbGFncz1yZS5VTklDT0RFKQ0KICAgIHMgPSBzLnN0cmlwKCkNCiAgICBzID0gcmUuc3ViKHIiXHMrIiwgIiAiLCBzKQ0KICAgIHJldHVybiBzDQoNCg0KIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0NCiMgVHlwZS1zcGVjaWZpYyBmb3JtYXR0ZXJzDQojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQ0KDQpkZWYgZm9ybWF0X251bWJlcih2YWx1ZTogQW55KSAtPiBPcHRpb25hbFtVbmlvbltpbnQsIGZsb2F0XV06DQogICAgIiIiRm9ybWF0IGEgbnVtZXJpYyBhbnN3ZXIuIFJldHVybnMgaW50ZWdlciBpZiBleGFjdCwgZWxzZSByb3VuZGVkIHRvIDIgZGVjaW1hbHMuIiIiDQogICAgaWYgdmFsdWUgaXMgTm9uZToNCiAgICAgICAgcmV0dXJuIE5vbmUNCiAgICB0cnk6DQogICAgICAgIG51bSA9IGZsb2F0KHZhbHVlKQ0KICAgICAgICBpZiBtYXRoLmlzbmFuKG51bSkgb3IgbWF0aC5pc2luZihudW0pOg0KICAgICAgICAgICAgcmV0dXJuIE5vbmUNCiAgICAgICAgIyBDaGVjayBpZiBpdCdzIGVmZmVjdGl2ZWx5IGFuIGludGVnZXINCiAgICAgICAgaWYgbnVtID09IGludChudW0pIGFuZCBhYnMobnVtKSA8IDFlMTU6DQogICAgICAgICAgICByZXR1cm4gaW50KG51bSkNCiAgICAgICAgcmV0dXJuIHJvdW5kKG51bSwgMikNCiAgICBleGNlcHQgKFZhbHVlRXJyb3IsIFR5cGVFcnJvcik6DQogICAgICAgIHJldHVybiBOb25lDQoNCg0KZGVmIGZvcm1hdF9wZXJjZW50KHZhbHVlOiBBbnkpIC0+IE9wdGlvbmFsW2Zsb2F0XToNCiAgICAiIiJGb3JtYXQgYSBwZXJjZW50YWdlIGFuc3dlci4gUm91bmRlZCB0byAyIGRlY2ltYWwgcGxhY2VzLiIiIg0KICAgIGlmIHZhbHVlIGlzIE5vbmU6DQogICAgICAgIHJldHVybiBOb25lDQogICAgdHJ5Og0KICAgICAgICBudW0gPSBmbG9hdCh2YWx1ZSkNCiAgICAgICAgaWYgbWF0aC5pc25hbihudW0pIG9yIG1hdGguaXNpbmYobnVtKToNCiAgICAgICAgICAgIHJldHVybiBOb25lDQogICAgICAgIHJldHVybiByb3VuZChudW0sIDIpDQogICAgZXhjZXB0IChWYWx1ZUVycm9yLCBUeXBlRXJyb3IpOg0KICAgICAgICByZXR1cm4gTm9uZQ0KDQoNCmRlZiBmb3JtYXRfc3RyaW5nKHZhbHVlOiBBbnkpIC0+IE9wdGlvbmFsW3N0cl06DQogICAgIiIiRm9ybWF0IGEgc3RyaW5nIGFuc3dlci4iIiINCiAgICBpZiB2YWx1ZSBpcyBOb25lOg0KICAgICAgICByZXR1cm4gTm9uZQ0KICAgIHMgPSBzdHIodmFsdWUpLnN0cmlwKCkNCiAgICBpZiBub3QgczoNCiAgICAgICAgcmV0dXJuIE5vbmUNCiAgICAjIFVuaWNvZGUgTkZDIG5vcm1hbGl6ZSBidXQgZG9uJ3QgbG93ZXJjYXNlIChwcmVzZXJ2ZSBvcmlnaW5hbCBjYXNpbmcpDQogICAgcmV0dXJuIHVuaWNvZGVkYXRhLm5vcm1hbGl6ZSgiTkZDIiwgcykNCg0KDQpkZWYgZm9ybWF0X2RhdGUodmFsdWU6IEFueSkgLT4gT3B0aW9uYWxbc3RyXToNCiAgICAiIiJGb3JtYXQgYSBkYXRlIGFuc3dlciBpbiBJU08gZm9ybWF0IChZWVlZLU1NLUREIG9yIFlZWVktTU0pLiIiIg0KICAgIGlmIHZhbHVlIGlzIE5vbmU6DQogICAgICAgIHJldHVybiBOb25lDQoNCiAgICBzID0gc3RyKHZhbHVlKS5zdHJpcCgpDQoNCiAgICAjIEFscmVhZHkgaW4gSVNPIGZvcm1hdD8NCiAgICBpZiByZS5tYXRjaChyIl5cZHs0fS1cZHsyfS1cZHsyfSQiLCBzKToNCiAgICAgICAgcmV0dXJuIHMNCiAgICBpZiByZS5tYXRjaChyIl5cZHs0fS1cZHsyfSQiLCBzKToNCiAgICAgICAgcmV0dXJuIHMNCg0KICAgICMgVHJ5IHBhcnNpbmcgdmFyaW91cyBmb3JtYXRzDQogICAgZm9ybWF0cyA9IFsNCiAgICAgICAgIiVZLSVtLSVkIiwgIiVkLyVtLyVZIiwgIiVtLyVkLyVZIiwNCiAgICAgICAgIiVZLyVtLyVkIiwgIiVkLSVtLSVZIiwgIiVZLSVtIiwNCiAgICBdDQogICAgZm9yIGZtdCBpbiBmb3JtYXRzOg0KICAgICAgICB0cnk6DQogICAgICAgICAgICBkdCA9IGRhdGV0aW1lLnN0cnB0aW1lKHMsIGZtdCkNCiAgICAgICAgICAgIGlmIGZtdCA9PSAiJVktJW0iOg0KICAgICAgICAgICAgICAgIHJldHVybiBkdC5zdHJmdGltZSgiJVktJW0iKQ0KICAgICAgICAgICAgcmV0dXJuIGR0LnN0cmZ0aW1lKCIlWS0lbS0lZCIpDQogICAgICAgIGV4Y2VwdCBWYWx1ZUVycm9yOg0KICAgICAgICAgICAgY29udGludWUNCg0KICAgICMgSGFuZGxlIHBhbmRhcyBUaW1lc3RhbXANCiAgICBpZiBpc2luc3RhbmNlKHZhbHVlLCAocGQuVGltZXN0YW1wLCBkYXRldGltZSwgZGF0ZSkpOg0KICAgICAgICBpZiBpc2luc3RhbmNlKHZhbHVlLCBkYXRlKSBhbmQgbm90IGlzaW5zdGFuY2UodmFsdWUsIGRhdGV0aW1lKToNCiAgICAgICAgICAgIHJldHVybiB2YWx1ZS5pc29mb3JtYXQoKQ0KICAgICAgICByZXR1cm4gdmFsdWUuc3RyZnRpbWUoIiVZLSVtLSVkIikNCg0KICAgIHJldHVybiBOb25lDQoNCg0KZGVmIGZvcm1hdF9ib29sZWFuKHZhbHVlOiBBbnkpIC0+IE9wdGlvbmFsW2Jvb2xdOg0KICAgICIiIkZvcm1hdCBhIGJvb2xlYW4gYW5zd2VyLiIiIg0KICAgIGlmIHZhbHVlIGlzIE5vbmU6DQogICAgICAgIHJldHVybiBOb25lDQogICAgaWYgaXNpbnN0YW5jZSh2YWx1ZSwgYm9vbCk6DQogICAgICAgIHJldHVybiB2YWx1ZQ0KICAgIGlmIGlzaW5zdGFuY2UodmFsdWUsIChpbnQsIGZsb2F0KSk6DQogICAgICAgIHJldHVybiBib29sKHZhbHVlKQ0KICAgIHMgPSBzdHIodmFsdWUpLnN0cmlwKCkubG93ZXIoKQ0KICAgIGlmIHMgaW4gKCJ0cnVlIiwgInllcyIsICLEkcO6bmciLCAiY8OzIiwgIjEiKToNCiAgICAgICAgcmV0dXJuIFRydWUNCiAgICBpZiBzIGluICgiZmFsc2UiLCAibm8iLCAic2FpIiwgImtow7RuZyIsICIwIik6DQogICAgICAgIHJldHVybiBGYWxzZQ0KICAgIHJldHVybiBOb25lDQoNCg0KZGVmIGZvcm1hdF9saXN0KHZhbHVlOiBBbnkpIC0+IE9wdGlvbmFsW0xpc3RdOg0KICAgICIiIkZvcm1hdCBhIGxpc3QgYW5zd2VyLiIiIg0KICAgIGlmIHZhbHVlIGlzIE5vbmU6DQogICAgICAgIHJldHVybiBOb25lDQoNCiAgICAjIENvbnZlcnQgdmFyaW91cyB0eXBlcyB0byBsaXN0DQogICAgaWYgaXNpbnN0YW5jZSh2YWx1ZSwgcGQuU2VyaWVzKToNCiAgICAgICAgdmFsdWUgPSB2YWx1ZS50b2xpc3QoKQ0KICAgIGVsaWYgaXNpbnN0YW5jZSh2YWx1ZSwgcGQuSW5kZXgpOg0KICAgICAgICB2YWx1ZSA9IHZhbHVlLnRvbGlzdCgpDQogICAgZWxpZiBpc2luc3RhbmNlKHZhbHVlLCBucC5uZGFycmF5KToNCiAgICAgICAgdmFsdWUgPSB2YWx1ZS50b2xpc3QoKQ0KICAgIGVsaWYgaXNpbnN0YW5jZSh2YWx1ZSwgc3RyKToNCiAgICAgICAgdHJ5Og0KICAgICAgICAgICAgdmFsdWUgPSBqc29uLmxvYWRzKHZhbHVlKQ0KICAgICAgICBleGNlcHQganNvbi5KU09ORGVjb2RlRXJyb3I6DQogICAgICAgICAgICB2YWx1ZSA9IFt2LnN0cmlwKCkgZm9yIHYgaW4gdmFsdWUuc3BsaXQoIiwiKSBpZiB2LnN0cmlwKCldDQoNCiAgICBpZiBub3QgaXNpbnN0YW5jZSh2YWx1ZSwgbGlzdCk6DQogICAgICAgIHZhbHVlID0gW3ZhbHVlXQ0KDQogICAgIyBGb3JtYXQgZWFjaCBlbGVtZW50DQogICAgZm9ybWF0dGVkID0gW10NCiAgICBmb3IgaXRlbSBpbiB2YWx1ZToNCiAgICAgICAgaWYgaXNpbnN0YW5jZShpdGVtLCAoaW50LCBmbG9hdCkpOg0KICAgICAgICAgICAgZm9ybWF0dGVkLmFwcGVuZChmb3JtYXRfbnVtYmVyKGl0ZW0pKQ0KICAgICAgICBlbHNlOg0KICAgICAgICAgICAgZm9ybWF0dGVkLmFwcGVuZChmb3JtYXRfc3RyaW5nKGl0ZW0pKQ0KDQogICAgcmV0dXJuIGZvcm1hdHRlZA0KDQoNCmRlZiBmb3JtYXRfdGFibGUodmFsdWU6IEFueSkgLT4gT3B0aW9uYWxbRGljdF06DQogICAgIiIiDQogICAgRm9ybWF0IGEgdGFibGUgYW5zd2VyIGFzIHsiY29sdW1ucyI6IFsuLi5dLCAicm93cyI6IFtbLi4uXSwgLi4uXX0uDQogICAgIiIiDQogICAgaWYgdmFsdWUgaXMgTm9uZToNCiAgICAgICAgcmV0dXJuIE5vbmUNCg0KICAgIGlmIGlzaW5zdGFuY2UodmFsdWUsIHBkLkRhdGFGcmFtZSk6DQogICAgICAgIHJldHVybiB7DQogICAgICAgICAgICAiY29sdW1ucyI6IFtzdHIoYykgZm9yIGMgaW4gdmFsdWUuY29sdW1ucy50b2xpc3QoKV0sDQogICAgICAgICAgICAicm93cyI6IF90b19qc29uX2NvbXBhdGlibGUodmFsdWUudmFsdWVzLnRvbGlzdCgpKSwNCiAgICAgICAgfQ0KDQogICAgaWYgaXNpbnN0YW5jZSh2YWx1ZSwgZGljdCk6DQogICAgICAgIGlmICJjb2x1bW5zIiBpbiB2YWx1ZSBhbmQgInJvd3MiIGluIHZhbHVlOg0KICAgICAgICAgICAgcmV0dXJuIF90b19qc29uX2NvbXBhdGlibGUodmFsdWUpDQoNCiAgICByZXR1cm4gTm9uZQ0KDQoNCmRlZiBfdG9fanNvbl9jb21wYXRpYmxlKHZhbHVlOiBBbnkpIC0+IEFueToNCiAgICAiIiJDb252ZXJ0IGNvbW1vbiBwYW5kYXMgYW5kIE51bVB5IHZhbHVlcyB0byBKU09OLWNvbXBhdGlibGUgdmFsdWVzLiIiIg0KICAgIGlmIHZhbHVlIGlzIE5vbmUgb3IgdmFsdWUgaXMgcGQuTkEgb3IgdmFsdWUgaXMgcGQuTmFUOg0KICAgICAgICByZXR1cm4gTm9uZQ0KICAgIGlmIGlzaW5zdGFuY2UodmFsdWUsIHBkLlBlcmlvZCk6DQogICAgICAgIHJldHVybiBzdHIodmFsdWUpDQogICAgaWYgaXNpbnN0YW5jZSh2YWx1ZSwgKHBkLlRpbWVzdGFtcCwgZGF0ZXRpbWUsIGRhdGUpKToNCiAgICAgICAgcmV0dXJuIHZhbHVlLmlzb2Zvcm1hdCgpDQogICAgaWYgaXNpbnN0YW5jZSh2YWx1ZSwgbnAuZ2VuZXJpYyk6DQogICAgICAgIHJldHVybiBfdG9fanNvbl9jb21wYXRpYmxlKHZhbHVlLml0ZW0oKSkNCiAgICBpZiBpc2luc3RhbmNlKHZhbHVlLCBucC5uZGFycmF5KToNCiAgICAgICAgcmV0dXJuIF90b19qc29uX2NvbXBhdGlibGUodmFsdWUudG9saXN0KCkpDQogICAgaWYgaXNpbnN0YW5jZSh2YWx1ZSwgKHBkLlNlcmllcywgcGQuSW5kZXgpKToNCiAgICAgICAgcmV0dXJuIF90b19qc29uX2NvbXBhdGlibGUodmFsdWUudG9saXN0KCkpDQogICAgaWYgaXNpbnN0YW5jZSh2YWx1ZSwgZGljdCk6DQogICAgICAgIHJldHVybiB7DQogICAgICAgICAgICBfdG9fanNvbl9jb21wYXRpYmxlKGtleSk6IF90b19qc29uX2NvbXBhdGlibGUoaXRlbSkNCiAgICAgICAgICAgIGZvciBrZXksIGl0ZW0gaW4gdmFsdWUuaXRlbXMoKQ0KICAgICAgICB9DQogICAgaWYgaXNpbnN0YW5jZSh2YWx1ZSwgKGxpc3QsIHR1cGxlKSk6DQogICAgICAgIHJldHVybiBbX3RvX2pzb25fY29tcGF0aWJsZShpdGVtKSBmb3IgaXRlbSBpbiB2YWx1ZV0NCiAgICBpZiBpc2luc3RhbmNlKHZhbHVlLCBmbG9hdCkgYW5kIG5vdCBtYXRoLmlzZmluaXRlKHZhbHVlKToNCiAgICAgICAgcmV0dXJuIE5vbmUNCiAgICBpZiBpc2luc3RhbmNlKHZhbHVlLCAoc3RyLCBpbnQsIGZsb2F0LCBib29sKSk6DQogICAgICAgIHJldHVybiB2YWx1ZQ0KICAgIHJhaXNlIFR5cGVFcnJvcigNCiAgICAgICAgZiJVbnN1cHBvcnRlZCB2YWx1ZSBmb3IgSlNPTiBzZXJpYWxpemF0aW9uOiB7dHlwZSh2YWx1ZSkuX19uYW1lX199Ig0KICAgICkNCg0KDQojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQ0KIyBNYWluIGZvcm1hdHRlcg0KIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0NCg0KX0ZPUk1BVFRFUlMgPSB7DQogICAgIm51bWJlciI6IGZvcm1hdF9udW1iZXIsDQogICAgInBlcmNlbnQiOiBmb3JtYXRfcGVyY2VudCwNCiAgICAic3RyaW5nIjogZm9ybWF0X3N0cmluZywNCiAgICAiZGF0ZSI6IGZvcm1hdF9kYXRlLA0KICAgICJib29sZWFuIjogZm9ybWF0X2Jvb2xlYW4sDQogICAgImxpc3QiOiBmb3JtYXRfbGlzdCwNCiAgICAidGFibGUiOiBmb3JtYXRfdGFibGUsDQp9DQoNCg0KZGVmIGZvcm1hdF9hbnN3ZXIoDQogICAgdmFsdWU6IEFueSwNCiAgICBhbnN3ZXJfdHlwZTogc3RyLA0KICAgIHJlYXNvbjogT3B0aW9uYWxbc3RyXSA9IE5vbmUsDQopIC0+IERpY3Rbc3RyLCBBbnldOg0KICAgICIiIg0KICAgIEZvcm1hdCBhbiBhbnN3ZXIgZm9yIHN1Ym1pc3Npb24uDQoNCiAgICBQYXJhbWV0ZXJzDQogICAgLS0tLS0tLS0tLQ0KICAgIHZhbHVlIDogQW55DQogICAgICAgIFRoZSByYXcgYW5zd2VyIHZhbHVlLg0KICAgIGFuc3dlcl90eXBlIDogc3RyDQogICAgICAgIFRoZSBhbnN3ZXIgdHlwZSBzdHJpbmcuDQogICAgcmVhc29uIDogc3RyLCBvcHRpb25hbA0KICAgICAgICBSZWFzb24gZm9yIHVuYW5zd2VyYWJsZS9hYnN0YWluLg0KDQogICAgUmV0dXJucw0KICAgIC0tLS0tLS0NCiAgICBEaWN0W3N0ciwgQW55XQ0KICAgICAgICBGb3JtYXR0ZWQgc3VibWlzc2lvbiBlbnRyeSAod2l0aG91dCAiaWQiIGZpZWxkKS4NCiAgICAiIiINCiAgICBpZiBhbnN3ZXJfdHlwZSA9PSAiYWJzdGFpbiI6DQogICAgICAgIGVudHJ5ID0geyJ0eXBlIjogImFic3RhaW4ifQ0KICAgICAgICBpZiByZWFzb246DQogICAgICAgICAgICBlbnRyeVsicmVhc29uIl0gPSByZWFzb24NCiAgICAgICAgcmV0dXJuIGVudHJ5DQoNCiAgICBpZiBhbnN3ZXJfdHlwZSA9PSAidW5hbnN3ZXJhYmxlIjoNCiAgICAgICAgZW50cnkgPSB7InR5cGUiOiAidW5hbnN3ZXJhYmxlIn0NCiAgICAgICAgaWYgcmVhc29uOg0KICAgICAgICAgICAgZW50cnlbInJlYXNvbiJdID0gcmVhc29uDQogICAgICAgIHJldHVybiBlbnRyeQ0KDQogICAgZm9ybWF0dGVyID0gX0ZPUk1BVFRFUlMuZ2V0KGFuc3dlcl90eXBlKQ0KICAgIGlmIGZvcm1hdHRlciBpcyBOb25lOg0KICAgICAgICBsb2dnZXIud2FybmluZygiVW5rbm93biBhbnN3ZXIgdHlwZTogJXMsIGRlZmF1bHRpbmcgdG8gc3RyaW5nIiwgYW5zd2VyX3R5cGUpDQogICAgICAgIGZvcm1hdHRlciA9IGZvcm1hdF9zdHJpbmcNCiAgICAgICAgYW5zd2VyX3R5cGUgPSAic3RyaW5nIg0KDQogICAgZm9ybWF0dGVkX3ZhbHVlID0gX3RvX2pzb25fY29tcGF0aWJsZShmb3JtYXR0ZXIodmFsdWUpKQ0KDQogICAgaWYgZm9ybWF0dGVkX3ZhbHVlIGlzIE5vbmU6DQogICAgICAgIGxvZ2dlci53YXJuaW5nKCJGb3JtYXR0aW5nIHByb2R1Y2VkIE5vbmUgZm9yIHR5cGU9JXMsIHZhbHVlPSVyIiwNCiAgICAgICAgICAgICAgICAgICAgICAgYW5zd2VyX3R5cGUsIHZhbHVlKQ0KICAgICAgICByZXR1cm4geyJ0eXBlIjogImFic3RhaW4iLCAicmVhc29uIjogImZvcm1hdHRpbmdfZmFpbHVyZSJ9DQoNCiAgICByZXR1cm4geyJ0eXBlIjogYW5zd2VyX3R5cGUsICJhbnN3ZXIiOiBmb3JtYXR0ZWRfdmFsdWV9DQoNCg0KZGVmIGJ1aWxkX3N1Ym1pc3Npb25fbGluZShxdWVzdGlvbl9pZDogc3RyLCBhbnN3ZXI6IERpY3Rbc3RyLCBBbnldKSAtPiBzdHI6DQogICAgIiIiQnVpbGQgYSBzaW5nbGUgSlNPTkwgbGluZSBmb3Igc3VibWlzc2lvbi4iIiINCiAgICBlbnRyeSA9IHsiaWQiOiBxdWVzdGlvbl9pZH0NCiAgICBlbnRyeS51cGRhdGUoYW5zd2VyKQ0KICAgIHJldHVybiBqc29uLmR1bXBzKGVudHJ5LCBlbnN1cmVfYXNjaWk9RmFsc2UpDQo='
source_files['src/modules/code_executor.py'] = 'IiIiDQpjb2RlX2V4ZWN1dG9yLnB5IOKAlCBTYW5kYm94ZWQgQ29kZSBFeGVjdXRpb24gZm9yIERhdGFNYXRlDQoNCkV4ZWN1dGVzIExMTS1nZW5lcmF0ZWQgUHl0aG9uL3BhbmRhcyBjb2RlIGluIGEgcmVzdHJpY3RlZCBlbnZpcm9ubWVudA0Kd2l0aCBwcmUtbG9hZGVkIERhdGFGcmFtZXMsIGNhcHR1cmluZyByZXN1bHRzIGFuZCBoYW5kbGluZyBlcnJvcnMuDQoiIiINCg0KaW1wb3J0IGJ1aWx0aW5zDQppbXBvcnQgYXN0DQppbXBvcnQgaW8NCmltcG9ydCBrZXl3b3JkDQppbXBvcnQgbG9nZ2luZw0KaW1wb3J0IHJlDQppbXBvcnQgc2lnbmFsDQppbXBvcnQgc3lzDQppbXBvcnQgdHJhY2ViYWNrDQpmcm9tIGNvbnRleHRsaWIgaW1wb3J0IHJlZGlyZWN0X3N0ZG91dCwgcmVkaXJlY3Rfc3RkZXJyDQpmcm9tIHR5cGluZyBpbXBvcnQgQW55LCBEaWN0LCBPcHRpb25hbCwgVHVwbGUNCg0KaW1wb3J0IHBhbmRhcyBhcyBwZA0KDQpsb2dnZXIgPSBsb2dnaW5nLmdldExvZ2dlcihfX25hbWVfXykNCg0KX1JFU0VSVkVEX0VYRUNfTkFNRVMgPSBmcm96ZW5zZXQoew0KICAgICJwZCIsICJucCIsICJtYXRoIiwgImRhdGV0aW1lIiwgImR0IiwgImRhdGUiLCAidGltZWRlbHRhIiwgInJlIiwNCiAgICAiYXNfb2ZfZGF0ZSIsICJhc19vZl9kYXRlX29iaiIsICJyZXN1bHQiLCAiYW5zd2VyX3R5cGUiLCAiX19idWlsdGluc19fIiwNCn0pDQoNCl9GSUxFX1JFQURfQ0FMTFMgPSBmcm96ZW5zZXQoew0KICAgICJvcGVuIiwNCiAgICAicmVhZF9jc3YiLA0KICAgICJyZWFkX2V4Y2VsIiwNCiAgICAicmVhZF9md2YiLA0KICAgICJyZWFkX2pzb24iLA0KICAgICJyZWFkX2h0bWwiLA0KICAgICJyZWFkX3htbCIsDQogICAgInJlYWRfcGlja2xlIiwNCiAgICAicmVhZF9wYXJxdWV0IiwNCiAgICAicmVhZF9mZWF0aGVyIiwNCiAgICAicmVhZF9oZGYiLA0KICAgICJyZWFkX3N0YXRhIiwNCiAgICAicmVhZF9zYXMiLA0KICAgICJyZWFkX3NxbCIsDQogICAgInJlYWRfc3FsX3F1ZXJ5IiwNCiAgICAicmVhZF9zcWxfdGFibGUiLA0KICAgICJyZWFkX3RhYmxlIiwNCiAgICAicmVhZF9jbGlwYm9hcmQiLA0KICAgICJFeGNlbEZpbGUiLA0KICAgICJIREZTdG9yZSIsDQp9KQ0KDQoNCmRlZiBfdGFibGVfdmFyaWFibGVfbWFwcGluZyh0YWJsZXM6IERpY3Rbc3RyLCBwZC5EYXRhRnJhbWVdKSAtPiBEaWN0W3N0ciwgc3RyXToNCiAgICAiIiJNYXAgdGFibGUgbmFtZXMgdG8gdW5pcXVlLCB2YWxpZCBQeXRob24gaWRlbnRpZmllcnMuIiIiDQogICAgbWFwcGluZyA9IHt9DQogICAgdXNlZF9uYW1lcyA9IHNldChfUkVTRVJWRURfRVhFQ19OQU1FUykNCiAgICBmb3IgbmFtZSBpbiB0YWJsZXM6DQogICAgICAgIGNhbmRpZGF0ZSA9IHJlLnN1YihyIlxXIiwgIl8iLCBuYW1lLCBmbGFncz1yZS5VTklDT0RFKQ0KICAgICAgICBpZiBub3QgY2FuZGlkYXRlIG9yIGNhbmRpZGF0ZVswXS5pc2RpZ2l0KCk6DQogICAgICAgICAgICBjYW5kaWRhdGUgPSBmInRhYmxlX3tjYW5kaWRhdGV9Ig0KICAgICAgICBpZiBub3QgY2FuZGlkYXRlLmlzaWRlbnRpZmllcigpIG9yIGtleXdvcmQuaXNrZXl3b3JkKGNhbmRpZGF0ZSk6DQogICAgICAgICAgICBjYW5kaWRhdGUgPSBmIntjYW5kaWRhdGV9X3RhYmxlIg0KICAgICAgICBiYXNlX25hbWUgPSBjYW5kaWRhdGUNCiAgICAgICAgc3VmZml4ID0gMg0KICAgICAgICB3aGlsZSBjYW5kaWRhdGUgaW4gdXNlZF9uYW1lczoNCiAgICAgICAgICAgIGNhbmRpZGF0ZSA9IGYie2Jhc2VfbmFtZX1fe3N1ZmZpeH0iDQogICAgICAgICAgICBzdWZmaXggKz0gMQ0KICAgICAgICBtYXBwaW5nW25hbWVdID0gY2FuZGlkYXRlDQogICAgICAgIHVzZWRfbmFtZXMuYWRkKGNhbmRpZGF0ZSkNCiAgICByZXR1cm4gbWFwcGluZw0KDQoNCmNsYXNzIEV4ZWN1dGlvblJlc3VsdDoNCiAgICAiIiJSZXN1bHQgb2YgYSBjb2RlIGV4ZWN1dGlvbi4iIiINCg0KICAgIGRlZiBfX2luaXRfXyhzZWxmKToNCiAgICAgICAgc2VsZi5zdWNjZXNzOiBib29sID0gRmFsc2UNCiAgICAgICAgc2VsZi5yZXN1bHQ6IEFueSA9IE5vbmUNCiAgICAgICAgc2VsZi5hbnN3ZXJfdHlwZTogc3RyID0gImFic3RhaW4iDQogICAgICAgIHNlbGYuc3Rkb3V0OiBzdHIgPSAiIg0KICAgICAgICBzZWxmLnN0ZGVycjogc3RyID0gIiINCiAgICAgICAgc2VsZi5lcnJvcjogT3B0aW9uYWxbc3RyXSA9IE5vbmUNCiAgICAgICAgc2VsZi5jb2RlOiBzdHIgPSAiIg0KDQogICAgZGVmIF9fcmVwcl9fKHNlbGYpOg0KICAgICAgICByZXR1cm4gKGYiRXhlY3V0aW9uUmVzdWx0KHN1Y2Nlc3M9e3NlbGYuc3VjY2Vzc30sICINCiAgICAgICAgICAgICAgICBmInR5cGU9e3NlbGYuYW5zd2VyX3R5cGV9LCByZXN1bHQ9e3NlbGYucmVzdWx0IXJ9KSIpDQoNCg0KZGVmIF9leHRyYWN0X2NvZGVfZnJvbV9yZXNwb25zZShyZXNwb25zZTogc3RyKSAtPiBzdHI6DQogICAgIiIiDQogICAgRXh0cmFjdCBQeXRob24gY29kZSBmcm9tIHRoZSBMTE0gcmVzcG9uc2UuDQogICAgSGFuZGxlcyBjb2RlIGJsb2NrcyAoYGBgcHl0aG9uLi4uYGBgKSBhbmQgcmF3IGNvZGUuDQogICAgIiIiDQogICAgIyBUcnkgdG8gZmluZCBhIHB5dGhvbiBjb2RlIGJsb2NrDQogICAgaW1wb3J0IHJlDQogICAgcGF0dGVybnMgPSBbDQogICAgICAgIHIiYGBgcHl0aG9uXHMqXG4oLio/KWBgYCIsDQogICAgICAgIHIiYGBgXHMqXG4oLio/KWBgYCIsDQogICAgXQ0KICAgIGZvciBwYXR0ZXJuIGluIHBhdHRlcm5zOg0KICAgICAgICBtID0gcmUuc2VhcmNoKHBhdHRlcm4sIHJlc3BvbnNlLCByZS5ET1RBTEwpDQogICAgICAgIGlmIG06DQogICAgICAgICAgICByZXR1cm4gbS5ncm91cCgxKS5zdHJpcCgpDQoNCiAgICAjIElmIG5vIGNvZGUgYmxvY2ssIHRyeSB0byBmaW5kIGxpbmVzIHRoYXQgbG9vayBsaWtlIGNvZGUNCiAgICBsaW5lcyA9IHJlc3BvbnNlLnN0cmlwKCkuc3BsaXQoIlxuIikNCiAgICBjb2RlX2xpbmVzID0gW10NCiAgICBpbl9jb2RlID0gRmFsc2UNCiAgICBmb3IgbGluZSBpbiBsaW5lczoNCiAgICAgICAgc3RyaXBwZWQgPSBsaW5lLnN0cmlwKCkNCiAgICAgICAgaWYgKHN0cmlwcGVkLnN0YXJ0c3dpdGgoImltcG9ydCAiKSBvciBzdHJpcHBlZC5zdGFydHN3aXRoKCJmcm9tICIpIG9yDQogICAgICAgICAgICBzdHJpcHBlZC5zdGFydHN3aXRoKCJyZXN1bHQiKSBvciBzdHJpcHBlZC5zdGFydHN3aXRoKCJhbnN3ZXJfdHlwZSIpIG9yDQogICAgICAgICAgICBzdHJpcHBlZC5zdGFydHN3aXRoKCJkZiIpIG9yIHN0cmlwcGVkLnN0YXJ0c3dpdGgoIiMiKSBvcg0KICAgICAgICAgICAgIj0iIGluIHN0cmlwcGVkIG9yIHN0cmlwcGVkLnN0YXJ0c3dpdGgoImZvciAiKSBvcg0KICAgICAgICAgICAgc3RyaXBwZWQuc3RhcnRzd2l0aCgiaWYgIikgb3Igc3RyaXBwZWQuc3RhcnRzd2l0aCgiICAgICIpIG9yDQogICAgICAgICAgICBpbl9jb2RlKToNCiAgICAgICAgICAgIGNvZGVfbGluZXMuYXBwZW5kKGxpbmUpDQogICAgICAgICAgICBpbl9jb2RlID0gVHJ1ZQ0KICAgICAgICBlbGlmIGluX2NvZGUgYW5kIG5vdCBzdHJpcHBlZDoNCiAgICAgICAgICAgIGNvZGVfbGluZXMuYXBwZW5kKGxpbmUpDQoNCiAgICBpZiBjb2RlX2xpbmVzOg0KICAgICAgICByZXR1cm4gIlxuIi5qb2luKGNvZGVfbGluZXMpLnN0cmlwKCkNCg0KICAgICMgRmFsbGJhY2s6IHJldHVybiB0aGUgZW50aXJlIHJlc3BvbnNlIGFzIGNvZGUNCiAgICByZXR1cm4gcmVzcG9uc2Uuc3RyaXAoKQ0KDQoNCmRlZiBfZmlsZV9yZWFkX3Zpb2xhdGlvbihjb2RlOiBzdHIpIC0+IE9wdGlvbmFsW3N0cl06DQogICAgIiIiUmV0dXJuIGEgbWVzc2FnZSB3aGVuIGdlbmVyYXRlZCBjb2RlIGNhbGxzIGEgY29tbW9uIGZpbGUtcmVhZGluZyBBUEkuIiIiDQogICAgdHJ5Og0KICAgICAgICB0cmVlID0gYXN0LnBhcnNlKGNvZGUpDQogICAgZXhjZXB0IFN5bnRheEVycm9yOg0KICAgICAgICByZXR1cm4gTm9uZQ0KDQogICAgZm9yIG5vZGUgaW4gYXN0LndhbGsodHJlZSk6DQogICAgICAgIGlmIG5vdCBpc2luc3RhbmNlKG5vZGUsIGFzdC5DYWxsKToNCiAgICAgICAgICAgIGNvbnRpbnVlDQogICAgICAgIGlmIGlzaW5zdGFuY2Uobm9kZS5mdW5jLCBhc3QuTmFtZSkgYW5kIG5vZGUuZnVuYy5pZCBpbiBfRklMRV9SRUFEX0NBTExTOg0KICAgICAgICAgICAgcmV0dXJuIGYiRmlsZSBhY2Nlc3MgaXMgbm90IGFsbG93ZWQ6IGB7bm9kZS5mdW5jLmlkfSgpYCINCiAgICAgICAgaWYgaXNpbnN0YW5jZShub2RlLmZ1bmMsIGFzdC5BdHRyaWJ1dGUpIGFuZCBub2RlLmZ1bmMuYXR0ciBpbiBfRklMRV9SRUFEX0NBTExTOg0KICAgICAgICAgICAgcmV0dXJuIGYiRmlsZSBhY2Nlc3MgaXMgbm90IGFsbG93ZWQ6IGAue25vZGUuZnVuYy5hdHRyfSgpYCINCiAgICByZXR1cm4gTm9uZQ0KDQoNCmRlZiBfZGF0ZV9jb252ZXJzaW9uX3Zpb2xhdGlvbihjb2RlOiBzdHIpIC0+IE9wdGlvbmFsW3N0cl06DQogICAgIiIiUmVqZWN0IGNvbnZlcnRpbmcgYSBwYW5kYXMgZGF0ZXRpbWUgY29sdW1uIHRvIFB5dGhvbiBkYXRlIG9iamVjdHMuIiIiDQogICAgdHJ5Og0KICAgICAgICB0cmVlID0gYXN0LnBhcnNlKGNvZGUpDQogICAgZXhjZXB0IFN5bnRheEVycm9yOg0KICAgICAgICByZXR1cm4gTm9uZQ0KDQogICAgZm9yIG5vZGUgaW4gYXN0LndhbGsodHJlZSk6DQogICAgICAgIGlmICgNCiAgICAgICAgICAgIGlzaW5zdGFuY2Uobm9kZSwgYXN0LkF0dHJpYnV0ZSkNCiAgICAgICAgICAgIGFuZCBub2RlLmF0dHIgPT0gImRhdGUiDQogICAgICAgICAgICBhbmQgaXNpbnN0YW5jZShub2RlLnZhbHVlLCBhc3QuQXR0cmlidXRlKQ0KICAgICAgICAgICAgYW5kIG5vZGUudmFsdWUuYXR0ciA9PSAiZHQiDQogICAgICAgICk6DQogICAgICAgICAgICByZXR1cm4gKA0KICAgICAgICAgICAgICAgICJEYXRlVHlwZUVycm9yOiBLZWVwIHBhbmRhcyBkYXRldGltZSBjb2x1bW5zIGFzIHRpbWVzdGFtcHM7ICINCiAgICAgICAgICAgICAgICAiZG8gbm90IGNvbnZlcnQgdGhlbSB3aXRoIGAuZHQuZGF0ZWAuIg0KICAgICAgICAgICAgKQ0KICAgIHJldHVybiBOb25lDQoNCg0KIyBUb3AtbGV2ZWwgbW9kdWxlcyB0aGF0IExMTS1nZW5lcmF0ZWQgY29kZSBpcyBhbGxvd2VkIHRvIGltcG9ydC4NCl9BTExPV0VEX0lNUE9SVFMgPSBmcm96ZW5zZXQoew0KICAgICJwYW5kYXMiLCAibnVtcHkiLCAibWF0aCIsICJyZSIsICJkYXRldGltZSIsICJjYWxlbmRhciIsICJ1bmljb2RlZGF0YSIsDQogICAgImNvbGxlY3Rpb25zIiwgIml0ZXJ0b29scyIsICJmdW5jdG9vbHMiLCAic3RhdGlzdGljcyIsICJkZWNpbWFsIiwNCiAgICAiZnJhY3Rpb25zIiwgInN0cmluZyIsICJvcGVyYXRvciIsICJ0eXBpbmciLCAianNvbiIsDQp9KQ0KDQoNCmRlZiBfc2FmZV9pbXBvcnQobmFtZSwgZ2xvYmFscz1Ob25lLCBsb2NhbHM9Tm9uZSwgZnJvbWxpc3Q9KCksIGxldmVsPTApOg0KICAgICIiIlJlc3RyaWN0ZWQgcmVwbGFjZW1lbnQgZm9yIGBgX19pbXBvcnRfX2BgICh3aGl0ZWxpc3RlZCBtb2R1bGVzIG9ubHkpLiIiIg0KICAgIHJvb3QgPSBuYW1lLnNwbGl0KCIuIilbMF0NCiAgICBpZiBsZXZlbCAhPSAwIG9yIHJvb3Qgbm90IGluIF9BTExPV0VEX0lNUE9SVFM6DQogICAgICAgIHJhaXNlIEltcG9ydEVycm9yKGYiSW1wb3J0IG9mICd7bmFtZX0nIGlzIG5vdCBhbGxvd2VkIikNCiAgICByZXR1cm4gYnVpbHRpbnMuX19pbXBvcnRfXyhuYW1lLCBnbG9iYWxzLCBsb2NhbHMsIGZyb21saXN0LCBsZXZlbCkNCg0KDQpkZWYgX2NyZWF0ZV9zYWZlX2dsb2JhbHModGFibGVzOiBEaWN0W3N0ciwgcGQuRGF0YUZyYW1lXSwNCiAgICAgICAgICAgICAgICAgICAgICAgICAgYXNfb2ZfZGF0ZTogc3RyKSAtPiBEaWN0W3N0ciwgQW55XToNCiAgICAiIiJDcmVhdGUgdGhlIGV4ZWN1dGlvbiBnbG9iYWxzIHdpdGggcHJlLWxvYWRlZCBEYXRhRnJhbWVzIGFuZCBzYWZlIGJ1aWx0aW5zLiIiIg0KICAgIGltcG9ydCBkYXRldGltZSBhcyBkdA0KICAgIGltcG9ydCBtYXRoDQogICAgaW1wb3J0IG51bXB5IGFzIG5wDQoNCiAgICBzYWZlX2dsb2JhbHMgPSB7DQogICAgICAgICJfX2J1aWx0aW5zX18iOiB7DQogICAgICAgICAgICAiX19pbXBvcnRfXyI6IF9zYWZlX2ltcG9ydCwNCiAgICAgICAgICAgICJhYnMiOiBhYnMsDQogICAgICAgICAgICAiYWxsIjogYWxsLA0KICAgICAgICAgICAgImFueSI6IGFueSwNCiAgICAgICAgICAgICJib29sIjogYm9vbCwNCiAgICAgICAgICAgICJkaWN0IjogZGljdCwNCiAgICAgICAgICAgICJlbnVtZXJhdGUiOiBlbnVtZXJhdGUsDQogICAgICAgICAgICAiZmlsdGVyIjogZmlsdGVyLA0KICAgICAgICAgICAgImZsb2F0IjogZmxvYXQsDQogICAgICAgICAgICAiZm9ybWF0IjogZm9ybWF0LA0KICAgICAgICAgICAgImZyb3plbnNldCI6IGZyb3plbnNldCwNCiAgICAgICAgICAgICJoYXNhdHRyIjogaGFzYXR0ciwNCiAgICAgICAgICAgICJpbnQiOiBpbnQsDQogICAgICAgICAgICAiaXNpbnN0YW5jZSI6IGlzaW5zdGFuY2UsDQogICAgICAgICAgICAibGVuIjogbGVuLA0KICAgICAgICAgICAgImxpc3QiOiBsaXN0LA0KICAgICAgICAgICAgIm1hcCI6IG1hcCwNCiAgICAgICAgICAgICJtYXgiOiBtYXgsDQogICAgICAgICAgICAibWluIjogbWluLA0KICAgICAgICAgICAgInByaW50IjogcHJpbnQsDQogICAgICAgICAgICAicmFuZ2UiOiByYW5nZSwNCiAgICAgICAgICAgICJyZXByIjogcmVwciwNCiAgICAgICAgICAgICJyZXZlcnNlZCI6IHJldmVyc2VkLA0KICAgICAgICAgICAgInJvdW5kIjogcm91bmQsDQogICAgICAgICAgICAic2V0Ijogc2V0LA0KICAgICAgICAgICAgInNsaWNlIjogc2xpY2UsDQogICAgICAgICAgICAic29ydGVkIjogc29ydGVkLA0KICAgICAgICAgICAgInN0ciI6IHN0ciwNCiAgICAgICAgICAgICJzdW0iOiBzdW0sDQogICAgICAgICAgICAidHVwbGUiOiB0dXBsZSwNCiAgICAgICAgICAgICJ0eXBlIjogdHlwZSwNCiAgICAgICAgICAgICJ6aXAiOiB6aXAsDQogICAgICAgICAgICAiTm9uZSI6IE5vbmUsDQogICAgICAgICAgICAiVHJ1ZSI6IFRydWUsDQogICAgICAgICAgICAiRmFsc2UiOiBGYWxzZSwNCiAgICAgICAgICAgICJWYWx1ZUVycm9yIjogVmFsdWVFcnJvciwNCiAgICAgICAgICAgICJUeXBlRXJyb3IiOiBUeXBlRXJyb3IsDQogICAgICAgICAgICAiS2V5RXJyb3IiOiBLZXlFcnJvciwNCiAgICAgICAgICAgICJJbmRleEVycm9yIjogSW5kZXhFcnJvciwNCiAgICAgICAgICAgICJaZXJvRGl2aXNpb25FcnJvciI6IFplcm9EaXZpc2lvbkVycm9yLA0KICAgICAgICAgICAgIkV4Y2VwdGlvbiI6IEV4Y2VwdGlvbiwNCiAgICAgICAgICAgICJBdHRyaWJ1dGVFcnJvciI6IEF0dHJpYnV0ZUVycm9yLA0KICAgICAgICAgICAgIkltcG9ydEVycm9yIjogSW1wb3J0RXJyb3IsDQogICAgICAgICAgICAiTmFtZUVycm9yIjogTmFtZUVycm9yLA0KICAgICAgICAgICAgImdldGF0dHIiOiBnZXRhdHRyLA0KICAgICAgICAgICAgInBvdyI6IHBvdywNCiAgICAgICAgICAgICJkaXZtb2QiOiBkaXZtb2QsDQogICAgICAgICAgICAibmV4dCI6IG5leHQsDQogICAgICAgICAgICAiaXRlciI6IGl0ZXIsDQogICAgICAgICAgICAiY2FsbGFibGUiOiBjYWxsYWJsZSwNCiAgICAgICAgfSwNCiAgICAgICAgIyBTdGFuZGFyZCBsaWJyYXJpZXMNCiAgICAgICAgInBkIjogcGQsDQogICAgICAgICJucCI6IG5wLA0KICAgICAgICAibWF0aCI6IG1hdGgsDQogICAgICAgICJkYXRldGltZSI6IGR0LA0KICAgICAgICAiZHQiOiBkdCwNCiAgICAgICAgImRhdGUiOiBkdC5kYXRlLA0KICAgICAgICAidGltZWRlbHRhIjogZHQudGltZWRlbHRhLA0KICAgICAgICAicmUiOiBfX2ltcG9ydF9fKCJyZSIpLA0KICAgICAgICAjIEFzLW9mIGRhdGUNCiAgICAgICAgImFzX29mX2RhdGUiOiBhc19vZl9kYXRlLA0KICAgICAgICAiYXNfb2ZfZGF0ZV9vYmoiOiBkdC5kYXRlLmZyb21pc29mb3JtYXQoYXNfb2ZfZGF0ZSksDQogICAgICAgICMgUmVzdWx0IHBsYWNlaG9sZGVycw0KICAgICAgICAicmVzdWx0IjogTm9uZSwNCiAgICAgICAgImFuc3dlcl90eXBlIjogImFic3RhaW4iLA0KICAgIH0NCg0KICAgICMgTG9hZCB0YWJsZXMgdW5kZXIgdGhlaXIgb3JpZ2luYWwgdmFsaWQgbmFtZXMgYW5kIHVuaXF1ZSBzYW5pdGl6ZWQgbmFtZXMuDQogICAgdmFyaWFibGVfbmFtZXMgPSBfdGFibGVfdmFyaWFibGVfbWFwcGluZyh0YWJsZXMpDQogICAgZm9yIG5hbWUsIGRmIGluIHRhYmxlcy5pdGVtcygpOg0KICAgICAgICAjIE1ha2UgYSBjb3B5IHRvIHByZXZlbnQgbW9kaWZpY2F0aW9uIG9mIGNhY2hlZCBkYXRhDQogICAgICAgIGRmX2NvcHkgPSBkZi5jb3B5KCkNCiAgICAgICAgdmFyaWFibGVfbmFtZSA9IHZhcmlhYmxlX25hbWVzW25hbWVdDQogICAgICAgIHNhZmVfZ2xvYmFsc1t2YXJpYWJsZV9uYW1lXSA9IGRmX2NvcHkNCiAgICAgICAgaWYgKA0KICAgICAgICAgICAgdmFyaWFibGVfbmFtZSA9PSBuYW1lDQogICAgICAgICAgICBhbmQgbmFtZS5pc2lkZW50aWZpZXIoKQ0KICAgICAgICAgICAgYW5kIG5vdCBrZXl3b3JkLmlza2V5d29yZChuYW1lKQ0KICAgICAgICAgICAgYW5kIG5hbWUgbm90IGluIF9SRVNFUlZFRF9FWEVDX05BTUVTDQogICAgICAgICk6DQogICAgICAgICAgICBzYWZlX2dsb2JhbHNbbmFtZV0gPSBkZl9jb3B5DQoNCiAgICBpZiBsZW4odGFibGVzKSA9PSAxOg0KICAgICAgICBvbmx5X3RhYmxlID0gbmV4dChpdGVyKHRhYmxlcykpDQogICAgICAgIHNhZmVfZ2xvYmFsc1siZGYiXSA9IHNhZmVfZ2xvYmFsc1t2YXJpYWJsZV9uYW1lc1tvbmx5X3RhYmxlXV0NCg0KICAgIHJldHVybiBzYWZlX2dsb2JhbHMNCg0KDQpjbGFzcyBDb2RlRXhlY3V0b3I6DQogICAgIiIiRXhlY3V0ZSBMTE0tZ2VuZXJhdGVkIGNvZGUgaW4gYSBzYW5kYm94ZWQgZW52aXJvbm1lbnQuIiIiDQoNCiAgICBkZWYgX19pbml0X18oc2VsZiwgdGltZW91dF9zZWNvbmRzOiBpbnQgPSAzMCk6DQogICAgICAgIHNlbGYudGltZW91dF9zZWNvbmRzID0gdGltZW91dF9zZWNvbmRzDQoNCiAgICBkZWYgZXhlY3V0ZSgNCiAgICAgICAgc2VsZiwNCiAgICAgICAgY29kZV9vcl9yZXNwb25zZTogc3RyLA0KICAgICAgICB0YWJsZXM6IERpY3Rbc3RyLCBwZC5EYXRhRnJhbWVdLA0KICAgICAgICBhc19vZl9kYXRlOiBzdHIsDQogICAgICAgIGV4dHJhY3RfY29kZTogYm9vbCA9IFRydWUsDQogICAgKSAtPiBFeGVjdXRpb25SZXN1bHQ6DQogICAgICAgICIiIg0KICAgICAgICBFeGVjdXRlIGNvZGUgd2l0aCBwcmUtbG9hZGVkIERhdGFGcmFtZXMuDQoNCiAgICAgICAgUGFyYW1ldGVycw0KICAgICAgICAtLS0tLS0tLS0tDQogICAgICAgIGNvZGVfb3JfcmVzcG9uc2UgOiBzdHINCiAgICAgICAgICAgIEVpdGhlciByYXcgUHl0aG9uIGNvZGUgb3IgYW4gTExNIHJlc3BvbnNlIGNvbnRhaW5pbmcgY29kZS4NCiAgICAgICAgdGFibGVzIDogRGljdFtzdHIsIHBkLkRhdGFGcmFtZV0NCiAgICAgICAgICAgIEJ1c2luZXNzIGRhdGEgdGFibGVzIHRvIG1ha2UgYXZhaWxhYmxlIGluIHRoZSBleGVjdXRpb24gZW52Lg0KICAgICAgICBhc19vZl9kYXRlIDogc3RyDQogICAgICAgICAgICBUaGUgYXMtb2YgZGF0ZSBpbiBJU08gZm9ybWF0Lg0KICAgICAgICBleHRyYWN0X2NvZGUgOiBib29sDQogICAgICAgICAgICBJZiBUcnVlLCBhdHRlbXB0IHRvIGV4dHJhY3QgY29kZSBmcm9tIHRoZSByZXNwb25zZS4NCg0KICAgICAgICBSZXR1cm5zDQogICAgICAgIC0tLS0tLS0NCiAgICAgICAgRXhlY3V0aW9uUmVzdWx0DQogICAgICAgICAgICBUaGUgZXhlY3V0aW9uIHJlc3VsdCB3aXRoIGFuc3dlciwgdHlwZSwgc3Rkb3V0L3N0ZGVyci4NCiAgICAgICAgIiIiDQogICAgICAgIHJlc3VsdCA9IEV4ZWN1dGlvblJlc3VsdCgpDQoNCiAgICAgICAgIyBFeHRyYWN0IGNvZGUgZnJvbSBMTE0gcmVzcG9uc2UNCiAgICAgICAgaWYgZXh0cmFjdF9jb2RlOg0KICAgICAgICAgICAgY29kZSA9IF9leHRyYWN0X2NvZGVfZnJvbV9yZXNwb25zZShjb2RlX29yX3Jlc3BvbnNlKQ0KICAgICAgICBlbHNlOg0KICAgICAgICAgICAgY29kZSA9IGNvZGVfb3JfcmVzcG9uc2UNCiAgICAgICAgcmVzdWx0LmNvZGUgPSBjb2RlDQoNCiAgICAgICAgaWYgbm90IGNvZGUuc3RyaXAoKToNCiAgICAgICAgICAgIHJlc3VsdC5lcnJvciA9ICJObyBjb2RlIHRvIGV4ZWN1dGUiDQogICAgICAgICAgICByZXR1cm4gcmVzdWx0DQoNCiAgICAgICAgZmlsZV9yZWFkX3Zpb2xhdGlvbiA9IF9maWxlX3JlYWRfdmlvbGF0aW9uKGNvZGUpDQogICAgICAgIGlmIGZpbGVfcmVhZF92aW9sYXRpb246DQogICAgICAgICAgICByZXN1bHQuZXJyb3IgPSBmIkZpbGVBY2Nlc3NFcnJvcjoge2ZpbGVfcmVhZF92aW9sYXRpb259Ig0KICAgICAgICAgICAgcmV0dXJuIHJlc3VsdA0KDQogICAgICAgIGRhdGVfY29udmVyc2lvbl92aW9sYXRpb24gPSBfZGF0ZV9jb252ZXJzaW9uX3Zpb2xhdGlvbihjb2RlKQ0KICAgICAgICBpZiBkYXRlX2NvbnZlcnNpb25fdmlvbGF0aW9uOg0KICAgICAgICAgICAgcmVzdWx0LmVycm9yID0gZGF0ZV9jb252ZXJzaW9uX3Zpb2xhdGlvbg0KICAgICAgICAgICAgcmV0dXJuIHJlc3VsdA0KDQogICAgICAgICMgQnVpbGQgZXhlY3V0aW9uIGVudmlyb25tZW50DQogICAgICAgIGV4ZWNfZ2xvYmFscyA9IF9jcmVhdGVfc2FmZV9nbG9iYWxzKHRhYmxlcywgYXNfb2ZfZGF0ZSkNCg0KICAgICAgICAjIENhcHR1cmUgc3Rkb3V0L3N0ZGVycg0KICAgICAgICBzdGRvdXRfY2FwdHVyZSA9IGlvLlN0cmluZ0lPKCkNCiAgICAgICAgc3RkZXJyX2NhcHR1cmUgPSBpby5TdHJpbmdJTygpDQoNCiAgICAgICAgdHJ5Og0KICAgICAgICAgICAgd2l0aCByZWRpcmVjdF9zdGRvdXQoc3Rkb3V0X2NhcHR1cmUpLCByZWRpcmVjdF9zdGRlcnIoc3RkZXJyX2NhcHR1cmUpOg0KICAgICAgICAgICAgICAgIGV4ZWMoY29kZSwgZXhlY19nbG9iYWxzKQ0KDQogICAgICAgICAgICByZXN1bHQuc3Rkb3V0ID0gc3Rkb3V0X2NhcHR1cmUuZ2V0dmFsdWUoKQ0KICAgICAgICAgICAgcmVzdWx0LnN0ZGVyciA9IHN0ZGVycl9jYXB0dXJlLmdldHZhbHVlKCkNCg0KICAgICAgICAgICAgIyBFeHRyYWN0IHJlc3VsdHMNCiAgICAgICAgICAgIHJlc3VsdC5yZXN1bHQgPSBleGVjX2dsb2JhbHMuZ2V0KCJyZXN1bHQiLCBOb25lKQ0KICAgICAgICAgICAgcmVzdWx0LmFuc3dlcl90eXBlID0gZXhlY19nbG9iYWxzLmdldCgiYW5zd2VyX3R5cGUiLCAiYWJzdGFpbiIpDQogICAgICAgICAgICByZXN1bHQuc3VjY2VzcyA9IFRydWUNCg0KICAgICAgICAgICAgIyBWYWxpZGF0ZSByZXN1bHQNCiAgICAgICAgICAgIGlmIHJlc3VsdC5yZXN1bHQgaXMgTm9uZSBhbmQgcmVzdWx0LmFuc3dlcl90eXBlIG5vdCBpbiAoInVuYW5zd2VyYWJsZSIsICJhYnN0YWluIik6DQogICAgICAgICAgICAgICAgcmVzdWx0LmFuc3dlcl90eXBlID0gImFic3RhaW4iDQogICAgICAgICAgICAgICAgbG9nZ2VyLndhcm5pbmcoIkNvZGUgcHJvZHVjZWQgTm9uZSByZXN1bHQsIHNldHRpbmcgdG8gYWJzdGFpbiIpDQoNCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOg0KICAgICAgICAgICAgcmVzdWx0LnN0ZG91dCA9IHN0ZG91dF9jYXB0dXJlLmdldHZhbHVlKCkNCiAgICAgICAgICAgIHJlc3VsdC5zdGRlcnIgPSBzdGRlcnJfY2FwdHVyZS5nZXR2YWx1ZSgpDQogICAgICAgICAgICByZXN1bHQuZXJyb3IgPSBmInt0eXBlKGUpLl9fbmFtZV9ffToge3N0cihlKX1cbnt0cmFjZWJhY2suZm9ybWF0X2V4YygpfSINCiAgICAgICAgICAgIHJlc3VsdC5zdWNjZXNzID0gRmFsc2UNCiAgICAgICAgICAgIGxvZ2dlci53YXJuaW5nKCJDb2RlIGV4ZWN1dGlvbiBmYWlsZWQ6ICVzIiwgc3RyKGUpKQ0KDQogICAgICAgIHJldHVybiByZXN1bHQNCg0KICAgIGRlZiBnZXRfdGFibGVfdmFyX21hcHBpbmcoc2VsZiwgdGFibGVzOiBEaWN0W3N0ciwgcGQuRGF0YUZyYW1lXSkgLT4gRGljdFtzdHIsIHN0cl06DQogICAgICAgICIiIkdldCB0aGUgbWFwcGluZyBmcm9tIHRhYmxlIG5hbWVzIHRvIHZhcmlhYmxlIG5hbWVzLiIiIg0KICAgICAgICBtYXBwaW5nID0gX3RhYmxlX3ZhcmlhYmxlX21hcHBpbmcodGFibGVzKQ0KICAgICAgICBpZiBsZW4obWFwcGluZykgPT0gMToNCiAgICAgICAgICAgIG9ubHlfdGFibGUgPSBuZXh0KGl0ZXIobWFwcGluZykpDQogICAgICAgICAgICBtYXBwaW5nW29ubHlfdGFibGVdID0gImRmIg0KICAgICAgICByZXR1cm4gbWFwcGluZw0KDQogICAgZGVmIGdldF9oZWFkX3Jvd3Moc2VsZiwgdGFibGVzOiBEaWN0W3N0ciwgcGQuRGF0YUZyYW1lXSwNCiAgICAgICAgICAgICAgICAgICAgICAgbjogaW50ID0gMykgLT4gRGljdFtzdHIsIHN0cl06DQogICAgICAgICIiIkdldCB0aGUgZmlyc3QgbiByb3dzIG9mIGVhY2ggdGFibGUgYXMgYSBzdHJpbmcuIiIiDQogICAgICAgIHJlc3VsdCA9IHt9DQogICAgICAgIGZvciBuYW1lLCBkZiBpbiB0YWJsZXMuaXRlbXMoKToNCiAgICAgICAgICAgIHJlc3VsdFtuYW1lXSA9IGRmLmhlYWQobikudG9fc3RyaW5nKGluZGV4PUZhbHNlKQ0KICAgICAgICByZXR1cm4gcmVzdWx0DQo='
source_files['src/modules/confidence_gate.py'] = 'IiIiDQpjb25maWRlbmNlX2dhdGUucHkg4oCUIENvbmZpZGVuY2UtQmFzZWQgQWJzdGVudGlvbiBmb3IgRGF0YU1hdGUNCg0KU2luY2UgUEEgcGVuYWxpemVzIHdyb25nIGFuc3dlcnMgKC0xKSBidXQgYWJzdGFpbiA9IDAsIHRoaXMgbW9kdWxlDQppbXBsZW1lbnRzIGhldXJpc3RpY3MgdG8gZGVjaWRlIHdoZW4gdG8gYWJzdGFpbiByYXRoZXIgdGhhbiByaXNrDQphIHdyb25nIGFuc3dlci4NCiIiIg0KDQppbXBvcnQganNvbg0KaW1wb3J0IGxvZ2dpbmcNCmltcG9ydCBtYXRoDQpmcm9tIGRhdGV0aW1lIGltcG9ydCBkYXRlDQpmcm9tIHR5cGluZyBpbXBvcnQgQW55LCBEaWN0LCBMaXN0LCBPcHRpb25hbA0KDQppbXBvcnQgcGFuZGFzIGFzIHBkDQoNCmZyb20gLnF1ZXN0aW9uX3VuZGVyc3RhbmRpbmcgaW1wb3J0IFF1ZXN0aW9uSW5mbw0KDQpsb2dnZXIgPSBsb2dnaW5nLmdldExvZ2dlcihfX25hbWVfXykNCg0KDQpjbGFzcyBDb25maWRlbmNlR2F0ZToNCiAgICAiIiINCiAgICBEZWNpZGVzIHdoZXRoZXIgdG8gc3VibWl0IGFuIGFuc3dlciBvciBhYnN0YWluIGJhc2VkIG9uDQogICAgdmFyaW91cyBjb25maWRlbmNlIHNpZ25hbHMuDQogICAgIiIiDQoNCiAgICBkZWYgX19pbml0X18oDQogICAgICAgIHNlbGYsDQogICAgICAgIG1pbl9jb25maWRlbmNlOiBmbG9hdCA9IDAuMywNCiAgICAgICAgZW5hYmxlX2R1YWxfcnVuOiBib29sID0gRmFsc2UsDQogICAgKToNCiAgICAgICAgIiIiDQogICAgICAgIFBhcmFtZXRlcnMNCiAgICAgICAgLS0tLS0tLS0tLQ0KICAgICAgICBtaW5fY29uZmlkZW5jZSA6IGZsb2F0DQogICAgICAgICAgICBNaW5pbXVtIGNvbmZpZGVuY2Ugc2NvcmUgdG8gc3VibWl0IGFuIGFuc3dlciAoMC0xKS4NCiAgICAgICAgZW5hYmxlX2R1YWxfcnVuIDogYm9vbA0KICAgICAgICAgICAgSWYgVHJ1ZSwgc3VwcG9ydCBkdWFsLXJ1biBjb25zaXN0ZW5jeSBjaGVja2luZy4NCiAgICAgICAgIiIiDQogICAgICAgIHNlbGYubWluX2NvbmZpZGVuY2UgPSBtaW5fY29uZmlkZW5jZQ0KICAgICAgICBzZWxmLmVuYWJsZV9kdWFsX3J1biA9IGVuYWJsZV9kdWFsX3J1bg0KDQogICAgZGVmIGV2YWx1YXRlKA0KICAgICAgICBzZWxmLA0KICAgICAgICBhbnN3ZXI6IERpY3Rbc3RyLCBBbnldLA0KICAgICAgICBxdWVzdGlvbl9pbmZvOiBPcHRpb25hbFtRdWVzdGlvbkluZm9dID0gTm9uZSwNCiAgICAgICAgZXhlY3V0aW9uX3JldHJpZXM6IGludCA9IDAsDQogICAgICAgIG1heF9yZXRyaWVzOiBpbnQgPSAzLA0KICAgICAgICBzZWNvbmRfYW5zd2VyOiBPcHRpb25hbFtEaWN0W3N0ciwgQW55XV0gPSBOb25lLA0KICAgICkgLT4gRGljdFtzdHIsIEFueV06DQogICAgICAgICIiIg0KICAgICAgICBFdmFsdWF0ZSB3aGV0aGVyIHRvIHN1Ym1pdCB0aGUgYW5zd2VyIG9yIGFic3RhaW4uDQoNCiAgICAgICAgUGFyYW1ldGVycw0KICAgICAgICAtLS0tLS0tLS0tDQogICAgICAgIGFuc3dlciA6IERpY3Rbc3RyLCBBbnldDQogICAgICAgICAgICBUaGUgZm9ybWF0dGVkIGFuc3dlciBkaWN0ICh3aXRoICJ0eXBlIiBhbmQgImFuc3dlciIga2V5cykuDQogICAgICAgIHF1ZXN0aW9uX2luZm8gOiBRdWVzdGlvbkluZm8sIG9wdGlvbmFsDQogICAgICAgICAgICBBbmFseXplZCBxdWVzdGlvbiBpbmZvcm1hdGlvbi4NCiAgICAgICAgZXhlY3V0aW9uX3JldHJpZXMgOiBpbnQNCiAgICAgICAgICAgIE51bWJlciBvZiByZXRyaWVzIHRoYXQgd2VyZSBuZWVkZWQuDQogICAgICAgIG1heF9yZXRyaWVzIDogaW50DQogICAgICAgICAgICBNYXhpbXVtIHJldHJpZXMgYWxsb3dlZC4NCiAgICAgICAgc2Vjb25kX2Fuc3dlciA6IERpY3Rbc3RyLCBBbnldLCBvcHRpb25hbA0KICAgICAgICAgICAgQSBzZWNvbmQgYW5zd2VyIGZyb20gYSBkdWFsLXJ1biAoZm9yIGNvbnNpc3RlbmN5IGNoZWNrKS4NCg0KICAgICAgICBSZXR1cm5zDQogICAgICAgIC0tLS0tLS0NCiAgICAgICAgRGljdFtzdHIsIEFueV0NCiAgICAgICAgICAgIEVpdGhlciB0aGUgb3JpZ2luYWwgYW5zd2VyIG9yIGFuIGFic3RhaW4gZW50cnkuDQogICAgICAgICIiIg0KICAgICAgICAjIEFscmVhZHkgYWJzdGFpbiBvciB1bmFuc3dlcmFibGUg4oCUIHBhc3MgdGhyb3VnaA0KICAgICAgICBpZiBhbnN3ZXIuZ2V0KCJ0eXBlIikgaW4gKCJhYnN0YWluIiwgInVuYW5zd2VyYWJsZSIpOg0KICAgICAgICAgICAgcmV0dXJuIGFuc3dlcg0KDQogICAgICAgIGNvbmZpZGVuY2UgPSAxLjANCiAgICAgICAgcmVhc29ucyA9IFtdDQoNCiAgICAgICAgIyAxLiBQZW5hbHR5IGZvciByZXRyaWVzIChtb3JlIHJldHJpZXMgPSBsZXNzIGNvbmZpZGVuY2UpDQogICAgICAgIGlmIGV4ZWN1dGlvbl9yZXRyaWVzID4gMDoNCiAgICAgICAgICAgIHJldHJ5X3BlbmFsdHkgPSAwLjE1ICogZXhlY3V0aW9uX3JldHJpZXMNCiAgICAgICAgICAgIGNvbmZpZGVuY2UgLT0gcmV0cnlfcGVuYWx0eQ0KICAgICAgICAgICAgcmVhc29ucy5hcHBlbmQoZiJyZXRyaWVzPXtleGVjdXRpb25fcmV0cmllc30iKQ0KDQogICAgICAgICMgMi4gUXVlc3Rpb24gY2xhc3NpZmljYXRpb24gY29uZmlkZW5jZQ0KICAgICAgICBpZiBxdWVzdGlvbl9pbmZvIGFuZCBxdWVzdGlvbl9pbmZvLmNvbmZpZGVuY2UgPCAwLjQ6DQogICAgICAgICAgICBjb25maWRlbmNlIC09IDAuMg0KICAgICAgICAgICAgcmVhc29ucy5hcHBlbmQoZiJsb3dfdHlwZV9jb25maWRlbmNlPXtxdWVzdGlvbl9pbmZvLmNvbmZpZGVuY2U6LjJmfSIpDQoNCiAgICAgICAgIyAzLiBDaGVjayBmb3IgaW1wbGF1c2libGUgdmFsdWVzDQogICAgICAgIGltcGxhdXNpYmxlID0gc2VsZi5fY2hlY2tfcGxhdXNpYmlsaXR5KGFuc3dlcikNCiAgICAgICAgaWYgaW1wbGF1c2libGU6DQogICAgICAgICAgICBjb25maWRlbmNlIC09IDAuMw0KICAgICAgICAgICAgcmVhc29ucy5hcHBlbmQoZiJpbXBsYXVzaWJsZToge2ltcGxhdXNpYmxlfSIpDQoNCiAgICAgICAgIyA0LiBEdWFsLXJ1biBjb25zaXN0ZW5jeSBjaGVjaw0KICAgICAgICBpZiBzZWxmLmVuYWJsZV9kdWFsX3J1biBhbmQgc2Vjb25kX2Fuc3dlciBpcyBub3QgTm9uZToNCiAgICAgICAgICAgIGlmIG5vdCBzZWxmLl9hbnN3ZXJzX21hdGNoKGFuc3dlciwgc2Vjb25kX2Fuc3dlcik6DQogICAgICAgICAgICAgICAgY29uZmlkZW5jZSAtPSAwLjUNCiAgICAgICAgICAgICAgICByZWFzb25zLmFwcGVuZCgiZHVhbF9ydW5faW5jb25zaXN0ZW50IikNCg0KICAgICAgICAjIDUuIFR5cGUgbWlzbWF0Y2ggYmV0d2VlbiBleHBlY3RlZCBhbmQgYWN0dWFsDQogICAgICAgIGlmIHF1ZXN0aW9uX2luZm86DQogICAgICAgICAgICBpZiAocXVlc3Rpb25faW5mby5leHBlY3RlZF90eXBlICE9IGFuc3dlci5nZXQoInR5cGUiKSBhbmQNCiAgICAgICAgICAgICAgICBxdWVzdGlvbl9pbmZvLmNvbmZpZGVuY2UgPiAwLjUpOg0KICAgICAgICAgICAgICAgIGNvbmZpZGVuY2UgLT0gMC4yDQogICAgICAgICAgICAgICAgcmVhc29ucy5hcHBlbmQoDQogICAgICAgICAgICAgICAgICAgIGYidHlwZV9taXNtYXRjaDogZXhwZWN0ZWQ9e3F1ZXN0aW9uX2luZm8uZXhwZWN0ZWRfdHlwZX0sICINCiAgICAgICAgICAgICAgICAgICAgZiJnb3Q9e2Fuc3dlci5nZXQoJ3R5cGUnKX0iDQogICAgICAgICAgICAgICAgKQ0KDQogICAgICAgICMgRGVjaXNpb24NCiAgICAgICAgaWYgY29uZmlkZW5jZSA8IHNlbGYubWluX2NvbmZpZGVuY2U6DQogICAgICAgICAgICBsb2dnZXIuaW5mbygiQWJzdGFpbmluZyAoY29uZmlkZW5jZT0lLjJmLCByZWFzb25zPSVzKSIsIGNvbmZpZGVuY2UsIHJlYXNvbnMpDQogICAgICAgICAgICByZXR1cm4gew0KICAgICAgICAgICAgICAgICJ0eXBlIjogImFic3RhaW4iLA0KICAgICAgICAgICAgICAgICJyZWFzb24iOiBmImxvd19jb25maWRlbmNlPXtjb25maWRlbmNlOi4yZn07ICIgKyAiOyAiLmpvaW4ocmVhc29ucyksDQogICAgICAgICAgICB9DQoNCiAgICAgICAgbG9nZ2VyLmRlYnVnKCJBY2NlcHRpbmcgYW5zd2VyIChjb25maWRlbmNlPSUuMmYpIiwgY29uZmlkZW5jZSkNCiAgICAgICAgcmV0dXJuIGFuc3dlcg0KDQogICAgZGVmIGV2YWx1YXRlX3J1bnMoDQogICAgICAgIHNlbGYsDQogICAgICAgIGFuc3dlcnM6IExpc3RbRGljdFtzdHIsIEFueV1dLA0KICAgICAgICAqLA0KICAgICAgICBmYWlsZWRfcnVuczogaW50ID0gMCwNCiAgICAgICAgZXhlY3V0aW9uX3JldHJpZXM6IGludCA9IDAsDQogICAgICAgIHF1ZXN0aW9uX2luZm86IE9wdGlvbmFsW1F1ZXN0aW9uSW5mb10gPSBOb25lLA0KICAgICkgLT4gRGljdFtzdHIsIEFueV06DQogICAgICAgICIiIlNlbGVjdCB0aGUgbW9kYWwgYW5zd2VyIGFuZCBzY29yZSBhZ3JlZW1lbnQgYWNyb3NzIHJlcGVhdGVkIHJ1bnMuIiIiDQogICAgICAgIGlmIG5vdCBhbnN3ZXJzOg0KICAgICAgICAgICAgcmV0dXJuIHsidHlwZSI6ICJhYnN0YWluIiwgInJlYXNvbiI6ICJub19ydW5zIn0NCg0KICAgICAgICB2YWxpZF9hbnN3ZXJzID0gWw0KICAgICAgICAgICAgYW5zd2VyIGZvciBhbnN3ZXIgaW4gYW5zd2Vycw0KICAgICAgICAgICAgaWYgYW5zd2VyLmdldCgidHlwZSIpIG5vdCBpbiAoImFic3RhaW4iLCkNCiAgICAgICAgXQ0KICAgICAgICBpZiBub3QgdmFsaWRfYW5zd2VyczoNCiAgICAgICAgICAgIHJldHVybiB7DQogICAgICAgICAgICAgICAgInR5cGUiOiAiYWJzdGFpbiIsDQogICAgICAgICAgICAgICAgInJlYXNvbiI6IGYiYWxsX3J1bnNfZmFpbGVkPXtmYWlsZWRfcnVuc30iLA0KICAgICAgICAgICAgfQ0KDQogICAgICAgIGdyb3VwczogRGljdFtzdHIsIExpc3RbRGljdFtzdHIsIEFueV1dXSA9IHt9DQogICAgICAgIGZvciBhbnN3ZXIgaW4gdmFsaWRfYW5zd2VyczoNCiAgICAgICAgICAgIGtleSA9IHNlbGYuX2Fuc3dlcl9rZXkoYW5zd2VyKQ0KICAgICAgICAgICAgZ3JvdXBzLnNldGRlZmF1bHQoa2V5LCBbXSkuYXBwZW5kKGFuc3dlcikNCiAgICAgICAgd2lubmluZ19rZXksIHdpbm5pbmdfZ3JvdXAgPSBtYXgoDQogICAgICAgICAgICBncm91cHMuaXRlbXMoKSwNCiAgICAgICAgICAgIGtleT1sYW1iZGEgaXRlbTogKGxlbihpdGVtWzFdKSwgaXRlbVswXSksDQogICAgICAgICkNCiAgICAgICAgc2VsZWN0ZWQgPSBkaWN0KHdpbm5pbmdfZ3JvdXBbMF0pDQoNCiAgICAgICAgY29uZmlkZW5jZSA9IDEuMA0KICAgICAgICByZWFzb25zOiBMaXN0W3N0cl0gPSBbXQ0KICAgICAgICBydW5fY291bnQgPSBsZW4oYW5zd2VycykNCiAgICAgICAgZHVwbGljYXRlX2NvdW50ID0gbGVuKHdpbm5pbmdfZ3JvdXApDQogICAgICAgIGlmIGR1cGxpY2F0ZV9jb3VudCA+IDE6DQogICAgICAgICAgICBkdXBsaWNhdGVfYm9udXMgPSBtaW4oMC4yLCAwLjA1ICogKGR1cGxpY2F0ZV9jb3VudCAtIDEpKQ0KICAgICAgICAgICAgY29uZmlkZW5jZSArPSBkdXBsaWNhdGVfYm9udXMNCiAgICAgICAgICAgIHJlYXNvbnMuYXBwZW5kKGYibWF0Y2hpbmdfcnVucz17ZHVwbGljYXRlX2NvdW50fSIpDQoNCiAgICAgICAgaWYgbGVuKGdyb3VwcykgPiAxOg0KICAgICAgICAgICAgdHlwZV9jb3VudCA9IGxlbih7YW5zd2VyLmdldCgidHlwZSIpIGZvciBhbnN3ZXIgaW4gdmFsaWRfYW5zd2Vyc30pDQogICAgICAgICAgICBpZiB0eXBlX2NvdW50ID4gMToNCiAgICAgICAgICAgICAgICBjb25maWRlbmNlIC09IDAuNg0KICAgICAgICAgICAgICAgIHJlYXNvbnMuYXBwZW5kKGYiYW5zd2VyX3R5cGVfdmFyaWFuY2U9e3R5cGVfY291bnR9IikNCiAgICAgICAgICAgIGVsc2U6DQogICAgICAgICAgICAgICAgZGlzcGVyc2lvbl9wZW5hbHR5ID0gc2VsZi5fZGlzcGVyc2lvbl9wZW5hbHR5KHZhbGlkX2Fuc3dlcnMpDQogICAgICAgICAgICAgICAgY29uZmlkZW5jZSAtPSBkaXNwZXJzaW9uX3BlbmFsdHkNCiAgICAgICAgICAgICAgICByZWFzb25zLmFwcGVuZCgNCiAgICAgICAgICAgICAgICAgICAgZiJ2YWx1ZV92YXJpYW5jZT17bGVuKGdyb3Vwcyl9OyAiDQogICAgICAgICAgICAgICAgICAgIGYiZGlzcGVyc2lvbl9wZW5hbHR5PXtkaXNwZXJzaW9uX3BlbmFsdHk6LjJmfSINCiAgICAgICAgICAgICAgICApDQoNCiAgICAgICAgaWYgZmFpbGVkX3J1bnM6DQogICAgICAgICAgICBjb25maWRlbmNlIC09IG1pbigwLjcsIDAuMiAqIGZhaWxlZF9ydW5zKQ0KICAgICAgICAgICAgcmVhc29ucy5hcHBlbmQoZiJmYWlsZWRfcnVucz17ZmFpbGVkX3J1bnN9L3tydW5fY291bnR9IikNCg0KICAgICAgICBpZiBleGVjdXRpb25fcmV0cmllczoNCiAgICAgICAgICAgIGNvbmZpZGVuY2UgLT0gbWluKDAuNCwgMC4wNSAqIGV4ZWN1dGlvbl9yZXRyaWVzKQ0KICAgICAgICAgICAgcmVhc29ucy5hcHBlbmQoZiJyZXRyaWVzPXtleGVjdXRpb25fcmV0cmllc30iKQ0KDQogICAgICAgIGlmIHF1ZXN0aW9uX2luZm86DQogICAgICAgICAgICBpZiBxdWVzdGlvbl9pbmZvLmNvbmZpZGVuY2UgPCAwLjQ6DQogICAgICAgICAgICAgICAgY29uZmlkZW5jZSAtPSAwLjINCiAgICAgICAgICAgICAgICByZWFzb25zLmFwcGVuZChmImxvd190eXBlX2NvbmZpZGVuY2U9e3F1ZXN0aW9uX2luZm8uY29uZmlkZW5jZTouMmZ9IikNCiAgICAgICAgICAgIGlmICgNCiAgICAgICAgICAgICAgICBxdWVzdGlvbl9pbmZvLmV4cGVjdGVkX3R5cGUgIT0gc2VsZWN0ZWQuZ2V0KCJ0eXBlIikNCiAgICAgICAgICAgICAgICBhbmQgcXVlc3Rpb25faW5mby5jb25maWRlbmNlID4gMC41DQogICAgICAgICAgICApOg0KICAgICAgICAgICAgICAgIGNvbmZpZGVuY2UgLT0gMC4zDQogICAgICAgICAgICAgICAgcmVhc29ucy5hcHBlbmQoDQogICAgICAgICAgICAgICAgICAgIGYidHlwZV9taXNtYXRjaDogZXhwZWN0ZWQ9e3F1ZXN0aW9uX2luZm8uZXhwZWN0ZWRfdHlwZX0sICINCiAgICAgICAgICAgICAgICAgICAgZiJnb3Q9e3NlbGVjdGVkLmdldCgndHlwZScpfSINCiAgICAgICAgICAgICAgICApDQoNCiAgICAgICAgaW1wbGF1c2libGUgPSBzZWxmLl9jaGVja19wbGF1c2liaWxpdHkoc2VsZWN0ZWQpDQogICAgICAgIGlmIGltcGxhdXNpYmxlOg0KICAgICAgICAgICAgY29uZmlkZW5jZSAtPSAwLjMNCiAgICAgICAgICAgIHJlYXNvbnMuYXBwZW5kKGYiaW1wbGF1c2libGU6IHtpbXBsYXVzaWJsZX0iKQ0KDQogICAgICAgIGNvbmZpZGVuY2UgPSBtYXgoMC4wLCBtaW4oMS4wLCBjb25maWRlbmNlKSkNCiAgICAgICAgaWYgY29uZmlkZW5jZSA8IHNlbGYubWluX2NvbmZpZGVuY2U6DQogICAgICAgICAgICBsb2dnZXIuaW5mbygNCiAgICAgICAgICAgICAgICAiQWJzdGFpbmluZyBhZnRlciByZXBlYXRlZCBydW5zIChjb25maWRlbmNlPSUuMmYsIHJlYXNvbnM9JXMpIiwNCiAgICAgICAgICAgICAgICBjb25maWRlbmNlLA0KICAgICAgICAgICAgICAgIHJlYXNvbnMsDQogICAgICAgICAgICApDQogICAgICAgICAgICByZXR1cm4gew0KICAgICAgICAgICAgICAgICJ0eXBlIjogImFic3RhaW4iLA0KICAgICAgICAgICAgICAgICJyZWFzb24iOiBmImxvd19jb25maWRlbmNlPXtjb25maWRlbmNlOi4yZn07ICIgKyAiOyAiLmpvaW4ocmVhc29ucyksDQogICAgICAgICAgICB9DQogICAgICAgIHJldHVybiBzZWxlY3RlZA0KDQogICAgQHN0YXRpY21ldGhvZA0KICAgIGRlZiBfYW5zd2VyX2tleShhbnN3ZXI6IERpY3Rbc3RyLCBBbnldKSAtPiBzdHI6DQogICAgICAgICIiIkNyZWF0ZSBhIHN0YWJsZSBrZXkgZm9yIGNvbXBhcmluZyBKU09OLWNvbXBhdGlibGUgYW5zd2Vycy4iIiINCiAgICAgICAgcmV0dXJuIGpzb24uZHVtcHMoDQogICAgICAgICAgICB7InR5cGUiOiBhbnN3ZXIuZ2V0KCJ0eXBlIiksICJhbnN3ZXIiOiBhbnN3ZXIuZ2V0KCJhbnN3ZXIiKX0sDQogICAgICAgICAgICBlbnN1cmVfYXNjaWk9RmFsc2UsDQogICAgICAgICAgICBzb3J0X2tleXM9VHJ1ZSwNCiAgICAgICAgICAgIGRlZmF1bHQ9c3RyLA0KICAgICAgICApDQoNCiAgICBAc3RhdGljbWV0aG9kDQogICAgZGVmIF9kaXNwZXJzaW9uX3BlbmFsdHkoYW5zd2VyczogTGlzdFtEaWN0W3N0ciwgQW55XV0pIC0+IGZsb2F0Og0KICAgICAgICAiIiJQZW5hbGl6ZSBkaXNhZ3JlZW1lbnQgbW9yZSB3aGVuIG51bWVyaWMgcmVzdWx0cyBhcmUgZmFyIGFwYXJ0LiIiIg0KICAgICAgICBhbnN3ZXJfdHlwZSA9IGFuc3dlcnNbMF0uZ2V0KCJ0eXBlIikNCiAgICAgICAgdmFsdWVzID0gW2Fuc3dlci5nZXQoImFuc3dlciIpIGZvciBhbnN3ZXIgaW4gYW5zd2Vyc10NCiAgICAgICAgaWYgYW5zd2VyX3R5cGUgaW4gKCJudW1iZXIiLCAicGVyY2VudCIpOg0KICAgICAgICAgICAgdHJ5Og0KICAgICAgICAgICAgICAgIG51bWVyaWNfdmFsdWVzID0gW2Zsb2F0KHZhbHVlKSBmb3IgdmFsdWUgaW4gdmFsdWVzXQ0KICAgICAgICAgICAgZXhjZXB0IChUeXBlRXJyb3IsIFZhbHVlRXJyb3IpOg0KICAgICAgICAgICAgICAgIHJldHVybiAwLjYNCiAgICAgICAgICAgIHNjYWxlID0gbWF4KGFicyh2YWx1ZSkgZm9yIHZhbHVlIGluIG51bWVyaWNfdmFsdWVzICsgWzAuMF0pDQogICAgICAgICAgICByZWxhdGl2ZV9yYW5nZSA9ICgNCiAgICAgICAgICAgICAgICAobWF4KG51bWVyaWNfdmFsdWVzKSAtIG1pbihudW1lcmljX3ZhbHVlcykpIC8gbWF4KHNjYWxlLCAxLjApDQogICAgICAgICAgICApDQogICAgICAgICAgICByZXR1cm4gbWluKDAuNiwgbWF4KDAuMiwgcmVsYXRpdmVfcmFuZ2UpKQ0KICAgICAgICByZXR1cm4gbWluKDAuNiwgMC4yICogKGxlbihzZXQobWFwKHN0ciwgdmFsdWVzKSkpIC0gMSkpDQoNCiAgICBkZWYgX2NoZWNrX3BsYXVzaWJpbGl0eShzZWxmLCBhbnN3ZXI6IERpY3Rbc3RyLCBBbnldKSAtPiBPcHRpb25hbFtzdHJdOg0KICAgICAgICAiIiJDaGVjayBpZiB0aGUgYW5zd2VyIHNlZW1zIGltcGxhdXNpYmxlLiIiIg0KICAgICAgICBhbnN3ZXJfdHlwZSA9IGFuc3dlci5nZXQoInR5cGUiKQ0KICAgICAgICB2YWx1ZSA9IGFuc3dlci5nZXQoImFuc3dlciIpDQoNCiAgICAgICAgaWYgdmFsdWUgaXMgTm9uZToNCiAgICAgICAgICAgIHJldHVybiBOb25lDQoNCiAgICAgICAgaWYgYW5zd2VyX3R5cGUgPT0gIm51bWJlciI6DQogICAgICAgICAgICB0cnk6DQogICAgICAgICAgICAgICAgbnVtID0gZmxvYXQodmFsdWUpDQogICAgICAgICAgICAgICAgaWYgbWF0aC5pc25hbihudW0pIG9yIG1hdGguaXNpbmYobnVtKToNCiAgICAgICAgICAgICAgICAgICAgcmV0dXJuICJOYU4gb3IgSW5mIg0KICAgICAgICAgICAgICAgICMgRXh0cmVtZWx5IGxhcmdlIG51bWJlcnMgbWlnaHQgYmUgc3VzcGVjdA0KICAgICAgICAgICAgICAgIGlmIGFicyhudW0pID4gMWUxNToNCiAgICAgICAgICAgICAgICAgICAgcmV0dXJuIGYiZXh0cmVtZWx5X2xhcmdlPXtudW19Ig0KICAgICAgICAgICAgZXhjZXB0IChWYWx1ZUVycm9yLCBUeXBlRXJyb3IpOg0KICAgICAgICAgICAgICAgIHJldHVybiAibm90X2FfbnVtYmVyIg0KDQogICAgICAgIGVsaWYgYW5zd2VyX3R5cGUgPT0gInBlcmNlbnQiOg0KICAgICAgICAgICAgdHJ5Og0KICAgICAgICAgICAgICAgIG51bSA9IGZsb2F0KHZhbHVlKQ0KICAgICAgICAgICAgICAgIGlmIG1hdGguaXNuYW4obnVtKSBvciBtYXRoLmlzaW5mKG51bSk6DQogICAgICAgICAgICAgICAgICAgIHJldHVybiAiTmFOIG9yIEluZiINCiAgICAgICAgICAgICAgICAjIFBlcmNlbnRhZ2VzID4gMTAwMDAlIG9yIDwgLTEwMDAwJSBhcmUgc3VzcGVjdA0KICAgICAgICAgICAgICAgIGlmIGFicyhudW0pID4gMTAwMDA6DQogICAgICAgICAgICAgICAgICAgIHJldHVybiBmImV4dHJlbWVfcGVyY2VudD17bnVtfSINCiAgICAgICAgICAgIGV4Y2VwdCAoVmFsdWVFcnJvciwgVHlwZUVycm9yKToNCiAgICAgICAgICAgICAgICByZXR1cm4gIm5vdF9hX251bWJlciINCg0KICAgICAgICBlbGlmIGFuc3dlcl90eXBlID09ICJkYXRlIjoNCiAgICAgICAgICAgIGltcG9ydCByZQ0KICAgICAgICAgICAgcyA9IHN0cih2YWx1ZSkNCiAgICAgICAgICAgIGlmIG5vdCByZS5tYXRjaChyIl5cZHs0fS1cZHsyfSgtXGR7Mn0pPyQiLCBzKToNCiAgICAgICAgICAgICAgICByZXR1cm4gZiJpbnZhbGlkX2RhdGVfZm9ybWF0PXtzfSINCg0KICAgICAgICBlbGlmIGFuc3dlcl90eXBlID09ICJzdHJpbmciOg0KICAgICAgICAgICAgaWYgbGVuKHN0cih2YWx1ZSkpID4gNTAwOg0KICAgICAgICAgICAgICAgIHJldHVybiAic3RyaW5nX3Rvb19sb25nIg0KDQogICAgICAgIGVsaWYgYW5zd2VyX3R5cGUgPT0gImxpc3QiOg0KICAgICAgICAgICAgaWYgaXNpbnN0YW5jZSh2YWx1ZSwgbGlzdCkgYW5kIGxlbih2YWx1ZSkgPT0gMDoNCiAgICAgICAgICAgICAgICByZXR1cm4gImVtcHR5X2xpc3QiDQogICAgICAgICAgICBpZiBpc2luc3RhbmNlKHZhbHVlLCBsaXN0KSBhbmQgbGVuKHZhbHVlKSA+IDEwMDoNCiAgICAgICAgICAgICAgICByZXR1cm4gImxpc3RfdG9vX2xvbmciDQoNCiAgICAgICAgcmV0dXJuIE5vbmUNCg0KICAgIGRlZiBfYW5zd2Vyc19tYXRjaChzZWxmLCBhMTogRGljdFtzdHIsIEFueV0sIGEyOiBEaWN0W3N0ciwgQW55XSkgLT4gYm9vbDoNCiAgICAgICAgIiIiQ2hlY2sgaWYgdHdvIGFuc3dlcnMgYXJlIGNvbnNpc3RlbnQuIiIiDQogICAgICAgIGlmIGExLmdldCgidHlwZSIpICE9IGEyLmdldCgidHlwZSIpOg0KICAgICAgICAgICAgcmV0dXJuIEZhbHNlDQoNCiAgICAgICAgdjEgPSBhMS5nZXQoImFuc3dlciIpDQogICAgICAgIHYyID0gYTIuZ2V0KCJhbnN3ZXIiKQ0KDQogICAgICAgIGlmIHYxIGlzIE5vbmUgb3IgdjIgaXMgTm9uZToNCiAgICAgICAgICAgIHJldHVybiB2MSA9PSB2Mg0KDQogICAgICAgIGFuc3dlcl90eXBlID0gYTEuZ2V0KCJ0eXBlIikNCg0KICAgICAgICBpZiBhbnN3ZXJfdHlwZSBpbiAoIm51bWJlciIsICJwZXJjZW50Iik6DQogICAgICAgICAgICB0cnk6DQogICAgICAgICAgICAgICAgcmV0dXJuIGFicyhmbG9hdCh2MSkgLSBmbG9hdCh2MikpIDwgMC4wMQ0KICAgICAgICAgICAgZXhjZXB0IChWYWx1ZUVycm9yLCBUeXBlRXJyb3IpOg0KICAgICAgICAgICAgICAgIHJldHVybiBzdHIodjEpID09IHN0cih2MikNCg0KICAgICAgICByZXR1cm4gc3RyKHYxKSA9PSBzdHIodjIpDQo='
source_files['src/modules/data_loader.py'] = '...'
source_files['src/modules/prompt_builder.py'] = '...'
source_files['src/modules/question_logger.py'] = 'IiIiDQpsb2dnZXIucHkg4oCUIFBlci1RdWVzdGlvbiBMb2dnaW5nIGZvciBEYXRhTWF0ZQ0KDQpXcml0ZXMgZGV0YWlsZWQgcGVyLXF1ZXN0aW9uIGxvZ3MgdG8gL3Jlc3VsdC9sb2dzLyBmb3IgZGVidWdnaW5nLA0KcmVwcm9kdWNpYmlsaXR5IHZlcmlmaWNhdGlvbiwgYW5kIG9yZ2FuaXplciBpbnNwZWN0aW9uLg0KIiIiDQoNCmltcG9ydCBqc29uDQppbXBvcnQgbG9nZ2luZw0KaW1wb3J0IG9zDQppbXBvcnQgdGltZQ0KZnJvbSBkYXRldGltZSBpbXBvcnQgZGF0ZXRpbWUNCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aA0KZnJvbSB0eXBpbmcgaW1wb3J0IEFueSwgRGljdCwgT3B0aW9uYWwNCg0KbG9nZ2VyID0gbG9nZ2luZy5nZXRMb2dnZXIoX19uYW1lX18pDQoNCg0KY2xhc3MgUXVlc3Rpb25Mb2dnZXI6DQogICAgIiIiTG9ncyBkZXRhaWxlZCBpbmZvcm1hdGlvbiBmb3IgZWFjaCBxdWVzdGlvbiBwcm9jZXNzZWQuIiIiDQoNCiAgICBkZWYgX19pbml0X18oc2VsZiwgbG9nX2Rpcjogc3RyKToNCiAgICAgICAgIiIiDQogICAgICAgIFBhcmFtZXRlcnMNCiAgICAgICAgLS0tLS0tLS0tLQ0KICAgICAgICBsb2dfZGlyIDogc3RyDQogICAgICAgICAgICBEaXJlY3RvcnkgdG8gd3JpdGUgcGVyLXF1ZXN0aW9uIGxvZ3MgKGUuZy4sIC9yZXN1bHQvbG9ncy8pLg0KICAgICAgICAiIiINCiAgICAgICAgc2VsZi5sb2dfZGlyID0gUGF0aChsb2dfZGlyKQ0KICAgICAgICBzZWxmLmxvZ19kaXIubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQ0KICAgICAgICBzZWxmLl9zdGFydF90aW1lczogRGljdFtzdHIsIGZsb2F0XSA9IHt9DQogICAgICAgIHNlbGYuX3N1bW1hcnlfcGF0aCA9IHNlbGYubG9nX2RpciAvICJzdW1tYXJ5Lmpzb25sIg0KDQogICAgZGVmIHN0YXJ0X3F1ZXN0aW9uKHNlbGYsIHF1ZXN0aW9uX2lkOiBzdHIpOg0KICAgICAgICAiIiJSZWNvcmQgdGhlIHN0YXJ0IHRpbWUgZm9yIGEgcXVlc3Rpb24uIiIiDQogICAgICAgIHNlbGYuX3N0YXJ0X3RpbWVzW3F1ZXN0aW9uX2lkXSA9IHRpbWUudGltZSgpDQoNCiAgICBkZWYgbG9nX3F1ZXN0aW9uKA0KICAgICAgICBzZWxmLA0KICAgICAgICBxdWVzdGlvbl9pZDogc3RyLA0KICAgICAgICBxdWVzdGlvbl90ZXh0OiBzdHIsDQogICAgICAgIGJ1c2luZXNzX2lkOiBzdHIsDQogICAgICAgIGZpbmFsX2Fuc3dlcjogRGljdFtzdHIsIEFueV0sDQogICAgICAgIGdlbmVyYXRlZF9jb2RlOiBzdHIgPSAiIiwNCiAgICAgICAgZXhlY3V0aW9uX291dHB1dDogc3RyID0gIiIsDQogICAgICAgIHJldHJpZXM6IGludCA9IDAsDQogICAgICAgIGVycm9yX21lc3NhZ2VzOiBPcHRpb25hbFtsaXN0XSA9IE5vbmUsDQogICAgICAgIGV4dHJhOiBPcHRpb25hbFtEaWN0XSA9IE5vbmUsDQogICAgKToNCiAgICAgICAgIiIiDQogICAgICAgIFdyaXRlIGEgZGV0YWlsZWQgbG9nIGZvciBhIHNpbmdsZSBxdWVzdGlvbi4NCg0KICAgICAgICBQYXJhbWV0ZXJzDQogICAgICAgIC0tLS0tLS0tLS0NCiAgICAgICAgcXVlc3Rpb25faWQgOiBzdHINCiAgICAgICAgICAgIFRoZSBxdWVzdGlvbiBJRC4NCiAgICAgICAgcXVlc3Rpb25fdGV4dCA6IHN0cg0KICAgICAgICAgICAgVGhlIG9yaWdpbmFsIHF1ZXN0aW9uIHRleHQuDQogICAgICAgIGJ1c2luZXNzX2lkIDogc3RyDQogICAgICAgICAgICBUaGUgYnVzaW5lc3MgSUQuDQogICAgICAgIGZpbmFsX2Fuc3dlciA6IERpY3Rbc3RyLCBBbnldDQogICAgICAgICAgICBUaGUgZmluYWwgZm9ybWF0dGVkIGFuc3dlci4NCiAgICAgICAgZ2VuZXJhdGVkX2NvZGUgOiBzdHINCiAgICAgICAgICAgIFRoZSBnZW5lcmF0ZWQgY29kZSAobGFzdCBhdHRlbXB0KS4NCiAgICAgICAgZXhlY3V0aW9uX291dHB1dCA6IHN0cg0KICAgICAgICAgICAgc3Rkb3V0L3N0ZGVyciBmcm9tIGNvZGUgZXhlY3V0aW9uLg0KICAgICAgICByZXRyaWVzIDogaW50DQogICAgICAgICAgICBOdW1iZXIgb2YgcmV0cmllcyB1c2VkLg0KICAgICAgICBlcnJvcl9tZXNzYWdlcyA6IGxpc3QsIG9wdGlvbmFsDQogICAgICAgICAgICBFcnJvciBtZXNzYWdlcyBmcm9tIGZhaWxlZCBhdHRlbXB0cy4NCiAgICAgICAgZXh0cmEgOiBEaWN0LCBvcHRpb25hbA0KICAgICAgICAgICAgQW55IGFkZGl0aW9uYWwgbWV0YWRhdGEuDQogICAgICAgICIiIg0KICAgICAgICBlbGFwc2VkID0gMC4wDQogICAgICAgIGlmIHF1ZXN0aW9uX2lkIGluIHNlbGYuX3N0YXJ0X3RpbWVzOg0KICAgICAgICAgICAgZWxhcHNlZCA9IHRpbWUudGltZSgpIC0gc2VsZi5fc3RhcnRfdGltZXMucG9wKHF1ZXN0aW9uX2lkKQ0KDQogICAgICAgIGxvZ19lbnRyeSA9IHsNCiAgICAgICAgICAgICJxdWVzdGlvbl9pZCI6IHF1ZXN0aW9uX2lkLA0KICAgICAgICAgICAgImJ1c2luZXNzX2lkIjogYnVzaW5lc3NfaWQsDQogICAgICAgICAgICAicXVlc3Rpb24iOiBxdWVzdGlvbl90ZXh0LA0KICAgICAgICAgICAgImFuc3dlciI6IGZpbmFsX2Fuc3dlciwNCiAgICAgICAgICAgICJlbGFwc2VkX3NlY29uZHMiOiByb3VuZChlbGFwc2VkLCAyKSwNCiAgICAgICAgICAgICJyZXRyaWVzIjogcmV0cmllcywNCiAgICAgICAgICAgICJ0aW1lc3RhbXAiOiBkYXRldGltZS5ub3coKS5pc29mb3JtYXQoKSwNCiAgICAgICAgfQ0KDQogICAgICAgIGlmIGdlbmVyYXRlZF9jb2RlOg0KICAgICAgICAgICAgbG9nX2VudHJ5WyJnZW5lcmF0ZWRfY29kZSJdID0gZ2VuZXJhdGVkX2NvZGUNCiAgICAgICAgaWYgZXhlY3V0aW9uX291dHB1dDoNCiAgICAgICAgICAgIGxvZ19lbnRyeVsiZXhlY3V0aW9uX291dHB1dCJdID0gZXhlY3V0aW9uX291dHB1dFs6MjAwMF0NCiAgICAgICAgaWYgZXJyb3JfbWVzc2FnZXM6DQogICAgICAgICAgICBsb2dfZW50cnlbImVycm9ycyJdID0gZXJyb3JfbWVzc2FnZXMNCiAgICAgICAgaWYgZXh0cmE6DQogICAgICAgICAgICBsb2dfZW50cnlbImV4dHJhIl0gPSBleHRyYQ0KDQogICAgICAgICMgV3JpdGUgcGVyLXF1ZXN0aW9uIGxvZyBmaWxlDQogICAgICAgIHF1ZXN0aW9uX2xvZ19wYXRoID0gc2VsZi5sb2dfZGlyIC8gZiJ7cXVlc3Rpb25faWR9Lmpzb24iDQogICAgICAgIHRyeToNCiAgICAgICAgICAgIHdpdGggb3BlbihxdWVzdGlvbl9sb2dfcGF0aCwgInciLCBlbmNvZGluZz0idXRmLTgiKSBhcyBmOg0KICAgICAgICAgICAgICAgIGpzb24uZHVtcChsb2dfZW50cnksIGYsIGVuc3VyZV9hc2NpaT1GYWxzZSwgaW5kZW50PTIpDQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToNCiAgICAgICAgICAgIGxvZ2dlci5lcnJvcigiRmFpbGVkIHRvIHdyaXRlIGxvZyBmb3IgJXM6ICVzIiwgcXVlc3Rpb25faWQsIGUpDQoNCiAgICAgICAgIyBBcHBlbmQgdG8gc3VtbWFyeQ0KICAgICAgICBzdW1tYXJ5X2xpbmUgPSB7DQogICAgICAgICAgICAiaWQiOiBxdWVzdGlvbl9pZCwNCiAgICAgICAgICAgICJ0eXBlIjogZmluYWxfYW5zd2VyLmdldCgidHlwZSIsICJhYnN0YWluIiksDQogICAgICAgICAgICAiYW5zd2VyIjogZmluYWxfYW5zd2VyLmdldCgiYW5zd2VyIiksDQogICAgICAgICAgICAiZWxhcHNlZCI6IHJvdW5kKGVsYXBzZWQsIDIpLA0KICAgICAgICAgICAgInJldHJpZXMiOiByZXRyaWVzLA0KICAgICAgICB9DQogICAgICAgIHRyeToNCiAgICAgICAgICAgIHdpdGggb3BlbihzZWxmLl9zdW1tYXJ5X3BhdGgsICJhIiwgZW5jb2Rpbmc9InV0Zi04IikgYXMgZjoNCiAgICAgICAgICAgICAgICBmLndyaXRlKGpzb24uZHVtcHMoc3VtbWFyeV9saW5lLCBlbnN1cmVfYXNjaWk9RmFsc2UpICsgIlxuIikNCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOg0KICAgICAgICAgICAgbG9nZ2VyLmVycm9yKCJGYWlsZWQgdG8gd3JpdGUgc3VtbWFyeSBmb3IgJXM6ICVzIiwgcXVlc3Rpb25faWQsIGUpDQoNCiAgICAgICAgbG9nZ2VyLmluZm8oIkxvZ2dlZCBxdWVzdGlvbiAlczogdHlwZT0lcywgZWxhcHNlZD0lLjFmcywgcmV0cmllcz0lZCIsDQogICAgICAgICAgICAgICAgICAgICBxdWVzdGlvbl9pZCwgZmluYWxfYW5zd2VyLmdldCgidHlwZSIpLCBlbGFwc2VkLCByZXRyaWVzKQ0K'
source_files['src/modules/question_understanding.py'] = 'IiIiDQpxdWVzdGlvbl91bmRlcnN0YW5kaW5nLnB5IOKAlCBWaWV0bmFtZXNlIFF1ZXN0aW9uIEFuYWx5emVyIGZvciBEYXRhTWF0ZQ0KDQpDbGFzc2lmaWVzIHF1ZXN0aW9uIGludGVudCwgZXhwZWN0ZWQgYW5zd2VyIHR5cGUsIGFuZCBleHRyYWN0cyBrZXkgZW50aXRpZXMNCih0aW1lIGV4cHJlc3Npb25zLCBhZ2dyZWdhdGlvbiBpbnRlbnQsIHJhbmtpbmcgY3JpdGVyaWEsIGZpbHRlciBwcmVkaWNhdGVzKS4NCiIiIg0KDQppbXBvcnQgbG9nZ2luZw0KaW1wb3J0IHJlDQppbXBvcnQgdW5pY29kZWRhdGENCmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcywgZmllbGQNCmZyb20gZGF0ZXRpbWUgaW1wb3J0IGRhdGUNCmZyb20gdHlwaW5nIGltcG9ydCBEaWN0LCBMaXN0LCBPcHRpb25hbCwgVHVwbGUNCg0KZnJvbSAudGltZV9yZXNvbHZlciBpbXBvcnQgTHVuYXJDYWxlbmRhciwgVGltZVJlc29sdmVyDQoNCmxvZ2dlciA9IGxvZ2dpbmcuZ2V0TG9nZ2VyKF9fbmFtZV9fKQ0KDQoNCmRlZiBfbm9ybWFsaXplX2Zvcl9tYXRjaGluZyh0ZXh0OiBzdHIpIC0+IHN0cjoNCiAgICBub3JtYWxpemVkID0gdW5pY29kZWRhdGEubm9ybWFsaXplKCJORkQiLCB0ZXh0LmNhc2Vmb2xkKCkpDQogICAgbm9ybWFsaXplZCA9ICIiLmpvaW4oDQogICAgICAgIGNoYXJhY3RlciBmb3IgY2hhcmFjdGVyIGluIG5vcm1hbGl6ZWQNCiAgICAgICAgaWYgbm90IHVuaWNvZGVkYXRhLmNvbWJpbmluZyhjaGFyYWN0ZXIpDQogICAgKQ0KICAgIHJldHVybiBub3JtYWxpemVkLnJlcGxhY2UoIsSRIiwgImQiKQ0KDQoNCmRlZiBfcGF0dGVybl9tYXRjaGVzKHBhdHRlcm46IHN0ciwgcXVlc3Rpb246IHN0cikgLT4gYm9vbDoNCiAgICBpZiByZS5zZWFyY2gocGF0dGVybiwgcXVlc3Rpb24sIHJlLklHTk9SRUNBU0UpOg0KICAgICAgICByZXR1cm4gVHJ1ZQ0KICAgIHJldHVybiByZS5zZWFyY2goDQogICAgICAgIF9ub3JtYWxpemVfZm9yX21hdGNoaW5nKHBhdHRlcm4pLA0KICAgICAgICBfbm9ybWFsaXplX2Zvcl9tYXRjaGluZyhxdWVzdGlvbiksDQogICAgICAgIHJlLklHTk9SRUNBU0UsDQogICAgKSBpcyBub3QgTm9uZQ0KDQoNCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tDQojIEFuc3dlciB0eXBlIGNsYXNzaWZpY2F0aW9uDQojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQ0KDQojIEtleXdvcmRzIHN0cm9uZ2x5IGFzc29jaWF0ZWQgd2l0aCBlYWNoIGFuc3dlciB0eXBlDQpfVFlQRV9JTkRJQ0FUT1JTID0gew0KICAgICJudW1iZXIiOiBbDQogICAgICAgIHIiYmFvIG5oacOqdSg/IVxzKiV8XHMqcGjhuqduIHRyxINtKSIsDQogICAgICAgIHIiXGJiYW9ccytuaGl1XGIiLA0KICAgICAgICByIlxiKD86Ym5oaXV8Ym4pXGIiLA0KICAgICAgICByInThu5VuZ1xzKyg/OnPhu5F8dGnhu4FufGRvYW5ofGNoaXxsxrDhu6NuZ3xnacOhKSIsDQogICAgICAgIHIic+G7kSBsxrDhu6NuZyIsDQogICAgICAgIHIiKD86Y8OzfMSRw6N8xJHGsOG7o2MpXHMrYmFvIG5oacOqdSIsDQogICAgICAgIHIibeG6pXlccysoPzpjw6FpfGNoaeG6v2N8xJHGoW58c+G6o24gcGjhuqltfG3hurd0IGjDoG5nfGtow6FjaHzEkcSpYXxtw7NufGJpbGx8aMOzYSDEkcahbikiLA0KICAgICAgICByInPhu5FccysoPzrEkcahbnxiaWxsfGjDs2EgxJHGoW4pIiwNCiAgICAgICAgciJcYig/OmLDoW58YmFuKVxzKyg/OsSRxrDhu6NjfGRjfGR1b2MpXHMrbmhpw6p1XHMrdGnhu4FuXGIiLA0KICAgICAgICByIlxiYmlsbFxiLipcYmxhXHMrbmhpZXVcYiIsDQogICAgICAgIHIixJHhur9tIiwNCiAgICAgICAgciJjb3VudCIsDQogICAgICAgIHIidOG7lW5nIGPhu5luZyIsDQogICAgXSwNCiAgICAicGVyY2VudCI6IFsNCiAgICAgICAgciJiYW8gbmhpw6p1XHMqKD86JXxwaOG6p24gdHLEg20pIiwNCiAgICAgICAgciJ04bu3IGzhu4ciLA0KICAgICAgICByInBo4bqnbiB0csSDbSIsDQogICAgICAgIHIiKD86dMSDbmd8Z2nhuqNtKVxzK2JhbyBuaGnDqnVccyolIiwNCiAgICAgICAgciJjaGnhur9tXHMrYmFvIG5oacOqdSIsDQogICAgICAgIHIiKD86dMSDbmd8Z2nhuqNtKVxzKyg/OmJhbyBuaGnDqnVccyspPyg/OiV8cGjhuqduIHRyxINtKSIsDQogICAgICAgIHIiXGIoPzp04bu3IGzhu4d8dGkgbGUpXGIiLA0KICAgIF0sDQogICAgInN0cmluZyI6IFsNCiAgICAgICAgciIoPzpsw6BccyspPyg/OmfDrHxuw6BvKVxzKlw/PyQiLA0KICAgICAgICByInTDqm5ccysoPzpnw6x8bsOgbykiLA0KICAgICAgICByIig/OnPhuqNuIHBo4bqpbXxt4bq3dCBow6BuZ3xraMOhY2h8a2jDoWNoIGjDoG5nfG5ow6AgY3VuZyBj4bqlcClccytuw6BvIiwNCiAgICAgICAgciIoPzpraMOhY2h8a2jDoWNoIGjDoG5nfHPhuqNuIHBo4bqpbXxt4bq3dCBow6BuZylccytuw6BvLipcYig/Om5o4bqldHxuaMOsKVxiIiwNCiAgICAgICAgciJcYmFpXGIuKlxiKD86bXVhfGNoaSB0acOqdSlcYi4qXGJ0aWVuXGIuKlxibmhhdFxiIiwNCiAgICAgICAgciIoPzpiw6FuIGNo4bqheXxuaGnhu4F1fMOtdClccytuaOG6pXQiLA0KICAgICAgICByImxv4bqhaVxzKyg/OmfDrHxuw6BvKSIsDQogICAgXSwNCiAgICAiZGF0ZSI6IFsNCiAgICAgICAgciJuZ8OgeVxzK27DoG8iLA0KICAgICAgICByImtoaVxzK27DoG8iLA0KICAgICAgICByImJhbyBnaeG7nSIsDQogICAgICAgIHIidGjhu51pIMSRaeG7g20iLA0KICAgICAgICByIig/Om5nw6B5fHRow6FuZ3xuxINtKVxzKyg/OmJhbyBuaGnDqnV8beG6pXl8bsOgbykiLA0KICAgICAgICByInThu6tccytuZ8OgeVxzK27DoG8iLA0KICAgICAgICByImzhuqduXHMrKD86Y3Xhu5FpfGfhuqduIG5o4bqldHzEkeG6p3UgdGnDqm4pIiwNCiAgICBdLA0KICAgICJib29sZWFuIjogWw0KICAgICAgICByIl4oPzpjw7N8bGnhu4d1KVxzKy4rXHMra2jDtG5nXHMqXD8/JCIsDQogICAgICAgIHIiXGJjw7NcYi4rXGIoPzpraMO0bmd8a298a2hvbmcpXHMqXD8/JCIsDQogICAgICAgIHIixJHDum5nXHMraGF5XHMrc2FpIiwNCiAgICAgICAgciIoPzpjw7NccyspP8SRw7puZyBraMO0bmciLA0KICAgICAgICByIl4uK1xzKyg/OmPDsyBraMO0bmd8cGjhuqNpIGtow7RuZ3xjaMawYSlccypcPz8kIiwNCiAgICBdLA0KICAgICJsaXN0IjogWw0KICAgICAgICByIig/Omxp4buHdCBrw6p8ZGFuaCBzw6FjaHxsaXN0fGvhu4MgdMOqbnxuaOG7r25nKSIsDQogICAgICAgIHIiKD86dG9wfHjhur9wIGjhuqFuZylccytcZCsiLA0KICAgICAgICByIig/OmPDoWN8bmjhu69uZylccysoPzpz4bqjbiBwaOG6qW18beG6t3QgaMOgbmd8a2jDoWNoIGjDoG5nKSIsDQogICAgICAgIHIiKD86XGQrKVxzKyg/OnPhuqNuIHBo4bqpbXxt4bq3dCBow6BuZ3xraMOhY2ggaMOgbmcpXHMrKD86YsOhbiBjaOG6oXl8bmhp4buBdXzDrXQpIiwNCiAgICAgICAgciIoPzwhWy9cZF0pXGJcZCtccysoPzpz4bqjbiBwaOG6qW18beG6t3QgaMOgbmd8a2jDoWNofGtow6FjaCBow6BuZylccytuw6BvXGIiLA0KICAgICAgICByIlxiXGQrXHMrdGjDoW5nXGIuKlxibeG7l2kgdGjDoW5nXGIiLA0KICAgICAgICByIlxibOG6p24gbMaw4bujdFxiIiwNCiAgICBdLA0KICAgICJ0YWJsZSI6IFsNCiAgICAgICAgciJi4bqjbmciLA0KICAgICAgICByIig/OnRo4buRbmcga8OqfHThu5VuZyBo4bujcHxjaGkgdGnhur90KVxzKyg/OnRoZW98dOG7q25nKSIsDQogICAgICAgIHIiKD86dGhlb3x04burbmcpXHMrKD86dGjDoW5nfHR14bqnbnxuZ8OgeXxz4bqjbiBwaOG6qW18bG/huqFpfG5ow7NtKSIsDQogICAgICAgIHIiKD86c28gc8Ohbmh8YnJlYWtkb3duKSIsDQogICAgXSwNCn0NCg0KIyBLZXl3b3JkcyBpbmRpY2F0aW5nIHVuYW5zd2VyYWJsZQ0KX1VOQU5TV0VSQUJMRV9JTkRJQ0FUT1JTID0gWw0KICAgIHIia2jDtG5nXHMrY8OzXHMrKD86ZOG7ryBsaeG7h3V8dGjDtG5nIHRpbnxkYXRhKSIsDQogICAgciJuZ2/DoGlccytwaOG6oW0gdmkiLA0KXQ0KDQoNCkBkYXRhY2xhc3MNCmNsYXNzIFF1ZXN0aW9uSW5mbzoNCiAgICAiIiJTdHJ1Y3R1cmVkIGFuYWx5c2lzIG9mIGEgcXVlc3Rpb24uIiIiDQogICAgb3JpZ2luYWw6IHN0ciA9ICIiDQogICAgZXhwZWN0ZWRfdHlwZTogc3RyID0gInN0cmluZyIgICMgZGVmYXVsdCBmYWxsYmFjaw0KICAgIGNvbmZpZGVuY2U6IGZsb2F0ID0gMC41DQoNCiAgICAjIEV4dHJhY3RlZCBlbnRpdGllcw0KICAgIHRpbWVfZXhwcmVzc2lvbnM6IExpc3RbVHVwbGVbc3RyLCBkYXRlLCBkYXRlXV0gPSBmaWVsZChkZWZhdWx0X2ZhY3Rvcnk9bGlzdCkNCiAgICBhZ2dyZWdhdGlvbjogT3B0aW9uYWxbc3RyXSA9IE5vbmUgICMgc3VtLCBhdmcsIGNvdW50LCBtYXgsIG1pbg0KICAgIHJhbmtpbmdfZGlyZWN0aW9uOiBPcHRpb25hbFtzdHJdID0gTm9uZSAgIyAiZGVzYyIgb3IgImFzYyINCiAgICByYW5raW5nX2s6IE9wdGlvbmFsW2ludF0gPSBOb25lICAjIHRvcC1rDQogICAgcmFua2luZ19jcml0ZXJpb246IE9wdGlvbmFsW3N0cl0gPSBOb25lICAjIGUuZy4sICJxdWFudGl0eSIsICJyZXZlbnVlIg0KICAgIGZpbHRlcl9lbnRpdGllczogTGlzdFtzdHJdID0gZmllbGQoZGVmYXVsdF9mYWN0b3J5PWxpc3QpDQogICAgaXNfY29tcGFyaXNvbjogYm9vbCA9IEZhbHNlDQogICAgY29tcGFyaXNvbl9wZXJpb2RzOiBMaXN0W3N0cl0gPSBmaWVsZChkZWZhdWx0X2ZhY3Rvcnk9bGlzdCkNCg0KICAgIGRlZiB0b19kaWN0KHNlbGYpIC0+IERpY3Q6DQogICAgICAgIHJldHVybiB7DQogICAgICAgICAgICAiZXhwZWN0ZWRfdHlwZSI6IHNlbGYuZXhwZWN0ZWRfdHlwZSwNCiAgICAgICAgICAgICJjb25maWRlbmNlIjogc2VsZi5jb25maWRlbmNlLA0KICAgICAgICAgICAgImFnZ3JlZ2F0aW9uIjogc2VsZi5hZ2dyZWdhdGlvbiwNCiAgICAgICAgICAgICJyYW5raW5nIjogew0KICAgICAgICAgICAgICAgICJkaXJlY3Rpb24iOiBzZWxmLnJhbmtpbmdfZGlyZWN0aW9uLA0KICAgICAgICAgICAgICAgICJrIjogc2VsZi5yYW5raW5nX2ssDQogICAgICAgICAgICAgICAgImNyaXRlcmlvbiI6IHNlbGYucmFua2luZ19jcml0ZXJpb24sDQogICAgICAgICAgICB9IGlmIHNlbGYucmFua2luZ19kaXJlY3Rpb24gZWxzZSBOb25lLA0KICAgICAgICAgICAgImlzX2NvbXBhcmlzb24iOiBzZWxmLmlzX2NvbXBhcmlzb24sDQogICAgICAgICAgICAidGltZV9yYW5nZXMiOiBbDQogICAgICAgICAgICAgICAgeyJleHByIjogZSwgInN0YXJ0Ijogc3RyKHMpLCAiZW5kIjogc3RyKGQpfQ0KICAgICAgICAgICAgICAgIGZvciBlLCBzLCBkIGluIHNlbGYudGltZV9leHByZXNzaW9ucw0KICAgICAgICAgICAgXSwNCiAgICAgICAgfQ0KDQoNCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tDQojIEFnZ3JlZ2F0aW9uIGRldGVjdGlvbg0KIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0NCg0KX0FHR1JFR0FUSU9OX1BBVFRFUk5TID0gew0KICAgICJzdW0iOiBbciJ04buVbmciLCByImPhu5luZyIsIHIic3VtIiwgciJ04buVbmcgY+G7mW5nIl0sDQogICAgImF2ZyI6IFtyInRydW5nIGLDrG5oIiwgciJiw6xuaCBxdcOibiIsIHIiYXZlcmFnZSIsIHIiYXZnIiwgciJ0YiJdLA0KICAgICJjb3VudCI6IFtyIsSR4bq/bSIsIHIic+G7kSBsxrDhu6NuZyIsIHIiYmFvIG5oacOqdSIsIHIibeG6pXkiLCByImNvdW50Il0sDQogICAgIm1heCI6IFtyImzhu5tuIG5o4bqldCIsIHIiY2FvIG5o4bqldCIsIHIibmhp4buBdSBuaOG6pXQiLCByIm1heCIsIHIixJHhuq90IG5o4bqldCJdLA0KICAgICJtaW4iOiBbciJuaOG7jyBuaOG6pXQiLCByInRo4bqlcCBuaOG6pXQiLCByIsOtdCBuaOG6pXQiLCByIm1pbiIsIHIicuG6uyBuaOG6pXQiXSwNCn0NCg0KDQojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQ0KIyBSYW5raW5nIGRldGVjdGlvbg0KIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0NCg0KX1JBTktJTkdfREVTQ19LRVlXT1JEUyA9IFsNCiAgICByImLDoW4gY2jhuqF5IG5o4bqldCIsIHIibmhp4buBdSBuaOG6pXQiLCByImNhbyBuaOG6pXQiLCByImzhu5tuIG5o4bqldCIsDQogICAgciLEkeG6r3QgbmjhuqV0IiwgciJ0b3AiLCByImRvYW5oIHRodSBjYW8gbmjhuqV0IiwNCl0NCg0KX1JBTktJTkdfQVNDX0tFWVdPUkRTID0gWw0KICAgIHIiw610IG5o4bqldCIsIHIidGjhuqVwIG5o4bqldCIsIHIibmjhu48gbmjhuqV0IiwgciJy4bq7IG5o4bqldCIsIHIiYsOhbiDDrXQgbmjhuqV0IiwNCl0NCg0KDQojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQ0KIyBNYWluIGFuYWx5emVyDQojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQ0KDQpjbGFzcyBRdWVzdGlvbkFuYWx5emVyOg0KICAgICIiIkFuYWx5emUgVmlldG5hbWVzZSBxdWVzdGlvbnMgdG8gZXh0cmFjdCBzdHJ1Y3R1cmVkIGludGVudC4iIiINCg0KICAgIGRlZiBfX2luaXRfXyhzZWxmLCBjYWxlbmRhcjogT3B0aW9uYWxbTHVuYXJDYWxlbmRhcl0gPSBOb25lKToNCiAgICAgICAgc2VsZi50aW1lX3Jlc29sdmVyID0gVGltZVJlc29sdmVyKGNhbGVuZGFyKQ0KDQogICAgZGVmIGFuYWx5emUoc2VsZiwgcXVlc3Rpb246IHN0ciwgYXNfb2ZfZGF0ZTogZGF0ZSkgLT4gUXVlc3Rpb25JbmZvOg0KICAgICAgICAiIiINCiAgICAgICAgQW5hbHl6ZSBhIFZpZXRuYW1lc2UgcXVlc3Rpb24gYW5kIGV4dHJhY3Qgc3RydWN0dXJlZCBpbmZvcm1hdGlvbi4NCg0KICAgICAgICBQYXJhbWV0ZXJzDQogICAgICAgIC0tLS0tLS0tLS0NCiAgICAgICAgcXVlc3Rpb24gOiBzdHINCiAgICAgICAgICAgIFRoZSBxdWVzdGlvbiB0ZXh0IGluIFZpZXRuYW1lc2UuDQogICAgICAgIGFzX29mX2RhdGUgOiBkYXRlDQogICAgICAgICAgICBUaGUgYXMtb2YgZGF0ZSBmb3IgcmVzb2x2aW5nIHRpbWUgZXhwcmVzc2lvbnMuDQoNCiAgICAgICAgUmV0dXJucw0KICAgICAgICAtLS0tLS0tDQogICAgICAgIFF1ZXN0aW9uSW5mbw0KICAgICAgICAgICAgU3RydWN0dXJlZCBhbmFseXNpcyBvZiB0aGUgcXVlc3Rpb24uDQogICAgICAgICIiIg0KICAgICAgICBpbmZvID0gUXVlc3Rpb25JbmZvKG9yaWdpbmFsPXF1ZXN0aW9uKQ0KICAgICAgICBxX2xvd2VyID0gcXVlc3Rpb24ubG93ZXIoKS5zdHJpcCgpDQoNCiAgICAgICAgIyAxLiBDbGFzc2lmeSBleHBlY3RlZCBhbnN3ZXIgdHlwZQ0KICAgICAgICBpbmZvLmV4cGVjdGVkX3R5cGUsIGluZm8uY29uZmlkZW5jZSA9IHNlbGYuX2NsYXNzaWZ5X3R5cGUocV9sb3dlcikNCg0KICAgICAgICAjIDIuIEV4dHJhY3QgdGltZSBleHByZXNzaW9ucw0KICAgICAgICBpbmZvLnRpbWVfZXhwcmVzc2lvbnMgPSBzZWxmLnRpbWVfcmVzb2x2ZXIuZXh0cmFjdF9hbmRfcmVzb2x2ZSgNCiAgICAgICAgICAgIHF1ZXN0aW9uLCBhc19vZl9kYXRlDQogICAgICAgICkNCg0KICAgICAgICAjIDMuIERldGVjdCBhZ2dyZWdhdGlvbg0KICAgICAgICBpbmZvLmFnZ3JlZ2F0aW9uID0gc2VsZi5fZGV0ZWN0X2FnZ3JlZ2F0aW9uKHFfbG93ZXIpDQoNCiAgICAgICAgIyA0LiBEZXRlY3QgcmFua2luZw0KICAgICAgICBzZWxmLl9kZXRlY3RfcmFua2luZyhxX2xvd2VyLCBpbmZvKQ0KDQogICAgICAgICMgNS4gRGV0ZWN0IGNvbXBhcmlzb24NCiAgICAgICAgaW5mby5pc19jb21wYXJpc29uID0gYm9vbChyZS5zZWFyY2goDQogICAgICAgICAgICByInNvXHMrduG7m2l8c29ccytzw6FuaHx0xINuZ3xnaeG6o218Y2jDqm5oXHMrbOG7h2NofGPDuW5nXHMra+G7syIsDQogICAgICAgICAgICBxX2xvd2VyDQogICAgICAgICkpDQoNCiAgICAgICAgcmV0dXJuIGluZm8NCg0KICAgIGRlZiBfY2xhc3NpZnlfdHlwZShzZWxmLCBxdWVzdGlvbjogc3RyKSAtPiBUdXBsZVtzdHIsIGZsb2F0XToNCiAgICAgICAgIiIiQ2xhc3NpZnkgdGhlIGV4cGVjdGVkIGFuc3dlciB0eXBlLiIiIg0KICAgICAgICAjIENoZWNrIGZvciB1bmFuc3dlcmFibGUgaW5kaWNhdG9ycyBmaXJzdA0KICAgICAgICBmb3IgcGF0dGVybiBpbiBfVU5BTlNXRVJBQkxFX0lORElDQVRPUlM6DQogICAgICAgICAgICBpZiBfcGF0dGVybl9tYXRjaGVzKHBhdHRlcm4sIHF1ZXN0aW9uKToNCiAgICAgICAgICAgICAgICByZXR1cm4gKCJ1bmFuc3dlcmFibGUiLCAwLjcpDQoNCiAgICAgICAgIyBDaGVjayBmb3Igc3Ryb25nIGJvb2xlYW4gc2VudGVuY2UgcGF0dGVybnMgKGUuZy4sICJDw7MgLi4uIGtow7RuZz8iKQ0KICAgICAgICBpZiAoDQogICAgICAgICAgICBfcGF0dGVybl9tYXRjaGVzKA0KICAgICAgICAgICAgICAgIHIiXig/OmPDs3xsaeG7h3UpXHMrLitccysoPzpraMO0bmd8Y2jGsGF8a298a2hvbmcpXHMqXD8/JCIsDQogICAgICAgICAgICAgICAgcXVlc3Rpb24sDQogICAgICAgICAgICApDQogICAgICAgICAgICBvciBfcGF0dGVybl9tYXRjaGVzKA0KICAgICAgICAgICAgICAgIHIiKD86xJHDum5nXHMraGF5XHMrc2FpfMSRw7puZ1xzK2tow7RuZ3xwaOG6o2lccytraMO0bmcpXHMqXD8/JCIsDQogICAgICAgICAgICAgICAgcXVlc3Rpb24sDQogICAgICAgICAgICApDQogICAgICAgICAgICBvciBfcGF0dGVybl9tYXRjaGVzKA0KICAgICAgICAgICAgICAgIHIiXGJjw7NcYi4rXGIoPzpraMO0bmd8a298a2hvbmcpXHMqXD8/JCIsDQogICAgICAgICAgICAgICAgcXVlc3Rpb24sDQogICAgICAgICAgICApDQogICAgICAgICk6DQogICAgICAgICAgICByZXR1cm4gKCJib29sZWFuIiwgMC45KQ0KDQogICAgICAgIG5vcm1hbGl6ZWRfcXVlc3Rpb24gPSBfbm9ybWFsaXplX2Zvcl9tYXRjaGluZyhxdWVzdGlvbikuc3RyaXAoKQ0KICAgICAgICBpZiByZS5tYXRjaChyIl4oPzpjaG9ccyspP2JhbmdccysoPzp0b25nIGhvcHxjaGkgdGlldHx0aG9uZyBrZXxcZCspXGIiLCBub3JtYWxpemVkX3F1ZXN0aW9uKToNCiAgICAgICAgICAgIHJldHVybiAoInRhYmxlIiwgMC45KQ0KICAgICAgICBpZiBfcGF0dGVybl9tYXRjaGVzKA0KICAgICAgICAgICAgciJcYig/Om5nw6B5fHRow6FuZ3xuxINtKVxzK27DoG9cYnxcYmtoaSBuw6BvXGJ8YmFvIGdp4budIiwNCiAgICAgICAgICAgIHF1ZXN0aW9uLA0KICAgICAgICApOg0KICAgICAgICAgICAgcmV0dXJuICgiZGF0ZSIsIDAuOSkNCiAgICAgICAgaWYgKA0KICAgICAgICAgICAgcmUuc2VhcmNoKHIiXGJsYW4gbHVvdFxiIiwgbm9ybWFsaXplZF9xdWVzdGlvbikNCiAgICAgICAgICAgIG9yIGxlbihyZS5maW5kYWxsKHIiXGIoPzpiYW8gbmhpZXV8YmFvIG5oaXV8Ym5oaXV8Ym4pXGIiLCBub3JtYWxpemVkX3F1ZXN0aW9uKSkgPj0gMg0KICAgICAgICAgICAgb3IgcmUuc2VhcmNoKHIiXGJcZCtccyt0aGFuZ1xiLipcYm1vaSB0aGFuZ1xiIiwgbm9ybWFsaXplZF9xdWVzdGlvbikNCiAgICAgICAgICAgIG9yIHJlLnNlYXJjaCgNCiAgICAgICAgICAgICAgICByIig/PCFbL1xkXSlcYlxkK1xzKyg/OmtoYWNoKD86IGhhbmcpP3xzYW4gcGhhbXxtYXQgaGFuZylcYiINCiAgICAgICAgICAgICAgICByIi4qXGIoPzptdWF8Y2hpIHRpZXV8YmFuIGNoYXl8bmhhdHx0b3B8YmVzdCBzZWxsZXIpXGIiLA0KICAgICAgICAgICAgICAgIG5vcm1hbGl6ZWRfcXVlc3Rpb24sDQogICAgICAgICAgICApDQogICAgICAgICAgICBvciAoDQogICAgICAgICAgICAgICAgImJlc3Qgc2VsbGVyIiBpbiBub3JtYWxpemVkX3F1ZXN0aW9uDQogICAgICAgICAgICAgICAgYW5kIHJlLnNlYXJjaChyIig/PCFbL1xkXSlcYlxkK1xzK2Jlc3Qgc2VsbGVyXGIiLCBub3JtYWxpemVkX3F1ZXN0aW9uKQ0KICAgICAgICAgICAgKQ0KICAgICAgICAgICAgb3IgcmUuc2VhcmNoKA0KICAgICAgICAgICAgICAgIHIiXGJtYXlccytcdytcYi4qXGJ2YVxiLipcYm1heVxzK1x3K1xiIiwNCiAgICAgICAgICAgICAgICBub3JtYWxpemVkX3F1ZXN0aW9uLA0KICAgICAgICAgICAgKQ0KICAgICAgICApOg0KICAgICAgICAgICAgcmV0dXJuICgibGlzdCIsIDAuOSkNCiAgICAgICAgaWYgcmUuc2VhcmNoKA0KICAgICAgICAgICAgciIoPzwhWy9cZF0pXGJcZCtccysoPzpzYW4gcGhhbXxtYXQgaGFuZ3xraGFjaCg/OiBoYW5nKT8pXHMrbmFvXGIiLA0KICAgICAgICAgICAgbm9ybWFsaXplZF9xdWVzdGlvbiwNCiAgICAgICAgKToNCiAgICAgICAgICAgIHJldHVybiAoImxpc3QiLCAwLjkpDQoNCiAgICAgICAgcGVyY2VudF9zaG9ydGhhbmQgPSByZS5zZWFyY2goDQogICAgICAgICAgICByIlxiKD86Ym58Ym5oaXV8YmFvIG5oaWV1fGJhbyBuaGl1KVxzKiUiLA0KICAgICAgICAgICAgbm9ybWFsaXplZF9xdWVzdGlvbiwNCiAgICAgICAgKQ0KICAgICAgICBjaGFuZ2VkX3BlcmNlbnRhZ2UgPSByZS5zZWFyY2goDQogICAgICAgICAgICByIlxiKD86dGFuZ3xnaWFtfHRoYXkgZG9pKVxiLipcYig/OmJhbyBuaGlldXxuaGlldXxuaGl1fGJuaGl1fGJuKVxzKiUiLA0KICAgICAgICAgICAgbm9ybWFsaXplZF9xdWVzdGlvbiwNCiAgICAgICAgKQ0KICAgICAgICBpZiAoDQogICAgICAgICAgICBfcGF0dGVybl9tYXRjaGVzKHIiXGIoPzp04bu3IGzhu4d8cGjhuqduIHRyxINtfHRpIGxlfHBoYW4gdHJhbSlcYiIsIHF1ZXN0aW9uKQ0KICAgICAgICAgICAgb3IgX3BhdHRlcm5fbWF0Y2hlcyhyIlxiY2hp4bq/bVxiIiwgcXVlc3Rpb24pDQogICAgICAgICAgICBvciBwZXJjZW50X3Nob3J0aGFuZCBpcyBub3QgTm9uZQ0KICAgICAgICAgICAgb3IgY2hhbmdlZF9wZXJjZW50YWdlIGlzIG5vdCBOb25lDQogICAgICAgICAgICBvciByZS5zZWFyY2gociIlXHMqXD8/JCIsIHF1ZXN0aW9uKQ0KICAgICAgICApOg0KICAgICAgICAgICAgcmV0dXJuICgicGVyY2VudCIsIDAuOSkNCiAgICAgICAgaWYgKA0KICAgICAgICAgICAgcmUuc2VhcmNoKHIiXGIoPzpkb2FuaCB0aHV8ZHRodXxyZXZlbnVlKVxiIiwgbm9ybWFsaXplZF9xdWVzdGlvbikNCiAgICAgICAgICAgIGFuZCBub3QgcmUuc2VhcmNoKA0KICAgICAgICAgICAgICAgIHIiXGIoPzphaXxuYW98bmhhdHxuaGlldSBuaGF0fGNhbyBuaGF0fGxvbiBuaGF0fGl0IG5oYXR8dGhhcCBuaGF0KVxiIiwNCiAgICAgICAgICAgICAgICBub3JtYWxpemVkX3F1ZXN0aW9uLA0KICAgICAgICAgICAgKQ0KICAgICAgICApOg0KICAgICAgICAgICAgcmV0dXJuICgibnVtYmVyIiwgMC43NSkNCg0KICAgICAgICBzY29yZXM6IERpY3Rbc3RyLCBmbG9hdF0gPSB7fQ0KDQogICAgICAgIGZvciBhbnN3ZXJfdHlwZSwgcGF0dGVybnMgaW4gX1RZUEVfSU5ESUNBVE9SUy5pdGVtcygpOg0KICAgICAgICAgICAgc2NvcmUgPSAwLjANCiAgICAgICAgICAgIGZvciBwYXR0ZXJuIGluIHBhdHRlcm5zOg0KICAgICAgICAgICAgICAgIGlmIF9wYXR0ZXJuX21hdGNoZXMocGF0dGVybiwgcXVlc3Rpb24pOg0KICAgICAgICAgICAgICAgICAgICBzY29yZSArPSAxLjANCiAgICAgICAgICAgIGlmIHNjb3JlID4gMDoNCiAgICAgICAgICAgICAgICBzY29yZXNbYW5zd2VyX3R5cGVdID0gc2NvcmUNCg0KICAgICAgICBpZiBub3Qgc2NvcmVzOg0KICAgICAgICAgICAgcmV0dXJuICgic3RyaW5nIiwgMC4zKQ0KDQogICAgICAgIGJlc3RfdHlwZSA9IG1heChzY29yZXMsIGtleT1zY29yZXMuZ2V0KQ0KICAgICAgICBtYXhfc2NvcmUgPSBzY29yZXNbYmVzdF90eXBlXQ0KICAgICAgICBjb25maWRlbmNlID0gbWluKG1heF9zY29yZSAvIDMuMCwgMS4wKQ0KDQogICAgICAgIHJldHVybiAoYmVzdF90eXBlLCBjb25maWRlbmNlKQ0KDQogICAgZGVmIF9kZXRlY3RfYWdncmVnYXRpb24oc2VsZiwgcXVlc3Rpb246IHN0cikgLT4gT3B0aW9uYWxbc3RyXToNCiAgICAgICAgIiIiRGV0ZWN0IGFnZ3JlZ2F0aW9uIHR5cGUgZnJvbSB0aGUgcXVlc3Rpb24uIiIiDQogICAgICAgIGZvciBhZ2dfdHlwZSwgcGF0dGVybnMgaW4gX0FHR1JFR0FUSU9OX1BBVFRFUk5TLml0ZW1zKCk6DQogICAgICAgICAgICBmb3IgcGF0dGVybiBpbiBwYXR0ZXJuczoNCiAgICAgICAgICAgICAgICBpZiByZS5zZWFyY2gocGF0dGVybiwgcXVlc3Rpb24sIHJlLklHTk9SRUNBU0UpOg0KICAgICAgICAgICAgICAgICAgICByZXR1cm4gYWdnX3R5cGUNCiAgICAgICAgcmV0dXJuIE5vbmUNCg0KICAgIGRlZiBfZGV0ZWN0X3Jhbmtpbmcoc2VsZiwgcXVlc3Rpb246IHN0ciwgaW5mbzogUXVlc3Rpb25JbmZvKToNCiAgICAgICAgIiIiRGV0ZWN0IHJhbmtpbmcgaW50ZW50IGFuZCBleHRyYWN0IGsuIiIiDQogICAgICAgIG5vcm1hbGl6ZWRfcXVlc3Rpb24gPSBfbm9ybWFsaXplX2Zvcl9tYXRjaGluZyhxdWVzdGlvbikNCg0KICAgICAgICAjIENoZWNrIGRlc2NlbmRpbmcgKHRvcC9tb3N0KQ0KICAgICAgICBmb3IgcGF0dGVybiBpbiBfUkFOS0lOR19ERVNDX0tFWVdPUkRTOg0KICAgICAgICAgICAgaWYgX3BhdHRlcm5fbWF0Y2hlcyhwYXR0ZXJuLCBxdWVzdGlvbik6DQogICAgICAgICAgICAgICAgaW5mby5yYW5raW5nX2RpcmVjdGlvbiA9ICJkZXNjIg0KICAgICAgICAgICAgICAgIGJyZWFrDQoNCiAgICAgICAgIyBDaGVjayBhc2NlbmRpbmcgKGxlYXN0L2xvd2VzdCkNCiAgICAgICAgaWYgbm90IGluZm8ucmFua2luZ19kaXJlY3Rpb246DQogICAgICAgICAgICBmb3IgcGF0dGVybiBpbiBfUkFOS0lOR19BU0NfS0VZV09SRFM6DQogICAgICAgICAgICAgICAgaWYgX3BhdHRlcm5fbWF0Y2hlcyhwYXR0ZXJuLCBxdWVzdGlvbik6DQogICAgICAgICAgICAgICAgICAgIGluZm8ucmFua2luZ19kaXJlY3Rpb24gPSAiYXNjIg0KICAgICAgICAgICAgICAgICAgICBicmVhaw0KDQogICAgICAgIGlmIGluZm8ucmFua2luZ19kaXJlY3Rpb246DQogICAgICAgICAgICAjIEV4dHJhY3QgayAodG9wLWspDQogICAgICAgICAgICBtID0gcmUuc2VhcmNoKA0KICAgICAgICAgICAgICAgIHIiKD88IVsvXGRdKVxiKD86dG9wXHMrKT8oXGQrKVxzKyg/OnNhbiBwaGFtfG1hdCBoYW5nfGtoYWNoKD86IGhhbmcpP3xkb258YmlsbHxtb258ZGlhKVxiIiwNCiAgICAgICAgICAgICAgICBub3JtYWxpemVkX3F1ZXN0aW9uLA0KICAgICAgICAgICAgKQ0KICAgICAgICAgICAgaWYgbm90IG06DQogICAgICAgICAgICAgICAgbSA9IHJlLnNlYXJjaChyInRvcFxzKyhcZCspIiwgbm9ybWFsaXplZF9xdWVzdGlvbikNCiAgICAgICAgICAgIGlmIG06DQogICAgICAgICAgICAgICAgaW5mby5yYW5raW5nX2sgPSBpbnQobS5ncm91cCgxKSkNCiAgICAgICAgICAgIGVsc2U6DQogICAgICAgICAgICAgICAgIyAibmjhuqV0IiBpbXBsaWVzIGs9MQ0KICAgICAgICAgICAgICAgIGlmICJuaGF0IiBpbiBub3JtYWxpemVkX3F1ZXN0aW9uOg0KICAgICAgICAgICAgICAgICAgICBpbmZvLnJhbmtpbmdfayA9IDENCg0KICAgICAgICAgICAgIyBUcnkgdG8gZGV0ZWN0IHJhbmtpbmcgY3JpdGVyaW9uDQogICAgICAgICAgICBpZiByZS5zZWFyY2gociJiYW4gY2hheXxzbyBsdW9uZ3xxdWFudGl0eSIsIG5vcm1hbGl6ZWRfcXVlc3Rpb24pOg0KICAgICAgICAgICAgICAgIGluZm8ucmFua2luZ19jcml0ZXJpb24gPSAicXVhbnRpdHkiDQogICAgICAgICAgICBlbGlmIHJlLnNlYXJjaChyImRvYW5oIHRodXxyZXZlbnVlfHRpZW4iLCBub3JtYWxpemVkX3F1ZXN0aW9uKToNCiAgICAgICAgICAgICAgICBpbmZvLnJhbmtpbmdfY3JpdGVyaW9uID0gInJldmVudWUiDQoNCiAgICAgICAgICAgICMgSWYgcmFua2luZyB3aXRoIGs9MSBhbmQgZGlyZWN0aW9uLCBhbnN3ZXIgdHlwZSBpcyBsaWtlbHkgc3RyaW5nICh1bmxlc3MgZGF0ZS9saXN0L3RhYmxlKQ0KICAgICAgICAgICAgaWYgaW5mby5yYW5raW5nX2sgPT0gMSBhbmQgaW5mby5leHBlY3RlZF90eXBlIG5vdCBpbiAoImxpc3QiLCAidGFibGUiLCAiZGF0ZSIpOg0KICAgICAgICAgICAgICAgIGluZm8uZXhwZWN0ZWRfdHlwZSA9ICJzdHJpbmciDQogICAgICAgICAgICBlbGlmIGluZm8ucmFua2luZ19rIGFuZCBpbmZvLnJhbmtpbmdfayA+IDEgYW5kIGluZm8uZXhwZWN0ZWRfdHlwZSAhPSAidGFibGUiOg0KICAgICAgICAgICAgICAgIGluZm8uZXhwZWN0ZWRfdHlwZSA9ICJsaXN0Ig0K'
source_files['src/modules/readme_parser.py'] = 'IiIiDQpyZWFkbWVfcGFyc2VyLnB5IOKAlCBSRUFETUUubWQgUGFyc2VyIGZvciBEYXRhTWF0ZQ0KDQpFeHRyYWN0cyBidXNpbmVzcyBkZXNjcmlwdGlvbnMsIGNvbHVtbiBkZWZpbml0aW9ucywgYW5kIGRvbWFpbi1zcGVjaWZpYw0Kbm90ZXMgZnJvbSBlYWNoIGJ1c2luZXNzJ3MgUkVBRE1FLm1kIGZpbGUuIFByb2R1Y2VzIGEgc3RydWN0dXJlZCBjb250ZXh0DQpzdHJpbmcgZm9yIExMTSBwcm9tcHQgY29uc3RydWN0aW9uLg0KIiIiDQoNCmltcG9ydCBsb2dnaW5nDQppbXBvcnQgcmUNCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aA0KZnJvbSB0eXBpbmcgaW1wb3J0IERpY3QsIExpc3QsIE9wdGlvbmFsLCBUdXBsZQ0KDQppbXBvcnQgdW5pY29kZWRhdGENCg0KbG9nZ2VyID0gbG9nZ2luZy5nZXRMb2dnZXIoX19uYW1lX18pDQoNCg0KY2xhc3MgQnVzaW5lc3NDb250ZXh0Og0KICAgICIiIlBhcnNlZCBjb250ZXh0IGZyb20gYSBidXNpbmVzcydzIFJFQURNRS5tZC4iIiINCg0KICAgIGRlZiBfX2luaXRfXyhzZWxmKToNCiAgICAgICAgc2VsZi5yYXdfdGV4dDogc3RyID0gIiINCiAgICAgICAgc2VsZi5idXNpbmVzc19uYW1lOiBzdHIgPSAiIg0KICAgICAgICBzZWxmLmJ1c2luZXNzX3R5cGU6IHN0ciA9ICIiDQogICAgICAgIHNlbGYuZGVzY3JpcHRpb246IHN0ciA9ICIiDQogICAgICAgIHNlbGYuY29sdW1uX2RlZmluaXRpb25zOiBEaWN0W3N0ciwgc3RyXSA9IHt9DQogICAgICAgIHNlbGYubm90ZXM6IExpc3Rbc3RyXSA9IFtdDQogICAgICAgIHNlbGYuc2VjdGlvbnM6IERpY3Rbc3RyLCBzdHJdID0ge30NCg0KICAgIGRlZiB0b19wcm9tcHRfc3RyaW5nKHNlbGYsIG1heF9sZW5ndGg6IGludCA9IDIwMDApIC0+IHN0cjoNCiAgICAgICAgIiIiR2VuZXJhdGUgYSBwcm9tcHQtZnJpZW5kbHkgc3VtbWFyeSBvZiB0aGUgYnVzaW5lc3MgY29udGV4dC4iIiINCiAgICAgICAgcGFydHMgPSBbXQ0KDQogICAgICAgIGlmIHNlbGYuYnVzaW5lc3NfbmFtZToNCiAgICAgICAgICAgIHBhcnRzLmFwcGVuZChmIkJ1c2luZXNzOiB7c2VsZi5idXNpbmVzc19uYW1lfSIpDQogICAgICAgIGlmIHNlbGYuYnVzaW5lc3NfdHlwZToNCiAgICAgICAgICAgIHBhcnRzLmFwcGVuZChmIlR5cGU6IHtzZWxmLmJ1c2luZXNzX3R5cGV9IikNCiAgICAgICAgaWYgc2VsZi5kZXNjcmlwdGlvbjoNCiAgICAgICAgICAgIHBhcnRzLmFwcGVuZChmIkRlc2NyaXB0aW9uOiB7c2VsZi5kZXNjcmlwdGlvbn0iKQ0KDQogICAgICAgIGlmIHNlbGYuY29sdW1uX2RlZmluaXRpb25zOg0KICAgICAgICAgICAgcGFydHMuYXBwZW5kKCJcbkNvbHVtbiBkZWZpbml0aW9uczoiKQ0KICAgICAgICAgICAgZm9yIGNvbCwgZGVmbiBpbiBzZWxmLmNvbHVtbl9kZWZpbml0aW9ucy5pdGVtcygpOg0KICAgICAgICAgICAgICAgIHBhcnRzLmFwcGVuZChmIiAgLSB7Y29sfToge2RlZm59IikNCg0KICAgICAgICBpZiBzZWxmLm5vdGVzOg0KICAgICAgICAgICAgcGFydHMuYXBwZW5kKCJcbk5vdGVzOiIpDQogICAgICAgICAgICBmb3Igbm90ZSBpbiBzZWxmLm5vdGVzOg0KICAgICAgICAgICAgICAgIHBhcnRzLmFwcGVuZChmIiAgLSB7bm90ZX0iKQ0KDQogICAgICAgIHJlc3VsdCA9ICJcbiIuam9pbihwYXJ0cykNCiAgICAgICAgaWYgbGVuKHJlc3VsdCkgPiBtYXhfbGVuZ3RoOg0KICAgICAgICAgICAgcmVzdWx0ID0gcmVzdWx0WzptYXhfbGVuZ3RoIC0gM10gKyAiLi4uIg0KICAgICAgICByZXR1cm4gcmVzdWx0DQoNCg0KZGVmIF9leHRyYWN0X3NlY3Rpb25zKHRleHQ6IHN0cikgLT4gRGljdFtzdHIsIHN0cl06DQogICAgIiIiU3BsaXQgbWFya2Rvd24gdGV4dCBpbnRvIHNlY3Rpb25zIGJ5IGhlYWRlcnMuIiIiDQogICAgc2VjdGlvbnMgPSB7fQ0KICAgIGN1cnJlbnRfaGVhZGVyID0gImludHJvIg0KICAgIGN1cnJlbnRfbGluZXM6IExpc3Rbc3RyXSA9IFtdDQoNCiAgICBmb3IgbGluZSBpbiB0ZXh0LnNwbGl0KCJcbiIpOg0KICAgICAgICBoZWFkZXJfbWF0Y2ggPSByZS5tYXRjaChyIl4jezEsNH1ccysoLispIiwgbGluZSkNCiAgICAgICAgaWYgaGVhZGVyX21hdGNoOg0KICAgICAgICAgICAgaWYgY3VycmVudF9saW5lczoNCiAgICAgICAgICAgICAgICBzZWN0aW9uc1tjdXJyZW50X2hlYWRlcl0gPSAiXG4iLmpvaW4oY3VycmVudF9saW5lcykuc3RyaXAoKQ0KICAgICAgICAgICAgY3VycmVudF9oZWFkZXIgPSBoZWFkZXJfbWF0Y2guZ3JvdXAoMSkuc3RyaXAoKQ0KICAgICAgICAgICAgY3VycmVudF9saW5lcyA9IFtdDQogICAgICAgIGVsc2U6DQogICAgICAgICAgICBjdXJyZW50X2xpbmVzLmFwcGVuZChsaW5lKQ0KDQogICAgaWYgY3VycmVudF9saW5lczoNCiAgICAgICAgc2VjdGlvbnNbY3VycmVudF9oZWFkZXJdID0gIlxuIi5qb2luKGN1cnJlbnRfbGluZXMpLnN0cmlwKCkNCg0KICAgIHJldHVybiBzZWN0aW9ucw0KDQoNCmRlZiBfZXh0cmFjdF9jb2x1bW5fZGVmaW5pdGlvbnModGV4dDogc3RyKSAtPiBEaWN0W3N0ciwgc3RyXToNCiAgICAiIiINCiAgICBUcnkgdG8gZXh0cmFjdCBjb2x1bW4gZGVmaW5pdGlvbnMgZnJvbSBtYXJrZG93biB0ZXh0Lg0KICAgIEhhbmRsZXMgZm9ybWF0cyBsaWtlOg0KICAgIC0gYGNvbHVtbl9uYW1lYDogZGVzY3JpcHRpb24NCiAgICAtICoqY29sdW1uX25hbWUqKjogZGVzY3JpcHRpb24NCiAgICAtIHwgY29sdW1uIHwgZGVzY3JpcHRpb24gfCAgKHRhYmxlIGZvcm1hdCkNCiAgICAtIGNvbHVtbl9uYW1lIOKAlCBkZXNjcmlwdGlvbg0KICAgICIiIg0KICAgIGRlZmluaXRpb25zID0ge30NCg0KICAgICMgUGF0dGVybiAxOiBiYWNrdGljayBvciBib2xkIGNvbHVtbiBuYW1lIGZvbGxvd2VkIGJ5IGNvbG9uL2Rhc2gNCiAgICBwYXR0ZXJuMSA9IHJlLmNvbXBpbGUoDQogICAgICAgIHIiKD86YChbXmBdKylgfFwqXCooW14qXSspXCpcKilccypbOuKAk+KAlC1dXHMqKC4rKSIsIHJlLlVOSUNPREUNCiAgICApDQogICAgZm9yIG0gaW4gcGF0dGVybjEuZmluZGl0ZXIodGV4dCk6DQogICAgICAgIGNvbF9uYW1lID0gbS5ncm91cCgxKSBvciBtLmdyb3VwKDIpDQogICAgICAgIGRlc2NyaXB0aW9uID0gbS5ncm91cCgzKS5zdHJpcCgpDQogICAgICAgIGRlZmluaXRpb25zW2NvbF9uYW1lLnN0cmlwKCldID0gZGVzY3JpcHRpb24NCg0KICAgICMgUGF0dGVybiAyOiBzaW1wbGUgIi0gY29sdW1uX25hbWU6IGRlc2NyaXB0aW9uIiBsaXN0IGl0ZW1zDQogICAgcGF0dGVybjIgPSByZS5jb21waWxlKA0KICAgICAgICByIl5bLSpdXHMrKFx3W1x3XHNdKj8pXHMqWzrigJPigJRdXHMqKC4rKSIsIHJlLk1VTFRJTElORSB8IHJlLlVOSUNPREUNCiAgICApDQogICAgZm9yIG0gaW4gcGF0dGVybjIuZmluZGl0ZXIodGV4dCk6DQogICAgICAgIGNvbF9uYW1lID0gbS5ncm91cCgxKS5zdHJpcCgpDQogICAgICAgIGRlc2NyaXB0aW9uID0gbS5ncm91cCgyKS5zdHJpcCgpDQogICAgICAgIGlmIGNvbF9uYW1lIG5vdCBpbiBkZWZpbml0aW9uczoNCiAgICAgICAgICAgIGRlZmluaXRpb25zW2NvbF9uYW1lXSA9IGRlc2NyaXB0aW9uDQoNCiAgICAjIFBhdHRlcm4gMzogbWFya2Rvd24gdGFibGUgcm93cyAofCBjb2wgfCBkZXNjIHwpDQogICAgdGFibGVfcGF0dGVybiA9IHJlLmNvbXBpbGUoDQogICAgICAgIHIiXlx8XHMqKFtefF0rPylccypcfFxzKihbXnxdKz8pXHMqXHwiLCByZS5NVUxUSUxJTkUNCiAgICApDQogICAgcm93cyA9IGxpc3QodGFibGVfcGF0dGVybi5maW5kaXRlcih0ZXh0KSkNCiAgICBpZiBsZW4ocm93cykgPj0gMzogICMgaGVhZGVyICsgc2VwYXJhdG9yICsgYXQgbGVhc3QgMSBkYXRhIHJvdw0KICAgICAgICBmb3Igcm93IGluIHJvd3NbMjpdOiAgIyBza2lwIGhlYWRlciBhbmQgc2VwYXJhdG9yDQogICAgICAgICAgICBjb2xfbmFtZSA9IHJvdy5ncm91cCgxKS5zdHJpcCgpDQogICAgICAgICAgICBkZXNjcmlwdGlvbiA9IHJvdy5ncm91cCgyKS5zdHJpcCgpDQogICAgICAgICAgICBpZiBjb2xfbmFtZSBhbmQgZGVzY3JpcHRpb24gYW5kIG5vdCBjb2xfbmFtZS5zdGFydHN3aXRoKCItIik6DQogICAgICAgICAgICAgICAgZGVmaW5pdGlvbnNbY29sX25hbWVdID0gZGVzY3JpcHRpb24NCg0KICAgIHJldHVybiBkZWZpbml0aW9ucw0KDQoNCmRlZiBfZXh0cmFjdF9idXNpbmVzc19pbmZvKHRleHQ6IHN0cikgLT4gVHVwbGVbc3RyLCBzdHJdOg0KICAgICIiIlRyeSB0byBleHRyYWN0IGJ1c2luZXNzIG5hbWUgYW5kIHR5cGUgZnJvbSB0aGUgUkVBRE1FLiIiIg0KICAgIG5hbWUgPSAiIg0KICAgIGJpel90eXBlID0gIiINCg0KICAgICMgTG9vayBmb3IgY29tbW9uIHBhdHRlcm5zDQogICAgbmFtZV9wYXR0ZXJucyA9IFsNCiAgICAgICAgcmUuY29tcGlsZShyIig/OnTDqm58bmFtZXxj4butYSBow6BuZ3xzaG9wKVxzKls677yaXVxzKiguKykiLCByZS5JR05PUkVDQVNFKSwNCiAgICAgICAgcmUuY29tcGlsZShyIl4jXHMrKC4rKSIsIHJlLk1VTFRJTElORSksDQogICAgXQ0KICAgIGZvciBwIGluIG5hbWVfcGF0dGVybnM6DQogICAgICAgIG0gPSBwLnNlYXJjaCh0ZXh0KQ0KICAgICAgICBpZiBtOg0KICAgICAgICAgICAgbmFtZSA9IG0uZ3JvdXAoMSkuc3RyaXAoKQ0KICAgICAgICAgICAgYnJlYWsNCg0KICAgIHR5cGVfcGF0dGVybnMgPSBbDQogICAgICAgIHJlLmNvbXBpbGUociIoPzpsb+G6oWl8dHlwZXxuZ8Ogbmh8bMSpbmggduG7sWMpXHMqWzrvvJpdXHMqKC4rKSIsIHJlLklHTk9SRUNBU0UpLA0KICAgIF0NCiAgICBmb3IgcCBpbiB0eXBlX3BhdHRlcm5zOg0KICAgICAgICBtID0gcC5zZWFyY2godGV4dCkNCiAgICAgICAgaWYgbToNCiAgICAgICAgICAgIGJpel90eXBlID0gbS5ncm91cCgxKS5zdHJpcCgpDQogICAgICAgICAgICBicmVhaw0KDQogICAgcmV0dXJuIG5hbWUsIGJpel90eXBlDQoNCg0KZGVmIHBhcnNlX3JlYWRtZShyZWFkbWVfcGF0aDogc3RyKSAtPiBCdXNpbmVzc0NvbnRleHQ6DQogICAgIiIiDQogICAgUGFyc2UgYSBidXNpbmVzcyBSRUFETUUubWQgZmlsZSBhbmQgZXh0cmFjdCBzdHJ1Y3R1cmVkIGNvbnRleHQuDQoNCiAgICBQYXJhbWV0ZXJzDQogICAgLS0tLS0tLS0tLQ0KICAgIHJlYWRtZV9wYXRoIDogc3RyDQogICAgICAgIFBhdGggdG8gdGhlIFJFQURNRS5tZCBmaWxlLg0KDQogICAgUmV0dXJucw0KICAgIC0tLS0tLS0NCiAgICBCdXNpbmVzc0NvbnRleHQNCiAgICAgICAgU3RydWN0dXJlZCBjb250ZXh0IGV4dHJhY3RlZCBmcm9tIHRoZSBSRUFETUUuDQogICAgIiIiDQogICAgY3R4ID0gQnVzaW5lc3NDb250ZXh0KCkNCiAgICBwYXRoID0gUGF0aChyZWFkbWVfcGF0aCkNCg0KICAgIGlmIG5vdCBwYXRoLmV4aXN0cygpOg0KICAgICAgICBsb2dnZXIud2FybmluZygiUkVBRE1FIG5vdCBmb3VuZDogJXMiLCByZWFkbWVfcGF0aCkNCiAgICAgICAgcmV0dXJuIGN0eA0KDQogICAgdHJ5Og0KICAgICAgICByYXcgPSBwYXRoLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKQ0KICAgIGV4Y2VwdCBVbmljb2RlRGVjb2RlRXJyb3I6DQogICAgICAgIHRyeToNCiAgICAgICAgICAgIHJhdyA9IHBhdGgucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOC1zaWciKQ0KICAgICAgICBleGNlcHQgRXhjZXB0aW9uOg0KICAgICAgICAgICAgcmF3ID0gcGF0aC5yZWFkX3RleHQoZW5jb2Rpbmc9ImxhdGluLTEiKQ0KDQogICAgcmF3ID0gdW5pY29kZWRhdGEubm9ybWFsaXplKCJORkMiLCByYXcpDQogICAgY3R4LnJhd190ZXh0ID0gcmF3DQoNCiAgICAjIEV4dHJhY3Qgc2VjdGlvbnMNCiAgICBjdHguc2VjdGlvbnMgPSBfZXh0cmFjdF9zZWN0aW9ucyhyYXcpDQoNCiAgICAjIEV4dHJhY3QgYnVzaW5lc3MgaW5mbw0KICAgIGN0eC5idXNpbmVzc19uYW1lLCBjdHguYnVzaW5lc3NfdHlwZSA9IF9leHRyYWN0X2J1c2luZXNzX2luZm8ocmF3KQ0KDQogICAgIyBVc2UgaW50cm8gb3IgZmlyc3Qgc2VjdGlvbiBhcyBkZXNjcmlwdGlvbg0KICAgIGlmICJpbnRybyIgaW4gY3R4LnNlY3Rpb25zIGFuZCBjdHguc2VjdGlvbnNbImludHJvIl06DQogICAgICAgIGN0eC5kZXNjcmlwdGlvbiA9IGN0eC5zZWN0aW9uc1siaW50cm8iXVs6NTAwXQ0KICAgIGVsaWYgY3R4LnNlY3Rpb25zOg0KICAgICAgICBmaXJzdF9rZXkgPSBuZXh0KGl0ZXIoY3R4LnNlY3Rpb25zKSkNCiAgICAgICAgY3R4LmRlc2NyaXB0aW9uID0gY3R4LnNlY3Rpb25zW2ZpcnN0X2tleV1bOjUwMF0NCg0KICAgICMgRXh0cmFjdCBjb2x1bW4gZGVmaW5pdGlvbnMNCiAgICBjdHguY29sdW1uX2RlZmluaXRpb25zID0gX2V4dHJhY3RfY29sdW1uX2RlZmluaXRpb25zKHJhdykNCg0KICAgICMgRXh0cmFjdCBub3RlcyAoYnVsbGV0IHBvaW50cyB0aGF0IGxvb2sgbGlrZSBzcGVjaWFsIGluc3RydWN0aW9ucykNCiAgICBub3RlX3BhdHRlcm4gPSByZS5jb21waWxlKA0KICAgICAgICByIig/OmzGsHUgw718bm90ZXxnaGkgY2jDunxjaMO6IMO9fMSR4bq3YyBiaeG7h3R8c3BlY2lhbClccypbOu+8ml0/XHMqKC4rKSIsDQogICAgICAgIHJlLklHTk9SRUNBU0UNCiAgICApDQogICAgZm9yIG0gaW4gbm90ZV9wYXR0ZXJuLmZpbmRpdGVyKHJhdyk6DQogICAgICAgIGN0eC5ub3Rlcy5hcHBlbmQobS5ncm91cCgxKS5zdHJpcCgpKQ0KDQogICAgbG9nZ2VyLmluZm8oIlBhcnNlZCBSRUFETUUgJXM6IG5hbWU9JyVzJywgdHlwZT0nJXMnLCAlZCBjb2wgZGVmcywgJWQgbm90ZXMiLA0KICAgICAgICAgICAgICAgIHJlYWRtZV9wYXRoLCBjdHguYnVzaW5lc3NfbmFtZSwgY3R4LmJ1c2luZXNzX3R5cGUsDQogICAgICAgICAgICAgICAgbGVuKGN0eC5jb2x1bW5fZGVmaW5pdGlvbnMpLCBsZW4oY3R4Lm5vdGVzKSkNCg0KICAgIHJldHVybiBjdHgNCg=='
source_files['src/modules/retry_engine.py'] = 'IiIiDQpyZXRyeV9lbmdpbmUucHkg4oCUIFJldHJ5ICYgU2VsZi1Db3JyZWN0aW9uIExvb3AgZm9yIERhdGFNYXRlDQoNCk1hbmFnZXMgdGhlIExMTSBjb2RlIGdlbmVyYXRpb24g4oaSIGV4ZWN1dGlvbiDihpIgcmV0cnkgY3ljbGUuDQpGZWVkcyBlcnJvciBtZXNzYWdlcyBiYWNrIHRvIHRoZSBMTE0gZm9yIHNlbGYtY29ycmVjdGlvbiB1cCB0byBOIHJldHJpZXMuDQoiIiINCg0KaW1wb3J0IGxvZ2dpbmcNCmltcG9ydCB0aW1lDQpmcm9tIHR5cGluZyBpbXBvcnQgQ2FsbGFibGUsIERpY3QsIE9wdGlvbmFsLCBUdXBsZQ0KDQppbXBvcnQgcGFuZGFzIGFzIHBkDQoNCmZyb20gLmNvZGVfZXhlY3V0b3IgaW1wb3J0IENvZGVFeGVjdXRvciwgRXhlY3V0aW9uUmVzdWx0DQpmcm9tIC5wcm9tcHRfYnVpbGRlciBpbXBvcnQgUHJvbXB0QnVpbGRlciwgUHJvbXB0VG9rZW5MaW1pdEVycm9yDQoNCmxvZ2dlciA9IGxvZ2dpbmcuZ2V0TG9nZ2VyKF9fbmFtZV9fKQ0KDQoNCmNsYXNzIFJldHJ5RW5naW5lOg0KICAgICIiIg0KICAgIE1hbmFnZXMgdGhlIGNvZGUgZ2VuZXJhdGlvbiDihpIgZXhlY3V0aW9uIOKGkiByZXRyeSBsb29wLg0KDQogICAgT24gZmFpbHVyZSwgZmVlZHMgdGhlIGVycm9yIGJhY2sgdG8gdGhlIExMTSBmb3Igc2VsZi1jb3JyZWN0aW9uLA0KICAgIHVwIHRvIG1heF9yZXRyaWVzIGF0dGVtcHRzLg0KICAgICIiIg0KDQogICAgZGVmIF9faW5pdF9fKA0KICAgICAgICBzZWxmLA0KICAgICAgICBjb2RlX2V4ZWN1dG9yOiBDb2RlRXhlY3V0b3IsDQogICAgICAgIHByb21wdF9idWlsZGVyOiBQcm9tcHRCdWlsZGVyLA0KICAgICAgICBsbG1fZ2VuZXJhdGVfZm46IENhbGxhYmxlW1tzdHJdLCBzdHJdLA0KICAgICAgICBtYXhfcmV0cmllczogaW50ID0gMywNCiAgICApOg0KICAgICAgICAiIiINCiAgICAgICAgUGFyYW1ldGVycw0KICAgICAgICAtLS0tLS0tLS0tDQogICAgICAgIGNvZGVfZXhlY3V0b3IgOiBDb2RlRXhlY3V0b3INCiAgICAgICAgICAgIFRoZSBzYW5kYm94ZWQgY29kZSBleGVjdXRvci4NCiAgICAgICAgcHJvbXB0X2J1aWxkZXIgOiBQcm9tcHRCdWlsZGVyDQogICAgICAgICAgICBUaGUgcHJvbXB0IGJ1aWxkZXIgZm9yIGNvbnN0cnVjdGluZyBwcm9tcHRzLg0KICAgICAgICBsbG1fZ2VuZXJhdGVfZm4gOiBDYWxsYWJsZVtbc3RyXSwgc3RyXQ0KICAgICAgICAgICAgRnVuY3Rpb24gdGhhdCB0YWtlcyBhIHByb21wdCBzdHJpbmcgYW5kIHJldHVybnMgdGhlIExMTSByZXNwb25zZS4NCiAgICAgICAgbWF4X3JldHJpZXMgOiBpbnQNCiAgICAgICAgICAgIE1heGltdW0gbnVtYmVyIG9mIHJldHJ5IGF0dGVtcHRzLg0KICAgICAgICAiIiINCiAgICAgICAgc2VsZi5leGVjdXRvciA9IGNvZGVfZXhlY3V0b3INCiAgICAgICAgc2VsZi5wcm9tcHRfYnVpbGRlciA9IHByb21wdF9idWlsZGVyDQogICAgICAgIHNlbGYubGxtX2dlbmVyYXRlID0gbGxtX2dlbmVyYXRlX2ZuDQogICAgICAgIHNlbGYubWF4X3JldHJpZXMgPSBtYXhfcmV0cmllcw0KDQogICAgZGVmIHJ1bigNCiAgICAgICAgc2VsZiwNCiAgICAgICAgcHJvbXB0OiBzdHIsDQogICAgICAgIHRhYmxlczogRGljdFtzdHIsIHBkLkRhdGFGcmFtZV0sDQogICAgICAgIGFzX29mX2RhdGU6IHN0ciwNCiAgICAgICAgZXhwZWN0ZWRfdHlwZTogT3B0aW9uYWxbc3RyXSA9IE5vbmUsDQogICAgICAgIHRva2VuX2NvdW50ZXI6IE9wdGlvbmFsW0NhbGxhYmxlW1tzdHJdLCBpbnRdXSA9IE5vbmUsDQogICAgICAgIG1heF9wcm9tcHRfdG9rZW5zOiBPcHRpb25hbFtpbnRdID0gTm9uZSwNCiAgICApIC0+IFR1cGxlW0V4ZWN1dGlvblJlc3VsdCwgaW50XToNCiAgICAgICAgIiIiDQogICAgICAgIFJ1biB0aGUgY29kZSBnZW5lcmF0aW9uIOKGkiBleGVjdXRpb24gbG9vcCB3aXRoIHJldHJpZXMuDQoNCiAgICAgICAgUGFyYW1ldGVycw0KICAgICAgICAtLS0tLS0tLS0tDQogICAgICAgIHByb21wdCA6IHN0cg0KICAgICAgICAgICAgVGhlIGluaXRpYWwgcHJvbXB0IGZvciBjb2RlIGdlbmVyYXRpb24uDQogICAgICAgIHRhYmxlcyA6IERpY3Rbc3RyLCBwZC5EYXRhRnJhbWVdDQogICAgICAgICAgICBCdXNpbmVzcyBkYXRhIHRhYmxlcy4NCiAgICAgICAgYXNfb2ZfZGF0ZSA6IHN0cg0KICAgICAgICAgICAgVGhlIGFzLW9mIGRhdGUgaW4gSVNPIGZvcm1hdC4NCiAgICAgICAgZXhwZWN0ZWRfdHlwZSA6IHN0ciwgb3B0aW9uYWwNCiAgICAgICAgICAgIEV4cGVjdGVkIGFuc3dlciB0eXBlIGZvciB2YWxpZGF0aW9uLg0KICAgICAgICB0b2tlbl9jb3VudGVyIDogQ2FsbGFibGVbW3N0cl0sIGludF0sIG9wdGlvbmFsDQogICAgICAgICAgICBDb3VudHMgdG9rZW5zIHVzaW5nIHRoZSBnZW5lcmF0aW9uIG1vZGVsJ3MgdG9rZW5pemVyLg0KICAgICAgICBtYXhfcHJvbXB0X3Rva2VucyA6IGludCwgb3B0aW9uYWwNCiAgICAgICAgICAgIE1heGltdW0gaW5wdXQgdG9rZW5zIGFsbG93ZWQgZm9yIGEgcmV0cnkgcHJvbXB0Lg0KDQogICAgICAgIFJldHVybnMNCiAgICAgICAgLS0tLS0tLQ0KICAgICAgICBUdXBsZVtFeGVjdXRpb25SZXN1bHQsIGludF0NCiAgICAgICAgICAgIFRoZSBmaW5hbCBleGVjdXRpb24gcmVzdWx0IGFuZCB0aGUgbnVtYmVyIG9mIHJldHJpZXMgdXNlZC4NCiAgICAgICAgIiIiDQogICAgICAgIGN1cnJlbnRfcHJvbXB0ID0gcHJvbXB0DQogICAgICAgIGJlc3RfcmVzdWx0OiBPcHRpb25hbFtFeGVjdXRpb25SZXN1bHRdID0gTm9uZQ0KICAgICAgICByZXRyaWVzID0gMA0KICAgICAgICBsYXN0X2dlbmVyYXRpb25fZXJyb3I6IE9wdGlvbmFsW3N0cl0gPSBOb25lDQogICAgICAgIHNlZW5fZ2VuZXJhdGlvbl9mYWlsdXJlcyA9IHNldCgpDQogICAgICAgIHNlZW5fZXhlY3V0aW9uX2ZhaWx1cmVzID0gc2V0KCkNCg0KICAgICAgICBmb3IgYXR0ZW1wdCBpbiByYW5nZSgxICsgc2VsZi5tYXhfcmV0cmllcyk6DQogICAgICAgICAgICAjIEdlbmVyYXRlIGNvZGUgZnJvbSBMTE0NCiAgICAgICAgICAgIHQwID0gdGltZS50aW1lKCkNCiAgICAgICAgICAgIHRyeToNCiAgICAgICAgICAgICAgICBsbG1fcmVzcG9uc2UgPSBzZWxmLmxsbV9nZW5lcmF0ZShjdXJyZW50X3Byb21wdCkNCiAgICAgICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToNCiAgICAgICAgICAgICAgICBsb2dnZXIuZXJyb3IoIkxMTSBnZW5lcmF0aW9uIGZhaWxlZCBvbiBhdHRlbXB0ICVkOiAlcyIsIGF0dGVtcHQgKyAxLCBlKQ0KICAgICAgICAgICAgICAgIGxhc3RfZ2VuZXJhdGlvbl9lcnJvciA9IGYie3R5cGUoZSkuX19uYW1lX199OiB7ZX0iDQogICAgICAgICAgICAgICAgaWYgc2VsZi5faXNfY29udGV4dF9sZW5ndGhfZXJyb3IoZSk6DQogICAgICAgICAgICAgICAgICAgIGJlc3RfcmVzdWx0ID0gRXhlY3V0aW9uUmVzdWx0KCkNCiAgICAgICAgICAgICAgICAgICAgYmVzdF9yZXN1bHQuYW5zd2VyX3R5cGUgPSAiYWJzdGFpbiINCiAgICAgICAgICAgICAgICAgICAgYmVzdF9yZXN1bHQuZXJyb3IgPSBsYXN0X2dlbmVyYXRpb25fZXJyb3INCiAgICAgICAgICAgICAgICAgICAgcmV0cmllcyA9IGF0dGVtcHQNCiAgICAgICAgICAgICAgICAgICAgYnJlYWsNCiAgICAgICAgICAgICAgICBnZW5lcmF0aW9uX2ZhaWx1cmUgPSAoDQogICAgICAgICAgICAgICAgICAgIGN1cnJlbnRfcHJvbXB0LA0KICAgICAgICAgICAgICAgICAgICB0eXBlKGUpLl9fbmFtZV9fLA0KICAgICAgICAgICAgICAgICAgICBzdHIoZSkuc3BsaXRsaW5lcygpWzBdLA0KICAgICAgICAgICAgICAgICkNCiAgICAgICAgICAgICAgICBpZiBnZW5lcmF0aW9uX2ZhaWx1cmUgaW4gc2Vlbl9nZW5lcmF0aW9uX2ZhaWx1cmVzOg0KICAgICAgICAgICAgICAgICAgICBsb2dnZXIud2FybmluZygNCiAgICAgICAgICAgICAgICAgICAgICAgICJTdG9wcGluZyBhZnRlciByZXBlYXRlZCBnZW5lcmF0aW9uIGZhaWx1cmUgb24gYXR0ZW1wdCAlZCIsDQogICAgICAgICAgICAgICAgICAgICAgICBhdHRlbXB0ICsgMSwNCiAgICAgICAgICAgICAgICAgICAgKQ0KICAgICAgICAgICAgICAgICAgICByZXRyaWVzID0gYXR0ZW1wdA0KICAgICAgICAgICAgICAgICAgICBicmVhaw0KICAgICAgICAgICAgICAgIHNlZW5fZ2VuZXJhdGlvbl9mYWlsdXJlcy5hZGQoZ2VuZXJhdGlvbl9mYWlsdXJlKQ0KICAgICAgICAgICAgICAgIHJldHJpZXMgPSBhdHRlbXB0DQogICAgICAgICAgICAgICAgY29udGludWUNCiAgICAgICAgICAgIGdlbl90aW1lID0gdGltZS50aW1lKCkgLSB0MA0KICAgICAgICAgICAgbG9nZ2VyLmluZm8oIkxMTSBnZW5lcmF0aW9uIGF0dGVtcHQgJWQgdG9vayAlLjFmcyIsIGF0dGVtcHQgKyAxLCBnZW5fdGltZSkNCg0KICAgICAgICAgICAgIyBFeGVjdXRlIHRoZSBnZW5lcmF0ZWQgY29kZQ0KICAgICAgICAgICAgdDAgPSB0aW1lLnRpbWUoKQ0KICAgICAgICAgICAgcmVzdWx0ID0gc2VsZi5leGVjdXRvci5leGVjdXRlKA0KICAgICAgICAgICAgICAgIGxsbV9yZXNwb25zZSwgdGFibGVzLCBhc19vZl9kYXRlLCBleHRyYWN0X2NvZGU9VHJ1ZQ0KICAgICAgICAgICAgKQ0KICAgICAgICAgICAgZXhlY190aW1lID0gdGltZS50aW1lKCkgLSB0MA0KICAgICAgICAgICAgbG9nZ2VyLmluZm8oIkNvZGUgZXhlY3V0aW9uIGF0dGVtcHQgJWQgdG9vayAlLjFmcywgc3VjY2Vzcz0lcyIsDQogICAgICAgICAgICAgICAgICAgICAgICBhdHRlbXB0ICsgMSwgZXhlY190aW1lLCByZXN1bHQuc3VjY2VzcykNCg0KICAgICAgICAgICAgIyBDaGVjayBpZiBzdWNjZXNzZnVsDQogICAgICAgICAgICBpZiByZXN1bHQuc3VjY2VzczoNCiAgICAgICAgICAgICAgICAjIFZhbGlkYXRlIGFuc3dlciB0eXBlIGlmIGV4cGVjdGVkDQogICAgICAgICAgICAgICAgaWYgZXhwZWN0ZWRfdHlwZSBhbmQgcmVzdWx0LmFuc3dlcl90eXBlICE9IGV4cGVjdGVkX3R5cGU6DQogICAgICAgICAgICAgICAgICAgICMgVHlwZSBtaXNtYXRjaCDigJQgcmV0cnkgd2l0aCBoaW50DQogICAgICAgICAgICAgICAgICAgIGlmIGF0dGVtcHQgPCBzZWxmLm1heF9yZXRyaWVzOg0KICAgICAgICAgICAgICAgICAgICAgICAgbG9nZ2VyLmluZm8oDQogICAgICAgICAgICAgICAgICAgICAgICAgICAgIlR5cGUgbWlzbWF0Y2g6IGV4cGVjdGVkPSVzLCBnb3Q9JXMuIFJldHJ5aW5nLiIsDQogICAgICAgICAgICAgICAgICAgICAgICAgICAgZXhwZWN0ZWRfdHlwZSwgcmVzdWx0LmFuc3dlcl90eXBlDQogICAgICAgICAgICAgICAgICAgICAgICApDQogICAgICAgICAgICAgICAgICAgICAgICBlcnJvcl9tc2cgPSAoDQogICAgICAgICAgICAgICAgICAgICAgICAgICAgZiJUaGUgYW5zd2VyIHR5cGUgc2hvdWxkIGJlICd7ZXhwZWN0ZWRfdHlwZX0nIGJ1dCAiDQogICAgICAgICAgICAgICAgICAgICAgICAgICAgZiJ5b3VyIGNvZGUgcHJvZHVjZWQgJ3tyZXN1bHQuYW5zd2VyX3R5cGV9Jy4gIg0KICAgICAgICAgICAgICAgICAgICAgICAgICAgIGYiUGxlYXNlIGZpeCB0aGUgY29kZSB0byByZXR1cm4gdGhlIGNvcnJlY3QgdHlwZS4iDQogICAgICAgICAgICAgICAgICAgICAgICApDQogICAgICAgICAgICAgICAgICAgICAgICB0cnk6DQogICAgICAgICAgICAgICAgICAgICAgICAgICAgY3VycmVudF9wcm9tcHQgPSBzZWxmLnByb21wdF9idWlsZGVyLmJ1aWxkX3JldHJ5X3Byb21wdCgNCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgcHJvbXB0LA0KICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICByZXN1bHQuY29kZSwNCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZXJyb3JfbXNnLA0KICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBhdmFpbGFibGVfY29udGV4dD1zZWxmLl9hdmFpbGFibGVfY29udGV4dCh0YWJsZXMsIGVycm9yX21zZyksDQogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHRva2VuX2NvdW50ZXI9dG9rZW5fY291bnRlciwNCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgbWF4X3Rva2Vucz1tYXhfcHJvbXB0X3Rva2VucywNCiAgICAgICAgICAgICAgICAgICAgICAgICAgICApDQogICAgICAgICAgICAgICAgICAgICAgICBleGNlcHQgUHJvbXB0VG9rZW5MaW1pdEVycm9yIGFzIGV4YzoNCiAgICAgICAgICAgICAgICAgICAgICAgICAgICByZXN1bHQuYW5zd2VyX3R5cGUgPSAiYWJzdGFpbiINCiAgICAgICAgICAgICAgICAgICAgICAgICAgICByZXN1bHQuZXJyb3IgPSBzdHIoZXhjKQ0KICAgICAgICAgICAgICAgICAgICAgICAgICAgIHJldHVybiAocmVzdWx0LCBhdHRlbXB0KQ0KICAgICAgICAgICAgICAgICAgICAgICAgYmVzdF9yZXN1bHQgPSByZXN1bHQNCiAgICAgICAgICAgICAgICAgICAgICAgIHJldHJpZXMgPSBhdHRlbXB0ICsgMQ0KICAgICAgICAgICAgICAgICAgICAgICAgY29udGludWUNCg0KICAgICAgICAgICAgICAgICMgVmFsaWRhdGUgcmVzdWx0IGlzIG5vdCBOb25lICh1bmxlc3MgdW5hbnN3ZXJhYmxlL2Fic3RhaW4pDQogICAgICAgICAgICAgICAgaWYgKHJlc3VsdC5yZXN1bHQgaXMgTm9uZSBhbmQNCiAgICAgICAgICAgICAgICAgICAgcmVzdWx0LmFuc3dlcl90eXBlIG5vdCBpbiAoInVuYW5zd2VyYWJsZSIsICJhYnN0YWluIikpOg0KICAgICAgICAgICAgICAgICAgICBpZiBhdHRlbXB0IDwgc2VsZi5tYXhfcmV0cmllczoNCiAgICAgICAgICAgICAgICAgICAgICAgIGxvZ2dlci5pbmZvKCJSZXN1bHQgaXMgTm9uZSBidXQgdHlwZSBpcyAlcy4gUmV0cnlpbmcuIiwNCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHJlc3VsdC5hbnN3ZXJfdHlwZSkNCiAgICAgICAgICAgICAgICAgICAgICAgIGVycm9yX21zZyA9ICgNCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAiWW91ciBjb2RlIHNldCByZXN1bHQgPSBOb25lLiBQbGVhc2UgZW5zdXJlIGByZXN1bHRgICINCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAiaXMgYXNzaWduZWQgYSB2YWxpZCB2YWx1ZSwgb3Igc2V0IGFuc3dlcl90eXBlID0gIg0KICAgICAgICAgICAgICAgICAgICAgICAgICAgICIndW5hbnN3ZXJhYmxlJyBpZiB0aGUgZGF0YSBjYW5ub3QgYW5zd2VyIHRoZSBxdWVzdGlvbi4iDQogICAgICAgICAgICAgICAgICAgICAgICApDQogICAgICAgICAgICAgICAgICAgICAgICB0cnk6DQogICAgICAgICAgICAgICAgICAgICAgICAgICAgY3VycmVudF9wcm9tcHQgPSBzZWxmLnByb21wdF9idWlsZGVyLmJ1aWxkX3JldHJ5X3Byb21wdCgNCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgcHJvbXB0LA0KICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICByZXN1bHQuY29kZSwNCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZXJyb3JfbXNnLA0KICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBhdmFpbGFibGVfY29udGV4dD1zZWxmLl9hdmFpbGFibGVfY29udGV4dCh0YWJsZXMsIGVycm9yX21zZyksDQogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHRva2VuX2NvdW50ZXI9dG9rZW5fY291bnRlciwNCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgbWF4X3Rva2Vucz1tYXhfcHJvbXB0X3Rva2VucywNCiAgICAgICAgICAgICAgICAgICAgICAgICAgICApDQogICAgICAgICAgICAgICAgICAgICAgICBleGNlcHQgUHJvbXB0VG9rZW5MaW1pdEVycm9yIGFzIGV4YzoNCiAgICAgICAgICAgICAgICAgICAgICAgICAgICByZXN1bHQuYW5zd2VyX3R5cGUgPSAiYWJzdGFpbiINCiAgICAgICAgICAgICAgICAgICAgICAgICAgICByZXN1bHQuZXJyb3IgPSBzdHIoZXhjKQ0KICAgICAgICAgICAgICAgICAgICAgICAgICAgIHJldHVybiAocmVzdWx0LCBhdHRlbXB0KQ0KICAgICAgICAgICAgICAgICAgICAgICAgYmVzdF9yZXN1bHQgPSByZXN1bHQNCiAgICAgICAgICAgICAgICAgICAgICAgIHJldHJpZXMgPSBhdHRlbXB0ICsgMQ0KICAgICAgICAgICAgICAgICAgICAgICAgY29udGludWUNCg0KICAgICAgICAgICAgICAgICMgU3VjY2Vzcw0KICAgICAgICAgICAgICAgIHJldHVybiAocmVzdWx0LCBhdHRlbXB0KQ0KDQogICAgICAgICAgICBlbHNlOg0KICAgICAgICAgICAgICAgICMgRXhlY3V0aW9uIGZhaWxlZCDigJQgcmV0cnkgd2l0aCBlcnJvciBtZXNzYWdlDQogICAgICAgICAgICAgICAgYmVzdF9yZXN1bHQgPSByZXN1bHQNCiAgICAgICAgICAgICAgICByZXRyaWVzID0gYXR0ZW1wdA0KDQogICAgICAgICAgICAgICAgZXJyb3JfbWVzc2FnZSA9IHJlc3VsdC5lcnJvciBvciAiVW5rbm93biBleGVjdXRpb24gZXJyb3IiDQogICAgICAgICAgICAgICAgZmFpbHVyZV9zaWduYXR1cmUgPSAoDQogICAgICAgICAgICAgICAgICAgIHJlc3VsdC5jb2RlLnN0cmlwKCksDQogICAgICAgICAgICAgICAgICAgIGVycm9yX21lc3NhZ2Uuc3BsaXRsaW5lcygpWzBdLnN0cmlwKCksDQogICAgICAgICAgICAgICAgKQ0KICAgICAgICAgICAgICAgIGlmIGZhaWx1cmVfc2lnbmF0dXJlIGluIHNlZW5fZXhlY3V0aW9uX2ZhaWx1cmVzOg0KICAgICAgICAgICAgICAgICAgICBsb2dnZXIud2FybmluZygNCiAgICAgICAgICAgICAgICAgICAgICAgICJTdG9wcGluZyBhZnRlciByZXBlYXRlZCBjb2RlIGFuZCBlcnJvciBvbiBhdHRlbXB0ICVkOiAlcyIsDQogICAgICAgICAgICAgICAgICAgICAgICBhdHRlbXB0ICsgMSwNCiAgICAgICAgICAgICAgICAgICAgICAgIGZhaWx1cmVfc2lnbmF0dXJlWzFdLA0KICAgICAgICAgICAgICAgICAgICApDQogICAgICAgICAgICAgICAgICAgIGJyZWFrDQogICAgICAgICAgICAgICAgc2Vlbl9leGVjdXRpb25fZmFpbHVyZXMuYWRkKGZhaWx1cmVfc2lnbmF0dXJlKQ0KDQogICAgICAgICAgICAgICAgaWYgYXR0ZW1wdCA8IHNlbGYubWF4X3JldHJpZXM6DQogICAgICAgICAgICAgICAgICAgIGVycm9yX21zZyA9IGVycm9yX21lc3NhZ2UNCiAgICAgICAgICAgICAgICAgICAgIyBUcnVuY2F0ZSB2ZXJ5IGxvbmcgZXJyb3IgbWVzc2FnZXMNCiAgICAgICAgICAgICAgICAgICAgaWYgbGVuKGVycm9yX21zZykgPiAxMDAwOg0KICAgICAgICAgICAgICAgICAgICAgICAgZXJyb3JfbXNnID0gZXJyb3JfbXNnWzoxMDAwXSArICJcbi4uLiAodHJ1bmNhdGVkKSINCg0KICAgICAgICAgICAgICAgICAgICB0cnk6DQogICAgICAgICAgICAgICAgICAgICAgICBjdXJyZW50X3Byb21wdCA9IHNlbGYucHJvbXB0X2J1aWxkZXIuYnVpbGRfcmV0cnlfcHJvbXB0KA0KICAgICAgICAgICAgICAgICAgICAgICAgICAgIHByb21wdCwNCiAgICAgICAgICAgICAgICAgICAgICAgICAgICByZXN1bHQuY29kZSwNCiAgICAgICAgICAgICAgICAgICAgICAgICAgICBlcnJvcl9tc2csDQogICAgICAgICAgICAgICAgICAgICAgICAgICAgYXZhaWxhYmxlX2NvbnRleHQ9c2VsZi5fYXZhaWxhYmxlX2NvbnRleHQodGFibGVzLCBlcnJvcl9tc2cpLA0KICAgICAgICAgICAgICAgICAgICAgICAgICAgIHRva2VuX2NvdW50ZXI9dG9rZW5fY291bnRlciwNCiAgICAgICAgICAgICAgICAgICAgICAgICAgICBtYXhfdG9rZW5zPW1heF9wcm9tcHRfdG9rZW5zLA0KICAgICAgICAgICAgICAgICAgICAgICAgKQ0KICAgICAgICAgICAgICAgICAgICBleGNlcHQgUHJvbXB0VG9rZW5MaW1pdEVycm9yIGFzIGV4YzoNCiAgICAgICAgICAgICAgICAgICAgICAgIHJlc3VsdC5hbnN3ZXJfdHlwZSA9ICJhYnN0YWluIg0KICAgICAgICAgICAgICAgICAgICAgICAgcmVzdWx0LmVycm9yID0gc3RyKGV4YykNCiAgICAgICAgICAgICAgICAgICAgICAgIGJyZWFrDQogICAgICAgICAgICAgICAgICAgIGxvZ2dlci5pbmZvKCJSZXRyeWluZyBhZnRlciBlcnJvcjogJXMiLCBlcnJvcl9tc2dbOjIwMF0pDQoNCiAgICAgICAgIyBBbGwgcmV0cmllcyBleGhhdXN0ZWQg4oCUIHJldHVybiBiZXN0IHJlc3VsdCBvciBhYnN0YWluDQogICAgICAgIGlmIGJlc3RfcmVzdWx0IGlzIE5vbmU6DQogICAgICAgICAgICBiZXN0X3Jlc3VsdCA9IEV4ZWN1dGlvblJlc3VsdCgpDQogICAgICAgICAgICBiZXN0X3Jlc3VsdC5hbnN3ZXJfdHlwZSA9ICJhYnN0YWluIg0KICAgICAgICAgICAgYmVzdF9yZXN1bHQuZXJyb3IgPSBsYXN0X2dlbmVyYXRpb25fZXJyb3Igb3IgIkFsbCBhdHRlbXB0cyBmYWlsZWQiDQoNCiAgICAgICAgaWYgbm90IGJlc3RfcmVzdWx0LnN1Y2Nlc3M6DQogICAgICAgICAgICBiZXN0X3Jlc3VsdC5hbnN3ZXJfdHlwZSA9ICJhYnN0YWluIg0KDQogICAgICAgIHJldHVybiAoYmVzdF9yZXN1bHQsIHJldHJpZXMpDQoNCiAgICBAc3RhdGljbWV0aG9kDQogICAgZGVmIF9pc19jb250ZXh0X2xlbmd0aF9lcnJvcihlcnJvcjogRXhjZXB0aW9uKSAtPiBib29sOg0KICAgICAgICBtZXNzYWdlID0gc3RyKGVycm9yKS5jYXNlZm9sZCgpDQogICAgICAgIHJldHVybiBhbnkobWFya2VyIGluIG1lc3NhZ2UgZm9yIG1hcmtlciBpbiAoDQogICAgICAgICAgICAibWF4aW11bSBjb250ZXh0IGxlbmd0aCIsDQogICAgICAgICAgICAibWF4X21vZGVsX2xlbiIsDQogICAgICAgICAgICAiaW5wdXRfdG9rZW5zIiwNCiAgICAgICAgICAgICJjb250ZXh0IGxlbmd0aCIsDQogICAgICAgICAgICAiZXhjZWVkcyB0aGlzIGxpbWl0IiwNCiAgICAgICAgKSkNCg0KICAgIGRlZiBfYXZhaWxhYmxlX2NvbnRleHQoDQogICAgICAgIHNlbGYsDQogICAgICAgIHRhYmxlczogRGljdFtzdHIsIHBkLkRhdGFGcmFtZV0sDQogICAgICAgIGVycm9yX21lc3NhZ2U6IHN0ciwNCiAgICApIC0+IE9wdGlvbmFsW3N0cl06DQogICAgICAgIG1hcHBpbmcgPSBzZWxmLmV4ZWN1dG9yLmdldF90YWJsZV92YXJfbWFwcGluZyh0YWJsZXMpDQoNCiAgICAgICAgaWYgIk5hbWVFcnJvciIgaW4gZXJyb3JfbWVzc2FnZToNCiAgICAgICAgICAgIHJldHVybiBzZWxmLl9zY2hlbWFfY29udGV4dCh0YWJsZXMsIG1hcHBpbmcpDQoNCiAgICAgICAgaWYgYW55KG1hcmtlciBpbiBlcnJvcl9tZXNzYWdlIGZvciBtYXJrZXIgaW4gKA0KICAgICAgICAgICAgIktleUVycm9yIiwNCiAgICAgICAgICAgICJGaWxlTm90Rm91bmRFcnJvciIsDQogICAgICAgICAgICAiRmlsZUFjY2Vzc0Vycm9yIiwNCiAgICAgICAgKSk6DQogICAgICAgICAgICBjb250ZXh0ID0gc2VsZi5fc2NoZW1hX2NvbnRleHQodGFibGVzLCBtYXBwaW5nLCBpbmNsdWRlX2R0eXBlcz1GYWxzZSkNCiAgICAgICAgICAgIGlmIGFueShtYXJrZXIgaW4gZXJyb3JfbWVzc2FnZSBmb3IgbWFya2VyIGluICgNCiAgICAgICAgICAgICAgICAiRmlsZU5vdEZvdW5kRXJyb3IiLA0KICAgICAgICAgICAgICAgICJGaWxlQWNjZXNzRXJyb3IiLA0KICAgICAgICAgICAgKSk6DQogICAgICAgICAgICAgICAgY29udGV4dCA9ICgNCiAgICAgICAgICAgICAgICAgICAgIlRoZSBkYXRhIGlzIGFscmVhZHkgbG9hZGVkIGluIHRoZXNlIERhdGFGcmFtZXMuIERvIG5vdCByZWFkIG9yICINCiAgICAgICAgICAgICAgICAgICAgInJlbG9hZCBmaWxlczsgdXNlIG9ubHkgdGhlc2UgdGFibGVzIGFuZCBjb2x1bW5zLlxuIiArIGNvbnRleHQNCiAgICAgICAgICAgICAgICApDQogICAgICAgICAgICByZXR1cm4gY29udGV4dFs6NjAwMF0NCg0KICAgICAgICBpZiBhbnkobWFya2VyIGluIGVycm9yX21lc3NhZ2UgZm9yIG1hcmtlciBpbiAoDQogICAgICAgICAgICAiVGltZXN0YW1wIiwNCiAgICAgICAgICAgICJkYXRldGltZS5kYXRlIiwNCiAgICAgICAgICAgICJub3Qgc3VwcG9ydGVkIGJldHdlZW4gaW5zdGFuY2VzIG9mICdzdHInIGFuZCAnVGltZXN0YW1wJyIsDQogICAgICAgICAgICAiRGF0ZVBhcnNlRXJyb3IiLA0KICAgICAgICAgICAgIlBhcnNlckVycm9yIiwNCiAgICAgICAgICAgICJEYXRlVHlwZUVycm9yIiwNCiAgICAgICAgKSk6DQogICAgICAgICAgICBjb250ZXh0ID0gc2VsZi5fc2NoZW1hX2NvbnRleHQodGFibGVzLCBtYXBwaW5nLCBpbmNsdWRlX2R0eXBlcz1UcnVlKQ0KICAgICAgICAgICAgY29udGV4dCA9ICgNCiAgICAgICAgICAgICAgICAiRGF0ZSBjb3JyZWN0aW9uOiBwYXJzZSBzdHJpbmcgZGF0ZSBjb2x1bW5zIHdpdGggIg0KICAgICAgICAgICAgICAgICJgcGQudG9fZGF0ZXRpbWUoc2VyaWVzLCBlcnJvcnM9J2NvZXJjZScpYCwga2VlcCB2YWx1ZXMgYXMgcGFuZGFzICINCiAgICAgICAgICAgICAgICAidGltZXN0YW1wcywgYW5kIGNvbXBhcmUgd2l0aCBgcGQuVGltZXN0YW1wKC4uLilgLiBEbyBub3QgY29udmVydCAiDQogICAgICAgICAgICAgICAgImEgd2hvbGUgY29sdW1uIHdpdGggYC5kdC5kYXRlYC4gVXNlIHRoZSBhY3R1YWwgZGF0ZSBjb2x1bW5zIGJlbG93LlxuIg0KICAgICAgICAgICAgICAgICsgY29udGV4dA0KICAgICAgICAgICAgKQ0KICAgICAgICAgICAgcmV0dXJuIGNvbnRleHRbOjYwMDBdDQoNCiAgICAgICAgaWYgIlN5bnRheEVycm9yIiBpbiBlcnJvcl9tZXNzYWdlOg0KICAgICAgICAgICAgcmV0dXJuICgNCiAgICAgICAgICAgICAgICAiVGhlIGdlbmVyYXRlZCBjb2RlIGhhcyBhIHN5bnRheCBlcnJvci4gUmV0dXJuIG9uZSBjb21wbGV0ZSwgdmFsaWQgIg0KICAgICAgICAgICAgICAgICJQeXRob24gY29kZSBibG9jayB3aXRoIGJhbGFuY2VkIHF1b3Rlcy9icmFja2V0cyBhbmQgYXNzaWduIGJvdGggIg0KICAgICAgICAgICAgICAgICJgcmVzdWx0YCBhbmQgYGFuc3dlcl90eXBlYC4iDQogICAgICAgICAgICApDQoNCiAgICAgICAgaWYgIlR5cGVFcnJvciIgaW4gZXJyb3JfbWVzc2FnZSBvciAiVmFsdWVFcnJvciIgaW4gZXJyb3JfbWVzc2FnZToNCiAgICAgICAgICAgIHJldHVybiBzZWxmLl9zY2hlbWFfY29udGV4dCh0YWJsZXMsIG1hcHBpbmcsIGluY2x1ZGVfZHR5cGVzPVRydWUpDQoNCiAgICAgICAgcmV0dXJuIE5vbmUNCg0KICAgIEBzdGF0aWNtZXRob2QNCiAgICBkZWYgX3NjaGVtYV9jb250ZXh0KA0KICAgICAgICB0YWJsZXM6IERpY3Rbc3RyLCBwZC5EYXRhRnJhbWVdLA0KICAgICAgICBtYXBwaW5nOiBEaWN0W3N0ciwgc3RyXSwNCiAgICAgICAgaW5jbHVkZV9kdHlwZXM6IGJvb2wgPSBGYWxzZSwNCiAgICApIC0+IHN0cjoNCiAgICAgICAgdGFibGVfY29sdW1ucyA9IFtdDQogICAgICAgIGZvciB0YWJsZSwgZGF0YWZyYW1lIGluIHRhYmxlcy5pdGVtcygpOg0KICAgICAgICAgICAgdmFyaWFibGUgPSBtYXBwaW5nW3RhYmxlXQ0KICAgICAgICAgICAgY29sdW1ucyA9ICIsICIuam9pbigNCiAgICAgICAgICAgICAgICBmIntjb2x1bW59ICh7ZGF0YWZyYW1lW2NvbHVtbl0uZHR5cGV9KSINCiAgICAgICAgICAgICAgICBpZiBpbmNsdWRlX2R0eXBlcyBlbHNlIHN0cihjb2x1bW4pDQogICAgICAgICAgICAgICAgZm9yIGNvbHVtbiBpbiBkYXRhZnJhbWUuY29sdW1uc1s6ODBdDQogICAgICAgICAgICApDQogICAgICAgICAgICBpZiBsZW4oZGF0YWZyYW1lLmNvbHVtbnMpID4gODA6DQogICAgICAgICAgICAgICAgY29sdW1ucyArPSAiLCAuLi4iDQogICAgICAgICAgICB0YWJsZV9jb2x1bW5zLmFwcGVuZCgNCiAgICAgICAgICAgICAgICBmIi0gYHt2YXJpYWJsZX1gICh0YWJsZSBge3RhYmxlfWApOiB7Y29sdW1uc30iDQogICAgICAgICAgICApDQogICAgICAgIHJldHVybiAiQXZhaWxhYmxlIERhdGFGcmFtZSB2YXJpYWJsZXMgYW5kIGFjdHVhbCBjb2x1bW5zOlxuIiArICJcbiIuam9pbigNCiAgICAgICAgICAgIHRhYmxlX2NvbHVtbnMNCiAgICAgICAgKQ0K'
source_files['src/modules/rulebook_interpreter.py'] = 'IiIiDQpydWxlYm9va19pbnRlcnByZXRlci5weSDigJQgUnVsZWJvb2sgUGFyc2VyIGZvciBEYXRhTWF0ZQ0KDQpQYXJzZXMgcnVsZWJvb2subWQgdG8gZXh0cmFjdCBkZWZhdWx0IGJ1c2luZXNzIHJ1bGVzIGFuZCB2b2NhYnVsYXJ5DQpkZWZpbml0aW9ucyB0aGF0IHRoZSBhZ2VudCBtdXN0IGFwcGx5IHdoZW4gYW5zd2VyaW5nIHF1ZXN0aW9ucy4NCiIiIg0KDQppbXBvcnQgbG9nZ2luZw0KaW1wb3J0IHJlDQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgNCmZyb20gdHlwaW5nIGltcG9ydCBBbnksIERpY3QsIExpc3QsIE9wdGlvbmFsDQoNCmltcG9ydCB1bmljb2RlZGF0YQ0KDQpsb2dnZXIgPSBsb2dnaW5nLmdldExvZ2dlcihfX25hbWVfXykNCg0KDQpjbGFzcyBCdXNpbmVzc1J1bGU6DQogICAgIiIiQSBzaW5nbGUgYnVzaW5lc3MgcnVsZSBleHRyYWN0ZWQgZnJvbSB0aGUgcnVsZWJvb2suIiIiDQoNCiAgICBkZWYgX19pbml0X18oc2VsZiwga2V5d29yZDogc3RyLCBkZWZpbml0aW9uOiBzdHIsIGNhdGVnb3J5OiBzdHIgPSAiZ2VuZXJhbCIpOg0KICAgICAgICBzZWxmLmtleXdvcmQgPSBrZXl3b3JkDQogICAgICAgIHNlbGYuZGVmaW5pdGlvbiA9IGRlZmluaXRpb24NCiAgICAgICAgc2VsZi5jYXRlZ29yeSA9IGNhdGVnb3J5ICAjIGUuZy4sICJyZXZlbnVlIiwgImNvc3QiLCAidGltZSIsICJyYW5raW5nIiwgImdlbmVyYWwiDQoNCiAgICBkZWYgX19yZXByX18oc2VsZik6DQogICAgICAgIHJldHVybiBmIkJ1c2luZXNzUnVsZSh7c2VsZi5rZXl3b3JkIXJ9OiB7c2VsZi5kZWZpbml0aW9uWzo1MF19Li4uKSINCg0KDQpjbGFzcyBSdWxlYm9vazoNCiAgICAiIiJQYXJzZWQgcnVsZWJvb2sgY29udGFpbmluZyBhbGwgZGVmYXVsdCBidXNpbmVzcyBydWxlcy4iIiINCg0KICAgIGRlZiBfX2luaXRfXyhzZWxmKToNCiAgICAgICAgc2VsZi5yYXdfdGV4dDogc3RyID0gIiINCiAgICAgICAgc2VsZi5ydWxlczogTGlzdFtCdXNpbmVzc1J1bGVdID0gW10NCiAgICAgICAgc2VsZi52b2NhYnVsYXJ5OiBEaWN0W3N0ciwgc3RyXSA9IHt9DQogICAgICAgIHNlbGYuc2VjdGlvbnM6IERpY3Rbc3RyLCBzdHJdID0ge30NCg0KICAgIGRlZiBnZXRfcnVsZXNfYnlfY2F0ZWdvcnkoc2VsZiwgY2F0ZWdvcnk6IHN0cikgLT4gTGlzdFtCdXNpbmVzc1J1bGVdOg0KICAgICAgICByZXR1cm4gW3IgZm9yIHIgaW4gc2VsZi5ydWxlcyBpZiByLmNhdGVnb3J5ID09IGNhdGVnb3J5XQ0KDQogICAgZGVmIGdldF9yZWxldmFudF9ydWxlcyhzZWxmLCBxdWVzdGlvbjogc3RyKSAtPiBMaXN0W0J1c2luZXNzUnVsZV06DQogICAgICAgICIiIkZpbmQgcnVsZXMgcmVsZXZhbnQgdG8gYSBnaXZlbiBxdWVzdGlvbiBiYXNlZCBvbiBrZXl3b3JkIG1hdGNoaW5nLiIiIg0KICAgICAgICBxdWVzdGlvbl9sb3dlciA9IHF1ZXN0aW9uLmxvd2VyKCkNCiAgICAgICAgcmVsZXZhbnQgPSBbXQ0KICAgICAgICBmb3IgcnVsZSBpbiBzZWxmLnJ1bGVzOg0KICAgICAgICAgICAgIyBDaGVjayBpZiB0aGUgcnVsZSBrZXl3b3JkIGFwcGVhcnMgaW4gdGhlIHF1ZXN0aW9uDQogICAgICAgICAgICBpZiBydWxlLmtleXdvcmQubG93ZXIoKSBpbiBxdWVzdGlvbl9sb3dlcjoNCiAgICAgICAgICAgICAgICByZWxldmFudC5hcHBlbmQocnVsZSkNCiAgICAgICAgcmV0dXJuIHJlbGV2YW50DQoNCiAgICBkZWYgZ2V0X3ZvY2FidWxhcnlfbWF0Y2goc2VsZiwgdGVybTogc3RyKSAtPiBPcHRpb25hbFtzdHJdOg0KICAgICAgICAiIiJMb29rIHVwIGEgdGVybSBpbiB0aGUgcnVsZWJvb2sgdm9jYWJ1bGFyeS4iIiINCiAgICAgICAgdGVybV9sb3dlciA9IHRlcm0uc3RyaXAoKS5sb3dlcigpDQogICAgICAgIGZvciBrZXksIHZhbHVlIGluIHNlbGYudm9jYWJ1bGFyeS5pdGVtcygpOg0KICAgICAgICAgICAgaWYga2V5Lmxvd2VyKCkgPT0gdGVybV9sb3dlcjoNCiAgICAgICAgICAgICAgICByZXR1cm4gdmFsdWUNCiAgICAgICAgcmV0dXJuIE5vbmUNCg0KICAgIGRlZiB0b19wcm9tcHRfc3RyaW5nKHNlbGYsIHJlbGV2YW50X29ubHk6IE9wdGlvbmFsW3N0cl0gPSBOb25lLA0KICAgICAgICAgICAgICAgICAgICAgICAgICBtYXhfbGVuZ3RoOiBpbnQgPSAzMDAwKSAtPiBzdHI6DQogICAgICAgICIiIg0KICAgICAgICBHZW5lcmF0ZSBhIHByb21wdC1mcmllbmRseSByZXByZXNlbnRhdGlvbiBvZiB0aGUgcnVsZXMuDQoNCiAgICAgICAgUGFyYW1ldGVycw0KICAgICAgICAtLS0tLS0tLS0tDQogICAgICAgIHJlbGV2YW50X29ubHkgOiBzdHIsIG9wdGlvbmFsDQogICAgICAgICAgICBJZiBwcm92aWRlZCwgb25seSBpbmNsdWRlIHJ1bGVzIHJlbGV2YW50IHRvIHRoaXMgcXVlc3Rpb24gdGV4dC4NCiAgICAgICAgbWF4X2xlbmd0aCA6IGludA0KICAgICAgICAgICAgTWF4aW11bSBsZW5ndGggb2YgdGhlIG91dHB1dCBzdHJpbmcuDQogICAgICAgICIiIg0KICAgICAgICBpZiByZWxldmFudF9vbmx5Og0KICAgICAgICAgICAgcnVsZXMgPSBzZWxmLmdldF9yZWxldmFudF9ydWxlcyhyZWxldmFudF9vbmx5KQ0KICAgICAgICAgICAgaWYgbm90IHJ1bGVzOg0KICAgICAgICAgICAgICAgICMgRmFsbCBiYWNrIHRvIGFsbCBydWxlcyBpZiBub25lIG1hdGNoDQogICAgICAgICAgICAgICAgcnVsZXMgPSBzZWxmLnJ1bGVzDQogICAgICAgIGVsc2U6DQogICAgICAgICAgICBydWxlcyA9IHNlbGYucnVsZXMNCg0KICAgICAgICBwYXJ0cyA9IFsiRGVmYXVsdCBidXNpbmVzcyBydWxlcyAoYXBwbHkgdW5sZXNzIHRoZSBxdWVzdGlvbiBzYXlzIG90aGVyd2lzZSk6Il0NCiAgICAgICAgZm9yIHJ1bGUgaW4gcnVsZXM6DQogICAgICAgICAgICBwYXJ0cy5hcHBlbmQoZiItIHtydWxlLmtleXdvcmR9OiB7cnVsZS5kZWZpbml0aW9ufSIpDQoNCiAgICAgICAgcmVzdWx0ID0gIlxuIi5qb2luKHBhcnRzKQ0KICAgICAgICBpZiBsZW4ocmVzdWx0KSA+IG1heF9sZW5ndGg6DQogICAgICAgICAgICByZXN1bHQgPSByZXN1bHRbOm1heF9sZW5ndGggLSAzXSArICIuLi4iDQogICAgICAgIHJldHVybiByZXN1bHQNCg0KICAgIGRlZiB0b19mdWxsX3Byb21wdF9zdHJpbmcoc2VsZikgLT4gc3RyOg0KICAgICAgICAiIiJSZXR1cm4gdGhlIGNvbXBsZXRlIHJ1bGVib29rIHRleHQgZm9yIGluY2x1c2lvbiBpbiBwcm9tcHRzLiIiIg0KICAgICAgICByZXR1cm4gc2VsZi5yYXdfdGV4dA0KDQoNCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tDQojIEtleXdvcmQgY2F0ZWdvcmllcyBmb3IgYXV0by1jbGFzc2lmaWNhdGlvbg0KIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0NCg0KX0NBVEVHT1JZX0tFWVdPUkRTID0gew0KICAgICJyZXZlbnVlIjogWw0KICAgICAgICAiZG9hbmggdGh1IiwgInJldmVudWUiLCAiZG9hbmggc+G7kSIsICJ04buVbmcgdGnhu4FuIiwgInRow6BuaCB0aeG7gW4iLA0KICAgICAgICAiZ2nDoSBiw6FuIiwgInRp4buBbiBiw6FuIiwgImLDoW4gxJHGsOG7o2MiLA0KICAgIF0sDQogICAgImNvc3QiOiBbDQogICAgICAgICJnacOhIHbhu5FuIiwgImNoaSBwaMOtIiwgImNvc3QiLCAiZ2nDoSBuaOG6rXAiLCAiduG7kW4iLCAidGnhu4FuIG5o4bqtcCIsDQogICAgICAgICJwaMOtIiwgImdpw6EgbXVhIiwgIndlaWdodGVkLWF2ZXJhZ2UiLCAidHJ1bmcgYsOsbmgiLA0KICAgIF0sDQogICAgInRpbWUiOiBbDQogICAgICAgICJ0aMOhbmciLCAidHXhuqduIiwgIm5nw6B5IiwgIm7Eg20iLCAicXXDvSIsICJ3ZWVrIiwgIm1vbnRoIiwNCiAgICAgICAgIm1vbmRheSIsICJzdW5kYXkiLCAidGjhu6kiLA0KICAgIF0sDQogICAgInJhbmtpbmciOiBbDQogICAgICAgICJiw6FuIGNo4bqheSIsICJ0b3AiLCAibmhp4buBdSBuaOG6pXQiLCAiw610IG5o4bqldCIsICJjYW8gbmjhuqV0IiwgInRo4bqlcCBuaOG6pXQiLA0KICAgICAgICAieOG6v3AgaOG6oW5nIiwgInJhbmtpbmciLA0KICAgIF0sDQogICAgIm9yZGVyIjogWw0KICAgICAgICAixJHGoW4gaMOgbmciLCAib3JkZXIiLCAiaOG7p3kiLCAiY2FuY2VsbGVkIiwgInRy4bqjIiwgInJldHVybmVkIiwNCiAgICAgICAgImhvw6BuIiwgImdpYW8iLA0KICAgIF0sDQogICAgImRpc2NvdW50IjogWw0KICAgICAgICAiZ2nhuqNtIGdpw6EiLCAiZGlzY291bnQiLCAidm91Y2hlciIsICJraHV54bq/biBtw6NpIiwgImNoaeG6v3Qga2jhuqV1IiwNCiAgICBdLA0KICAgICJ0YXgiOiBbDQogICAgICAgICJ2YXQiLCAidGh14bq/IiwgInRheCIsICJwaMOtIGdpYW8iLCAiZGVsaXZlcnkiLA0KICAgIF0sDQogICAgInRldCI6IFsNCiAgICAgICAgInThur90IiwgImNo4bqhcCIsICJnacOqbmciLCAibcO5bmciLCAibHVuYXIiLCAiw6JtIGzhu4tjaCIsDQogICAgXSwNCn0NCg0KDQpkZWYgX2NsYXNzaWZ5X3J1bGUoa2V5d29yZDogc3RyLCBkZWZpbml0aW9uOiBzdHIpIC0+IHN0cjoNCiAgICAiIiJBdXRvLWNsYXNzaWZ5IGEgcnVsZSBpbnRvIGEgY2F0ZWdvcnkgYmFzZWQgb24gaXRzIGNvbnRlbnQuIiIiDQogICAgY29tYmluZWQgPSBmIntrZXl3b3JkfSB7ZGVmaW5pdGlvbn0iLmxvd2VyKCkNCiAgICBmb3IgY2F0ZWdvcnksIGtleXdvcmRzIGluIF9DQVRFR09SWV9LRVlXT1JEUy5pdGVtcygpOg0KICAgICAgICBmb3Iga3cgaW4ga2V5d29yZHM6DQogICAgICAgICAgICBpZiBrdyBpbiBjb21iaW5lZDoNCiAgICAgICAgICAgICAgICByZXR1cm4gY2F0ZWdvcnkNCiAgICByZXR1cm4gImdlbmVyYWwiDQoNCg0KIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0NCiMgUGFyc2VyDQojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQ0KDQpkZWYgX3BhcnNlX3J1bGVzX2Zyb21fdGV4dCh0ZXh0OiBzdHIpIC0+IExpc3RbQnVzaW5lc3NSdWxlXToNCiAgICAiIiJFeHRyYWN0IHJ1bGVzIGZyb20gcnVsZWJvb2sgdGV4dC4iIiINCiAgICB0ZXh0ID0gdGV4dC5yZXBsYWNlKCJcclxuIiwgIlxuIikucmVwbGFjZSgiXHIiLCAiXG4iKQ0KICAgIHJ1bGVzID0gW10NCg0KICAgICMgUGF0dGVybiAxOiAidGVybSIgPSBkZWZpbml0aW9uIChvciAidGVybSIg4oCUIGRlZmluaXRpb24pDQogICAgcGF0dGVybjEgPSByZS5jb21waWxlKA0KICAgICAgICByJ1siIiJdKFteIiIiXSspWyIiIl0nDQogICAgICAgIHInXHMqWz064oCT4oCUXVxzKicNCiAgICAgICAgcicoLis/KSg/PVxuWyIiIl18XG5cbnxcWiknLA0KICAgICAgICByZS5ET1RBTEwgfCByZS5VTklDT0RFDQogICAgKQ0KICAgIGZvciBtIGluIHBhdHRlcm4xLmZpbmRpdGVyKHRleHQpOg0KICAgICAgICBrZXl3b3JkID0gbS5ncm91cCgxKS5zdHJpcCgpDQogICAgICAgIGRlZmluaXRpb24gPSBtLmdyb3VwKDIpLnN0cmlwKCkNCiAgICAgICAgZGVmaW5pdGlvbiA9IHJlLnN1YihyJ1xzKycsICcgJywgZGVmaW5pdGlvbikNCiAgICAgICAgY2F0ZWdvcnkgPSBfY2xhc3NpZnlfcnVsZShrZXl3b3JkLCBkZWZpbml0aW9uKQ0KICAgICAgICBydWxlcy5hcHBlbmQoQnVzaW5lc3NSdWxlKGtleXdvcmQsIGRlZmluaXRpb24sIGNhdGVnb3J5KSkNCg0KICAgICMgUGF0dGVybiAyOiBidWxsZXQtcG9pbnQgcnVsZXM6IC0gKiprZXl3b3JkKio6IGRlZmluaXRpb24NCiAgICBwYXR0ZXJuMiA9IHJlLmNvbXBpbGUoDQogICAgICAgIHInWy0qXVxzK1wqP1wqPyhbXio6XG5dKz8pXCo/XCo/XHMqWzrvvJpdXHMqKC4rPykoPz1cblstKl18XG5cbnxcWiknLA0KICAgICAgICByZS5ET1RBTEwgfCByZS5VTklDT0RFDQogICAgKQ0KICAgIGZvciBtIGluIHBhdHRlcm4yLmZpbmRpdGVyKHRleHQpOg0KICAgICAgICBrZXl3b3JkID0gbS5ncm91cCgxKS5zdHJpcCgpDQogICAgICAgIGRlZmluaXRpb24gPSBtLmdyb3VwKDIpLnN0cmlwKCkNCiAgICAgICAgZGVmaW5pdGlvbiA9IHJlLnN1YihyJ1xzKycsICcgJywgZGVmaW5pdGlvbikNCiAgICAgICAgIyBBdm9pZCBkdXBsaWNhdGVzDQogICAgICAgIGlmIG5vdCBhbnkoci5rZXl3b3JkID09IGtleXdvcmQgZm9yIHIgaW4gcnVsZXMpOg0KICAgICAgICAgICAgY2F0ZWdvcnkgPSBfY2xhc3NpZnlfcnVsZShrZXl3b3JkLCBkZWZpbml0aW9uKQ0KICAgICAgICAgICAgcnVsZXMuYXBwZW5kKEJ1c2luZXNzUnVsZShrZXl3b3JkLCBkZWZpbml0aW9uLCBjYXRlZ29yeSkpDQoNCiAgICAjIFBhdHRlcm4gMzogc2ltcGxlIGxpbmVzIHdpdGggPSBvciA6IHNlcGFyYXRvcg0KICAgIHBhdHRlcm4zID0gcmUuY29tcGlsZSgNCiAgICAgICAgcideKFtBLVphLXrDgC3hu7ldW0EtWmEtesOALeG7uVxzXSo/KVxzKls9XVxzKiguKykkJywNCiAgICAgICAgcmUuTVVMVElMSU5FIHwgcmUuVU5JQ09ERQ0KICAgICkNCiAgICBmb3IgbSBpbiBwYXR0ZXJuMy5maW5kaXRlcih0ZXh0KToNCiAgICAgICAga2V5d29yZCA9IG0uZ3JvdXAoMSkuc3RyaXAoKQ0KICAgICAgICBkZWZpbml0aW9uID0gbS5ncm91cCgyKS5zdHJpcCgpDQogICAgICAgIGlmIG5vdCBhbnkoci5rZXl3b3JkID09IGtleXdvcmQgZm9yIHIgaW4gcnVsZXMpOg0KICAgICAgICAgICAgY2F0ZWdvcnkgPSBfY2xhc3NpZnlfcnVsZShrZXl3b3JkLCBkZWZpbml0aW9uKQ0KICAgICAgICAgICAgcnVsZXMuYXBwZW5kKEJ1c2luZXNzUnVsZShrZXl3b3JkLCBkZWZpbml0aW9uLCBjYXRlZ29yeSkpDQoNCiAgICAjIFBhdHRlcm4gNDogTWFya2Rvd24gdGFibGVzICh8IEx14bqtdCB8IE3hurdjIMSR4buLbmggfCwgfCBDw6FjaCBuw7NpIHwgTmdoxKlhIHwsIGV0Yy4pDQogICAgdGFibGVfcGF0dGVybiA9IHJlLmNvbXBpbGUoDQogICAgICAgIHIiXlx8XHMqKFtefFxuXSs/KVxzKlx8XHMqKFtefFxuXSs/KVxzKlx8IiwNCiAgICAgICAgcmUuTVVMVElMSU5FIHwgcmUuVU5JQ09ERQ0KICAgICkNCiAgICBmb3IgbSBpbiB0YWJsZV9wYXR0ZXJuLmZpbmRpdGVyKHRleHQpOg0KICAgICAgICBjb2wxID0gbS5ncm91cCgxKS5zdHJpcCgpDQogICAgICAgIGNvbDIgPSBtLmdyb3VwKDIpLnN0cmlwKCkNCiAgICAgICAgaWYgbm90IGNvbDEgb3Igbm90IGNvbDI6DQogICAgICAgICAgICBjb250aW51ZQ0KICAgICAgICBjb2wxX2NsZWFuID0gY29sMS5sb3dlcigpDQogICAgICAgIGlmIGNvbDFfY2xlYW4gaW4gKCJsdeG6rXQiLCAiY8OhY2ggbsOzaSIsICJjw6FjaCBnaGkiLCAidOG7qyIsICJ0eXBlIiwgIm5nw6BuaCIsICLEkcO6bmciLCAic2FpIikgb3IgIi0tLSIgaW4gY29sMSBvciAiLS0tIiBpbiBjb2wyOg0KICAgICAgICAgICAgY29udGludWUNCiAgICAgICAga2V5d29yZCA9IGNvbDENCiAgICAgICAgZGVmaW5pdGlvbiA9IHJlLnN1YihyJ1xzKycsICcgJywgY29sMikNCiAgICAgICAgaWYgbm90IGFueShyLmtleXdvcmQubG93ZXIoKSA9PSBrZXl3b3JkLmxvd2VyKCkgZm9yIHIgaW4gcnVsZXMpOg0KICAgICAgICAgICAgY2F0ZWdvcnkgPSBfY2xhc3NpZnlfcnVsZShrZXl3b3JkLCBkZWZpbml0aW9uKQ0KICAgICAgICAgICAgcnVsZXMuYXBwZW5kKEJ1c2luZXNzUnVsZShrZXl3b3JkLCBkZWZpbml0aW9uLCBjYXRlZ29yeSkpDQoNCiAgICByZXR1cm4gcnVsZXMNCg0KDQpkZWYgX3BhcnNlX3ZvY2FidWxhcnkodGV4dDogc3RyKSAtPiBEaWN0W3N0ciwgc3RyXToNCiAgICAiIiJFeHRyYWN0IHZvY2FidWxhcnkvdGVybWlub2xvZ3kgZGVmaW5pdGlvbnMgZnJvbSB0aGUgcnVsZWJvb2suIiIiDQogICAgdm9jYWIgPSB7fQ0KDQogICAgIyBMb29rIGZvciB2b2NhYnVsYXJ5L2dsb3NzYXJ5IHNlY3Rpb25zDQogICAgc2VjdGlvbnMgPSByZS5zcGxpdChyJ14jezEsM31ccysnLCB0ZXh0LCBmbGFncz1yZS5NVUxUSUxJTkUpDQogICAgZm9yIHNlY3Rpb24gaW4gc2VjdGlvbnM6DQogICAgICAgICMgRmluZCBkZWZpbml0aW9uIHBhdHRlcm5zDQogICAgICAgIGZvciBtIGluIHJlLmZpbmRpdGVyKA0KICAgICAgICAgICAgcidbLSpdXHMqWyIiIl0/KFteIiIiOlxuXSs/KVsiIiJdP1xzKls6PeKAk+KAlF1ccyooLis/KSg/PVxuWy0qXXxcblxufFxaKScsDQogICAgICAgICAgICBzZWN0aW9uLCByZS5ET1RBTEwgfCByZS5VTklDT0RFDQogICAgICAgICk6DQogICAgICAgICAgICB0ZXJtID0gbS5ncm91cCgxKS5zdHJpcCgpDQogICAgICAgICAgICBkZWZpbml0aW9uID0gbS5ncm91cCgyKS5zdHJpcCgpDQogICAgICAgICAgICBkZWZpbml0aW9uID0gcmUuc3ViKHInXHMrJywgJyAnLCBkZWZpbml0aW9uKQ0KICAgICAgICAgICAgaWYgbGVuKHRlcm0pIDwgNTA6ICAjIHJlYXNvbmFibGUgdGVybSBsZW5ndGgNCiAgICAgICAgICAgICAgICB2b2NhYlt0ZXJtXSA9IGRlZmluaXRpb24NCg0KICAgIHJldHVybiB2b2NhYg0KDQoNCmRlZiBwYXJzZV9ydWxlYm9vayhydWxlYm9va19wYXRoOiBzdHIpIC0+IFJ1bGVib29rOg0KICAgICIiIg0KICAgIFBhcnNlIGEgcnVsZWJvb2subWQgZmlsZSBhbmQgZXh0cmFjdCBzdHJ1Y3R1cmVkIGJ1c2luZXNzIHJ1bGVzLg0KDQogICAgUGFyYW1ldGVycw0KICAgIC0tLS0tLS0tLS0NCiAgICBydWxlYm9va19wYXRoIDogc3RyDQogICAgICAgIFBhdGggdG8gdGhlIHJ1bGVib29rLm1kIGZpbGUuDQoNCiAgICBSZXR1cm5zDQogICAgLS0tLS0tLQ0KICAgIFJ1bGVib29rDQogICAgICAgIFBhcnNlZCBydWxlYm9vayB3aXRoIHJ1bGVzIGFuZCB2b2NhYnVsYXJ5Lg0KICAgICIiIg0KICAgIHJ1bGVib29rID0gUnVsZWJvb2soKQ0KICAgIHBhdGggPSBQYXRoKHJ1bGVib29rX3BhdGgpDQoNCiAgICBpZiBub3QgcGF0aC5leGlzdHMoKToNCiAgICAgICAgbG9nZ2VyLndhcm5pbmcoIlJ1bGVib29rIG5vdCBmb3VuZDogJXMiLCBydWxlYm9va19wYXRoKQ0KICAgICAgICByZXR1cm4gcnVsZWJvb2sNCg0KICAgIHRyeToNCiAgICAgICAgcmF3ID0gcGF0aC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikNCiAgICBleGNlcHQgVW5pY29kZURlY29kZUVycm9yOg0KICAgICAgICByYXcgPSBwYXRoLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgtc2lnIikNCg0KICAgIHJhdyA9IHVuaWNvZGVkYXRhLm5vcm1hbGl6ZSgiTkZDIiwgcmF3KQ0KICAgIHJ1bGVib29rLnJhd190ZXh0ID0gcmF3DQoNCiAgICAjIEV4dHJhY3Qgc2VjdGlvbnMNCiAgICBjdXJyZW50X2hlYWRlciA9ICJpbnRybyINCiAgICBjdXJyZW50X2xpbmVzID0gW10NCiAgICBmb3IgbGluZSBpbiByYXcuc3BsaXQoIlxuIik6DQogICAgICAgIGhlYWRlcl9tYXRjaCA9IHJlLm1hdGNoKHIiXiN7MSwzfVxzKyguKykiLCBsaW5lKQ0KICAgICAgICBpZiBoZWFkZXJfbWF0Y2g6DQogICAgICAgICAgICBpZiBjdXJyZW50X2xpbmVzOg0KICAgICAgICAgICAgICAgIHJ1bGVib29rLnNlY3Rpb25zW2N1cnJlbnRfaGVhZGVyXSA9ICJcbiIuam9pbihjdXJyZW50X2xpbmVzKS5zdHJpcCgpDQogICAgICAgICAgICBjdXJyZW50X2hlYWRlciA9IGhlYWRlcl9tYXRjaC5ncm91cCgxKS5zdHJpcCgpDQogICAgICAgICAgICBjdXJyZW50X2xpbmVzID0gW10NCiAgICAgICAgZWxzZToNCiAgICAgICAgICAgIGN1cnJlbnRfbGluZXMuYXBwZW5kKGxpbmUpDQogICAgaWYgY3VycmVudF9saW5lczoNCiAgICAgICAgcnVsZWJvb2suc2VjdGlvbnNbY3VycmVudF9oZWFkZXJdID0gIlxuIi5qb2luKGN1cnJlbnRfbGluZXMpLnN0cmlwKCkNCg0KICAgICMgUGFyc2UgcnVsZXMNCiAgICBydWxlYm9vay5ydWxlcyA9IF9wYXJzZV9ydWxlc19mcm9tX3RleHQocmF3KQ0KDQogICAgIyBQYXJzZSB2b2NhYnVsYXJ5DQogICAgcnVsZWJvb2sudm9jYWJ1bGFyeSA9IF9wYXJzZV92b2NhYnVsYXJ5KHJhdykNCg0KICAgIGxvZ2dlci5pbmZvKCJQYXJzZWQgcnVsZWJvb2sgJXM6ICVkIHJ1bGVzLCAlZCB2b2NhYnVsYXJ5IGVudHJpZXMiLA0KICAgICAgICAgICAgICAgIHJ1bGVib29rX3BhdGgsIGxlbihydWxlYm9vay5ydWxlcyksIGxlbihydWxlYm9vay52b2NhYnVsYXJ5KSkNCg0KICAgIHJldHVybiBydWxlYm9vaw0K'
source_files['src/modules/schema_registry.py'] = 'IiIiDQpzY2hlbWFfcmVnaXN0cnkucHkg4oCUIFVuaWZpZWQgU2NoZW1hIFJlZ2lzdHJ5IGZvciBEYXRhTWF0ZQ0KDQpCdWlsZHMgYSBzdHJ1Y3R1cmVkIG1ldGFkYXRhIHN0b3JlIGZvciBlYWNoIGJ1c2luZXNzJ3MgZGF0YSB0YWJsZXMuDQpVc2VkIGRvd25zdHJlYW0gZm9yIHByb21wdCBjb25zdHJ1Y3Rpb24sIGNvZGUgZ2VuZXJhdGlvbiBncm91bmRpbmcsDQphbmQgZGF0YSB2YWxpZGF0aW9uLg0KIiIiDQoNCmltcG9ydCBsb2dnaW5nDQpmcm9tIHR5cGluZyBpbXBvcnQgQW55LCBEaWN0LCBMaXN0LCBPcHRpb25hbA0KDQppbXBvcnQgcGFuZGFzIGFzIHBkDQppbXBvcnQgdW5pY29kZWRhdGENCg0KbG9nZ2VyID0gbG9nZ2luZy5nZXRMb2dnZXIoX19uYW1lX18pDQoNCg0KY2xhc3MgQ29sdW1uSW5mbzoNCiAgICAiIiJNZXRhZGF0YSBmb3IgYSBzaW5nbGUgY29sdW1uLiIiIg0KDQogICAgX19zbG90c19fID0gKCJuYW1lIiwgImR0eXBlIiwgIm5vbl9udWxsX2NvdW50IiwgInRvdGFsX2NvdW50IiwNCiAgICAgICAgICAgICAgICAgInVuaXF1ZV9jb3VudCIsICJzYW1wbGVfdmFsdWVzIiwgIm1pbl92YWwiLCAibWF4X3ZhbCIsDQogICAgICAgICAgICAgICAgICJpc19udW1lcmljIiwgImlzX2RhdGV0aW1lIiwgImlzX2NhdGVnb3JpY2FsIikNCg0KICAgIGRlZiBfX2luaXRfXyhzZWxmLCBuYW1lOiBzdHIsIHNlcmllczogcGQuU2VyaWVzKToNCiAgICAgICAgc2VsZi5uYW1lID0gbmFtZQ0KICAgICAgICBzZWxmLmR0eXBlID0gc3RyKHNlcmllcy5kdHlwZSkNCiAgICAgICAgc2VsZi50b3RhbF9jb3VudCA9IGxlbihzZXJpZXMpDQogICAgICAgIHNlbGYubm9uX251bGxfY291bnQgPSBpbnQoc2VyaWVzLm5vdG5hKCkuc3VtKCkpDQogICAgICAgIHNlbGYudW5pcXVlX2NvdW50ID0gaW50KHNlcmllcy5udW5pcXVlKCkpDQoNCiAgICAgICAgbm9uX251bGwgPSBzZXJpZXMuZHJvcG5hKCkNCiAgICAgICAgc2VsZi5zYW1wbGVfdmFsdWVzID0gbm9uX251bGwuaGVhZCg1KS50b2xpc3QoKQ0KDQogICAgICAgIHNlbGYuaXNfbnVtZXJpYyA9IHBkLmFwaS50eXBlcy5pc19udW1lcmljX2R0eXBlKHNlcmllcykNCiAgICAgICAgc2VsZi5pc19kYXRldGltZSA9IHBkLmFwaS50eXBlcy5pc19kYXRldGltZTY0X2FueV9kdHlwZShzZXJpZXMpDQogICAgICAgIHNlbGYuaXNfY2F0ZWdvcmljYWwgPSAoDQogICAgICAgICAgICBub3Qgc2VsZi5pc19udW1lcmljDQogICAgICAgICAgICBhbmQgbm90IHNlbGYuaXNfZGF0ZXRpbWUNCiAgICAgICAgICAgIGFuZCBzZWxmLnVuaXF1ZV9jb3VudCA8PSA1MA0KICAgICAgICAgICAgYW5kIHNlbGYubm9uX251bGxfY291bnQgPiAwDQogICAgICAgICkNCg0KICAgICAgICBpZiBzZWxmLmlzX251bWVyaWMgYW5kIHNlbGYubm9uX251bGxfY291bnQgPiAwOg0KICAgICAgICAgICAgc2VsZi5taW5fdmFsID0gZmxvYXQobm9uX251bGwubWluKCkpDQogICAgICAgICAgICBzZWxmLm1heF92YWwgPSBmbG9hdChub25fbnVsbC5tYXgoKSkNCiAgICAgICAgZWxpZiBzZWxmLmlzX2RhdGV0aW1lIGFuZCBzZWxmLm5vbl9udWxsX2NvdW50ID4gMDoNCiAgICAgICAgICAgIHNlbGYubWluX3ZhbCA9IHN0cihub25fbnVsbC5taW4oKSkNCiAgICAgICAgICAgIHNlbGYubWF4X3ZhbCA9IHN0cihub25fbnVsbC5tYXgoKSkNCiAgICAgICAgZWxzZToNCiAgICAgICAgICAgIHNlbGYubWluX3ZhbCA9IE5vbmUNCiAgICAgICAgICAgIHNlbGYubWF4X3ZhbCA9IE5vbmUNCg0KICAgIGRlZiB0b19kaWN0KHNlbGYpIC0+IERpY3Rbc3RyLCBBbnldOg0KICAgICAgICByZXR1cm4gew0KICAgICAgICAgICAgIm5hbWUiOiBzZWxmLm5hbWUsDQogICAgICAgICAgICAiZHR5cGUiOiBzZWxmLmR0eXBlLA0KICAgICAgICAgICAgIm5vbl9udWxsIjogc2VsZi5ub25fbnVsbF9jb3VudCwNCiAgICAgICAgICAgICJ0b3RhbCI6IHNlbGYudG90YWxfY291bnQsDQogICAgICAgICAgICAidW5pcXVlIjogc2VsZi51bmlxdWVfY291bnQsDQogICAgICAgICAgICAic2FtcGxlcyI6IHNlbGYuc2FtcGxlX3ZhbHVlc1s6M10sDQogICAgICAgICAgICAibWluIjogc2VsZi5taW5fdmFsLA0KICAgICAgICAgICAgIm1heCI6IHNlbGYubWF4X3ZhbCwNCiAgICAgICAgICAgICJpc19udW1lcmljIjogc2VsZi5pc19udW1lcmljLA0KICAgICAgICAgICAgImlzX2RhdGV0aW1lIjogc2VsZi5pc19kYXRldGltZSwNCiAgICAgICAgICAgICJpc19jYXRlZ29yaWNhbCI6IHNlbGYuaXNfY2F0ZWdvcmljYWwsDQogICAgICAgIH0NCg0KDQpjbGFzcyBUYWJsZVNjaGVtYToNCiAgICAiIiJTY2hlbWEgZm9yIGEgc2luZ2xlIHRhYmxlL3NoZWV0LiIiIg0KDQogICAgZGVmIF9faW5pdF9fKHNlbGYsIG5hbWU6IHN0ciwgZGY6IHBkLkRhdGFGcmFtZSk6DQogICAgICAgIHNlbGYubmFtZSA9IG5hbWUNCiAgICAgICAgc2VsZi5yb3dfY291bnQgPSBsZW4oZGYpDQogICAgICAgIHNlbGYuY29sdW1uczogTGlzdFtDb2x1bW5JbmZvXSA9IFsNCiAgICAgICAgICAgIENvbHVtbkluZm8oY29sLCBkZltjb2xdKSBmb3IgY29sIGluIGRmLmNvbHVtbnMNCiAgICAgICAgXQ0KICAgICAgICBzZWxmLmNvbHVtbl9uYW1lcyA9IFtjLm5hbWUgZm9yIGMgaW4gc2VsZi5jb2x1bW5zXQ0KDQogICAgZGVmIGdldF9jb2x1bW4oc2VsZiwgbmFtZTogc3RyKSAtPiBPcHRpb25hbFtDb2x1bW5JbmZvXToNCiAgICAgICAgZm9yIGMgaW4gc2VsZi5jb2x1bW5zOg0KICAgICAgICAgICAgaWYgYy5uYW1lID09IG5hbWU6DQogICAgICAgICAgICAgICAgcmV0dXJuIGMNCiAgICAgICAgcmV0dXJuIE5vbmUNCg0KICAgIGRlZiB0b19wcm9tcHRfc3RyaW5nKHNlbGYsIGluY2x1ZGVfc2FtcGxlczogYm9vbCA9IFRydWUsIG1heF9zYW1wbGVfdmFsdWVzOiBpbnQgPSA1KSAtPiBzdHI6DQogICAgICAgICIiIkdlbmVyYXRlIGEgY29uY2lzZSBzY2hlbWEgZGVzY3JpcHRpb24gZm9yIExMTSBwcm9tcHRzLiIiIg0KICAgICAgICBsaW5lcyA9IFtmIlRhYmxlOiB7c2VsZi5uYW1lfSAoe3NlbGYucm93X2NvdW50fSByb3dzKSJdDQogICAgICAgIGxpbmVzLmFwcGVuZCgiQ29sdW1uczoiKQ0KICAgICAgICBmb3IgY29sIGluIHNlbGYuY29sdW1uczoNCiAgICAgICAgICAgIHBhcnRzID0gW2YiICAtIHtjb2wubmFtZX0gKHtjb2wuZHR5cGV9KSJdDQogICAgICAgICAgICBpZiBjb2wuaXNfY2F0ZWdvcmljYWwgYW5kIGNvbC51bmlxdWVfY291bnQgPD0gMjA6DQogICAgICAgICAgICAgICAgdmFscyA9IGNvbC5zYW1wbGVfdmFsdWVzWzptYXhfc2FtcGxlX3ZhbHVlc10NCiAgICAgICAgICAgICAgICBwYXJ0cy5hcHBlbmQoZiIgW2NhdGVnb3JpZXM6IHt2YWxzfV0iKQ0KICAgICAgICAgICAgZWxpZiBpbmNsdWRlX3NhbXBsZXMgYW5kIGNvbC5zYW1wbGVfdmFsdWVzOg0KICAgICAgICAgICAgICAgIHZhbHMgPSBjb2wuc2FtcGxlX3ZhbHVlc1s6M10NCiAgICAgICAgICAgICAgICBwYXJ0cy5hcHBlbmQoZiIgW2UuZy4ge3ZhbHN9XSIpDQogICAgICAgICAgICBpZiBjb2wubWluX3ZhbCBpcyBub3QgTm9uZToNCiAgICAgICAgICAgICAgICBwYXJ0cy5hcHBlbmQoZiIgW3JhbmdlOiB7Y29sLm1pbl92YWx9IOKAkyB7Y29sLm1heF92YWx9XSIpDQogICAgICAgICAgICBsaW5lcy5hcHBlbmQoIiIuam9pbihwYXJ0cykpDQogICAgICAgIHJldHVybiAiXG4iLmpvaW4obGluZXMpDQoNCg0KY2xhc3MgU2NoZW1hUmVnaXN0cnk6DQogICAgIiIiUmVnaXN0cnkgaG9sZGluZyBzY2hlbWFzIGZvciBhbGwgdGFibGVzIGluIGEgYnVzaW5lc3MncyBkYXRhLiIiIg0KDQogICAgZGVmIF9faW5pdF9fKHNlbGYpOg0KICAgICAgICBzZWxmLnRhYmxlczogRGljdFtzdHIsIFRhYmxlU2NoZW1hXSA9IHt9DQoNCiAgICBkZWYgcmVnaXN0ZXIoc2VsZiwgbmFtZTogc3RyLCBkZjogcGQuRGF0YUZyYW1lKToNCiAgICAgICAgIiIiUmVnaXN0ZXIgYSB0YWJsZSBpbnRvIHRoZSBzY2hlbWEgcmVnaXN0cnkuIiIiDQogICAgICAgIHNjaGVtYSA9IFRhYmxlU2NoZW1hKG5hbWUsIGRmKQ0KICAgICAgICBzZWxmLnRhYmxlc1tuYW1lXSA9IHNjaGVtYQ0KICAgICAgICBsb2dnZXIuaW5mbygiUmVnaXN0ZXJlZCB0YWJsZSAnJXMnOiAlZCByb3dzLCAlZCBjb2x1bW5zIiwNCiAgICAgICAgICAgICAgICAgICAgIG5hbWUsIHNjaGVtYS5yb3dfY291bnQsIGxlbihzY2hlbWEuY29sdW1ucykpDQoNCiAgICBkZWYgYnVpbGRfZnJvbV9kaWN0KHNlbGYsIHRhYmxlczogRGljdFtzdHIsIHBkLkRhdGFGcmFtZV0pIC0+ICJTY2hlbWFSZWdpc3RyeSI6DQogICAgICAgICIiIkJ1aWxkIHRoZSByZWdpc3RyeSBmcm9tIGEgZGljdCBvZiB7bmFtZTogRGF0YUZyYW1lfS4iIiINCiAgICAgICAgZm9yIG5hbWUsIGRmIGluIHRhYmxlcy5pdGVtcygpOg0KICAgICAgICAgICAgc2VsZi5yZWdpc3RlcihuYW1lLCBkZikNCiAgICAgICAgcmV0dXJuIHNlbGYNCg0KICAgIGRlZiBnZXRfdGFibGUoc2VsZiwgbmFtZTogc3RyKSAtPiBPcHRpb25hbFtUYWJsZVNjaGVtYV06DQogICAgICAgIHJldHVybiBzZWxmLnRhYmxlcy5nZXQobmFtZSkNCg0KICAgIGRlZiB0YWJsZV9uYW1lcyhzZWxmKSAtPiBMaXN0W3N0cl06DQogICAgICAgIHJldHVybiBsaXN0KHNlbGYudGFibGVzLmtleXMoKSkNCg0KICAgIGRlZiBnZXRfdGFibGVfbmFtZXMoc2VsZikgLT4gTGlzdFtzdHJdOg0KICAgICAgICAiIiJBbGlhcyBmb3IgdGFibGVfbmFtZXMuIiIiDQogICAgICAgIHJldHVybiBzZWxmLnRhYmxlX25hbWVzKCkNCg0KICAgIGRlZiB0b19wcm9tcHRfc3RyaW5nKHNlbGYsIGluY2x1ZGVfc2FtcGxlczogYm9vbCA9IFRydWUpIC0+IHN0cjoNCiAgICAgICAgIiIiR2VuZXJhdGUgYSBmdWxsIHNjaGVtYSBkZXNjcmlwdGlvbiBmb3IgTExNIHByb21wdHMuIiIiDQogICAgICAgIHBhcnRzID0gW10NCiAgICAgICAgZm9yIHNjaGVtYSBpbiBzZWxmLnRhYmxlcy52YWx1ZXMoKToNCiAgICAgICAgICAgIHBhcnRzLmFwcGVuZChzY2hlbWEudG9fcHJvbXB0X3N0cmluZyhpbmNsdWRlX3NhbXBsZXM9aW5jbHVkZV9zYW1wbGVzKSkNCiAgICAgICAgcmV0dXJuICJcblxuIi5qb2luKHBhcnRzKQ0KDQogICAgZGVmIHRvX2NvbXBhY3Rfc3RyaW5nKHNlbGYpIC0+IHN0cjoNCiAgICAgICAgIiIiR2VuZXJhdGUgYSBjb21wYWN0IHNjaGVtYSAoanVzdCB0YWJsZSBuYW1lcyBhbmQgY29sdW1uIG5hbWVzK3R5cGVzKS4iIiINCiAgICAgICAgcGFydHMgPSBbXQ0KICAgICAgICBmb3Igc2NoZW1hIGluIHNlbGYudGFibGVzLnZhbHVlcygpOg0KICAgICAgICAgICAgY29scyA9ICIsICIuam9pbihmIntjLm5hbWV9KHtjLmR0eXBlfSkiIGZvciBjIGluIHNjaGVtYS5jb2x1bW5zKQ0KICAgICAgICAgICAgcGFydHMuYXBwZW5kKGYie3NjaGVtYS5uYW1lfSBbe3NjaGVtYS5yb3dfY291bnR9IHJvd3NdOiB7Y29sc30iKQ0KICAgICAgICByZXR1cm4gIlxuIi5qb2luKHBhcnRzKQ0KDQogICAgZGVmIGZpbmRfZGF0ZV9jb2x1bW5zKHNlbGYpIC0+IERpY3Rbc3RyLCBMaXN0W3N0cl1dOg0KICAgICAgICAiIiJGaW5kIGFsbCBkYXRldGltZSBjb2x1bW5zIGFjcm9zcyBhbGwgdGFibGVzLiIiIg0KICAgICAgICByZXN1bHQgPSB7fQ0KICAgICAgICBmb3IgbmFtZSwgc2NoZW1hIGluIHNlbGYudGFibGVzLml0ZW1zKCk6DQogICAgICAgICAgICBkYXRlX2NvbHMgPSBbYy5uYW1lIGZvciBjIGluIHNjaGVtYS5jb2x1bW5zIGlmIGMuaXNfZGF0ZXRpbWVdDQogICAgICAgICAgICBpZiBkYXRlX2NvbHM6DQogICAgICAgICAgICAgICAgcmVzdWx0W25hbWVdID0gZGF0ZV9jb2xzDQogICAgICAgIHJldHVybiByZXN1bHQNCg0KICAgIGRlZiBmaW5kX251bWVyaWNfY29sdW1ucyhzZWxmKSAtPiBEaWN0W3N0ciwgTGlzdFtzdHJdXToNCiAgICAgICAgIiIiRmluZCBhbGwgbnVtZXJpYyBjb2x1bW5zIGFjcm9zcyBhbGwgdGFibGVzLiIiIg0KICAgICAgICByZXN1bHQgPSB7fQ0KICAgICAgICBmb3IgbmFtZSwgc2NoZW1hIGluIHNlbGYudGFibGVzLml0ZW1zKCk6DQogICAgICAgICAgICBudW1fY29scyA9IFtjLm5hbWUgZm9yIGMgaW4gc2NoZW1hLmNvbHVtbnMgaWYgYy5pc19udW1lcmljXQ0KICAgICAgICAgICAgaWYgbnVtX2NvbHM6DQogICAgICAgICAgICAgICAgcmVzdWx0W25hbWVdID0gbnVtX2NvbHMNCiAgICAgICAgcmV0dXJuIHJlc3VsdA0K'
source_files['src/modules/time_resolver.py'] = 'IiIiDQp0aW1lX3Jlc29sdmVyLnB5IOKAlCBWaWV0bmFtZXNlIFRpbWUgRXhwcmVzc2lvbiBSZXNvbHZlciBmb3IgRGF0YU1hdGUNCg0KUmVzb2x2ZXMgcmVsYXRpdmUgVmlldG5hbWVzZSB0aW1lIGV4cHJlc3Npb25zIChlLmcuLCAidGjDoW5nIG7DoHkiLCAidHXhuqduIHLhu5NpIiwNCiJjw7luZyBr4buzIG7Eg20gbmdvw6FpIikgaW50byBjb25jcmV0ZSAoc3RhcnRfZGF0ZSwgZW5kX2RhdGUpIHJhbmdlcyB1c2luZw0KdGhlIGFzX29mX2RhdGUgYW5kIHRoZSBzb2xhci1sdW5hciBjYWxlbmRhci4NCiIiIg0KDQppbXBvcnQgY3N2DQppbXBvcnQgbG9nZ2luZw0KaW1wb3J0IHJlDQpmcm9tIGRhdGV0aW1lIGltcG9ydCBkYXRlLCBkYXRldGltZSwgdGltZWRlbHRhDQpmcm9tIHR5cGluZyBpbXBvcnQgRGljdCwgTGlzdCwgT3B0aW9uYWwsIFR1cGxlDQoNCmxvZ2dlciA9IGxvZ2dpbmcuZ2V0TG9nZ2VyKF9fbmFtZV9fKQ0KDQoNCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tDQojIEx1bmFyIGNhbGVuZGFyIGxvYWRlcg0KIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0NCg0KY2xhc3MgTHVuYXJDYWxlbmRhcjoNCiAgICAiIiINCiAgICBTb2xhci1sdW5hciBjYWxlbmRhciBsb2FkZWQgZnJvbSBjYWxlbmRhci5jc3YuDQogICAgUHJvdmlkZXMgbWFwcGluZ3MgYmV0d2VlbiBzb2xhciBhbmQgbHVuYXIgZGF0ZXMgZm9yIFThur90LCB0aMOhbmcgQ2jhuqFwLCBldGMuDQogICAgIiIiDQoNCiAgICBkZWYgX19pbml0X18oc2VsZik6DQogICAgICAgICMgc29sYXJfZGF0ZSAtPiAobHVuYXJfeWVhciwgbHVuYXJfbW9udGgsIGx1bmFyX2RheSkNCiAgICAgICAgc2VsZi5zb2xhcl90b19sdW5hcjogRGljdFtkYXRlLCBUdXBsZVtpbnQsIGludCwgaW50XV0gPSB7fQ0KICAgICAgICAjIChsdW5hcl95ZWFyLCBsdW5hcl9tb250aCwgbHVuYXJfZGF5KSAtPiBzb2xhcl9kYXRlDQogICAgICAgIHNlbGYubHVuYXJfdG9fc29sYXI6IERpY3RbVHVwbGVbaW50LCBpbnQsIGludF0sIGRhdGVdID0ge30NCg0KICAgIGRlZiBsb2FkKHNlbGYsIGNhbGVuZGFyX3BhdGg6IHN0cik6DQogICAgICAgICIiIkxvYWQgY2FsZW5kYXIuY3N2IG1hcHBpbmcgc29sYXIg4oaUIGx1bmFyIGRhdGVzLiIiIg0KICAgICAgICB0cnk6DQogICAgICAgICAgICB3aXRoIG9wZW4oY2FsZW5kYXJfcGF0aCwgInIiLCBlbmNvZGluZz0idXRmLTgiKSBhcyBmOg0KICAgICAgICAgICAgICAgIHJlYWRlciA9IGNzdi5EaWN0UmVhZGVyKGYpDQogICAgICAgICAgICAgICAgZm9yIHJvdyBpbiByZWFkZXI6DQogICAgICAgICAgICAgICAgICAgIHRyeToNCiAgICAgICAgICAgICAgICAgICAgICAgIHNvbGFyID0gZGF0ZS5mcm9taXNvZm9ybWF0KHJvdy5nZXQoInNvbGFyX2RhdGUiLCByb3cuZ2V0KCJkYXRlIiwgIiIpKSkNCiAgICAgICAgICAgICAgICAgICAgICAgIGx1bmFyX3kgPSBpbnQocm93LmdldCgibHVuYXJfeWVhciIsIDApKQ0KICAgICAgICAgICAgICAgICAgICAgICAgbHVuYXJfbSA9IGludChyb3cuZ2V0KCJsdW5hcl9tb250aCIsIDApKQ0KICAgICAgICAgICAgICAgICAgICAgICAgbHVuYXJfZCA9IGludChyb3cuZ2V0KCJsdW5hcl9kYXkiLCAwKSkNCiAgICAgICAgICAgICAgICAgICAgICAgIGlmIGx1bmFyX3kgYW5kIGx1bmFyX20gYW5kIGx1bmFyX2Q6DQogICAgICAgICAgICAgICAgICAgICAgICAgICAgc2VsZi5zb2xhcl90b19sdW5hcltzb2xhcl0gPSAobHVuYXJfeSwgbHVuYXJfbSwgbHVuYXJfZCkNCiAgICAgICAgICAgICAgICAgICAgICAgICAgICBzZWxmLmx1bmFyX3RvX3NvbGFyWyhsdW5hcl95LCBsdW5hcl9tLCBsdW5hcl9kKV0gPSBzb2xhcg0KICAgICAgICAgICAgICAgICAgICBleGNlcHQgKFZhbHVlRXJyb3IsIEtleUVycm9yKToNCiAgICAgICAgICAgICAgICAgICAgICAgIGNvbnRpbnVlDQogICAgICAgICAgICBsb2dnZXIuaW5mbygiTG9hZGVkIGx1bmFyIGNhbGVuZGFyOiAlZCBlbnRyaWVzIiwgbGVuKHNlbGYuc29sYXJfdG9fbHVuYXIpKQ0KICAgICAgICBleGNlcHQgRmlsZU5vdEZvdW5kRXJyb3I6DQogICAgICAgICAgICBsb2dnZXIud2FybmluZygiQ2FsZW5kYXIgZmlsZSBub3QgZm91bmQ6ICVzIiwgY2FsZW5kYXJfcGF0aCkNCg0KICAgIGRlZiBnZXRfc29sYXJfZGF0ZShzZWxmLCBsdW5hcl95ZWFyOiBpbnQsIGx1bmFyX21vbnRoOiBpbnQsIGx1bmFyX2RheTogaW50KSAtPiBPcHRpb25hbFtkYXRlXToNCiAgICAgICAgIiIiQ29udmVydCBsdW5hciBkYXRlIHRvIHNvbGFyIGRhdGUuIiIiDQogICAgICAgIHJldHVybiBzZWxmLmx1bmFyX3RvX3NvbGFyLmdldCgobHVuYXJfeWVhciwgbHVuYXJfbW9udGgsIGx1bmFyX2RheSkpDQoNCiAgICBkZWYgZ2V0X2x1bmFyX2RhdGUoc2VsZiwgc29sYXJfZGF0ZTogZGF0ZSkgLT4gT3B0aW9uYWxbVHVwbGVbaW50LCBpbnQsIGludF1dOg0KICAgICAgICAiIiJDb252ZXJ0IHNvbGFyIGRhdGUgdG8gbHVuYXIgZGF0ZS4iIiINCiAgICAgICAgcmV0dXJuIHNlbGYuc29sYXJfdG9fbHVuYXIuZ2V0KHNvbGFyX2RhdGUpDQoNCiAgICBkZWYgZmluZF90ZXRfd2luZG93KHNlbGYsIHNvbGFyX3llYXI6IGludCkgLT4gT3B0aW9uYWxbVHVwbGVbZGF0ZSwgZGF0ZV1dOg0KICAgICAgICAiIiINCiAgICAgICAgRmluZCB0aGUgVOG6v3Qgd2luZG93OiAyMyB0aMOhbmcgQ2jhuqFwIHRvIG3DuW5nIDEwIHRow6FuZyBHacOqbmcuDQogICAgICAgIHRow6FuZyBDaOG6oXAgPSBsdW5hciBtb250aCAxMiwgdGjDoW5nIEdpw6puZyA9IGx1bmFyIG1vbnRoIDEuDQogICAgICAgIFRoZSBU4bq/dCBvZiBzb2xhcl95ZWFyIG1lYW5zIHRoZSBvbmUgdGhhdCBmYWxscyBpbi9hcm91bmQgSmFuLUZlYiBvZiBzb2xhcl95ZWFyLg0KICAgICAgICAiIiINCiAgICAgICAgIyAyMyB0aMOhbmcgQ2jhuqFwIG9mIGx1bmFyIHllYXIgKHNvbGFyX3llYXIgLSAxKQ0KICAgICAgICBzdGFydCA9IHNlbGYuZ2V0X3NvbGFyX2RhdGUoc29sYXJfeWVhciAtIDEsIDEyLCAyMykNCiAgICAgICAgIyBtw7luZyAxMCB0aMOhbmcgR2nDqm5nIG9mIGx1bmFyIHllYXIgc29sYXJfeWVhcg0KICAgICAgICBlbmQgPSBzZWxmLmdldF9zb2xhcl9kYXRlKHNvbGFyX3llYXIsIDEsIDEwKQ0KDQogICAgICAgIGlmIHN0YXJ0IGFuZCBlbmQ6DQogICAgICAgICAgICByZXR1cm4gKHN0YXJ0LCBlbmQpDQogICAgICAgIHJldHVybiBOb25lDQoNCg0KIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0NCiMgVGltZSBleHByZXNzaW9uIHBhdHRlcm5zDQojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQ0KDQojIFZpZXRuYW1lc2UgbW9udGggbmFtZXMgdG8gbW9udGggbnVtYmVycw0KX01PTlRIX05BTUVTID0gew0KICAgICJnacOqbmciOiAxLCAibeG7mXQiOiAxLCAiaGFpIjogMiwgImJhIjogMywgInTGsCI6IDQsICJi4buRbiI6IDQsDQogICAgIm7Eg20iOiA1LCAic8OhdSI6IDYsICJi4bqjeSI6IDcsICJ0w6FtIjogOCwgImNow61uIjogOSwgIm3GsOG7nWkiOiAxMCwNCiAgICAibcaw4budaSBt4buZdCI6IDExLCAibcaw4budaSBoYWkiOiAxMiwgImNo4bqhcCI6IDEyLA0KfQ0KDQojIE9yZGluYWwgdG8gbnVtYmVyDQpfT1JESU5BTCA9IHsNCiAgICAibmjhuqV0IjogMSwgImhhaSI6IDIsICJiYSI6IDMsICJi4buRbiI6IDQsICJ0xrAiOiA0LA0KICAgICJuxINtIjogNSwgInPDoXUiOiA2LCAiYuG6o3kiOiA3LCAidMOhbSI6IDgsICJjaMOtbiI6IDksICJtxrDhu51pIjogMTAsDQp9DQoNCg0KZGVmIF9tb25kYXlfb2Zfd2VlayhkOiBkYXRlKSAtPiBkYXRlOg0KICAgICIiIlJldHVybiB0aGUgTW9uZGF5IG9mIHRoZSB3ZWVrIGNvbnRhaW5pbmcgZCAod2VlayA9IE1vbuKAk1N1biBwZXIgcnVsZWJvb2spLiIiIg0KICAgIHJldHVybiBkIC0gdGltZWRlbHRhKGRheXM9ZC53ZWVrZGF5KCkpDQoNCg0KZGVmIF9tb250aF9yYW5nZSh5ZWFyOiBpbnQsIG1vbnRoOiBpbnQsIGFzX29mOiBkYXRlKSAtPiBUdXBsZVtkYXRlLCBkYXRlXToNCiAgICAiIiINCiAgICBSZXR1cm4gKGZpcnN0X2RheSwgbGFzdF9kYXlfb3JfYXNfb2YpIGZvciBhIGdpdmVuIG1vbnRoLg0KICAgIElmIHRoZSBtb250aCBpcyB0aGUgY3VycmVudCBtb250aCwgZW5kIGF0IGFzX29mX2RhdGUgKGluY2x1c2l2ZSkuDQogICAgIiIiDQogICAgZmlyc3QgPSBkYXRlKHllYXIsIG1vbnRoLCAxKQ0KICAgIGlmIG1vbnRoID09IDEyOg0KICAgICAgICBsYXN0ID0gZGF0ZSh5ZWFyICsgMSwgMSwgMSkgLSB0aW1lZGVsdGEoZGF5cz0xKQ0KICAgIGVsc2U6DQogICAgICAgIGxhc3QgPSBkYXRlKHllYXIsIG1vbnRoICsgMSwgMSkgLSB0aW1lZGVsdGEoZGF5cz0xKQ0KDQogICAgIyBJZiB0aGlzIGlzIHRoZSBjdXJyZW50IG1vbnRoLCBjYXAgYXQgYXNfb2ZfZGF0ZQ0KICAgIGlmIHllYXIgPT0gYXNfb2YueWVhciBhbmQgbW9udGggPT0gYXNfb2YubW9udGg6DQogICAgICAgIGxhc3QgPSBtaW4obGFzdCwgYXNfb2YpDQoNCiAgICByZXR1cm4gZmlyc3QsIGxhc3QNCg0KDQpkZWYgX3F1YXJ0ZXJfcmFuZ2UoeWVhcjogaW50LCBxdWFydGVyOiBpbnQsIGFzX29mOiBkYXRlKSAtPiBUdXBsZVtkYXRlLCBkYXRlXToNCiAgICAiIiJSZXR1cm4gKGZpcnN0X2RheSwgbGFzdF9kYXlfb3JfYXNfb2YpIGZvciBhIGdpdmVuIHF1YXJ0ZXIuIiIiDQogICAgc3RhcnRfbW9udGggPSAocXVhcnRlciAtIDEpICogMyArIDENCiAgICBlbmRfbW9udGggPSBzdGFydF9tb250aCArIDINCiAgICBmaXJzdCA9IGRhdGUoeWVhciwgc3RhcnRfbW9udGgsIDEpDQogICAgaWYgZW5kX21vbnRoID09IDEyOg0KICAgICAgICBsYXN0ID0gZGF0ZSh5ZWFyICsgMSwgMSwgMSkgLSB0aW1lZGVsdGEoZGF5cz0xKQ0KICAgIGVsc2U6DQogICAgICAgIGxhc3QgPSBkYXRlKHllYXIsIGVuZF9tb250aCArIDEsIDEpIC0gdGltZWRlbHRhKGRheXM9MSkNCg0KICAgIGlmIGZpcnN0IDw9IGFzX29mIDw9IGxhc3Q6DQogICAgICAgIGxhc3QgPSBtaW4obGFzdCwgYXNfb2YpDQogICAgcmV0dXJuIGZpcnN0LCBsYXN0DQoNCg0KZGVmIF95ZWFyX3JhbmdlKHllYXI6IGludCwgYXNfb2Y6IGRhdGUpIC0+IFR1cGxlW2RhdGUsIGRhdGVdOg0KICAgICIiIlJldHVybiAoZmlyc3RfZGF5LCBsYXN0X2RheV9vcl9hc19vZikgZm9yIGEgZ2l2ZW4geWVhci4iIiINCiAgICBmaXJzdCA9IGRhdGUoeWVhciwgMSwgMSkNCiAgICBsYXN0ID0gZGF0ZSh5ZWFyLCAxMiwgMzEpDQogICAgaWYgeWVhciA9PSBhc19vZi55ZWFyOg0KICAgICAgICBsYXN0ID0gbWluKGxhc3QsIGFzX29mKQ0KICAgIHJldHVybiBmaXJzdCwgbGFzdA0KDQoNCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tDQojIE1haW4gcmVzb2x2ZXINCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tDQoNCmNsYXNzIFRpbWVSZXNvbHZlcjoNCiAgICAiIiINCiAgICBSZXNvbHZlIFZpZXRuYW1lc2UgdGltZSBleHByZXNzaW9ucyB0byBkYXRlIHJhbmdlcy4NCg0KICAgIFVzYWdlOg0KICAgICAgICByZXNvbHZlciA9IFRpbWVSZXNvbHZlcihjYWxlbmRhcikNCiAgICAgICAgc3RhcnQsIGVuZCA9IHJlc29sdmVyLnJlc29sdmUoInRow6FuZyB0csaw4bubYyIsIGFzX29mX2RhdGUpDQogICAgIiIiDQoNCiAgICBkZWYgX19pbml0X18oc2VsZiwgY2FsZW5kYXI6IE9wdGlvbmFsW0x1bmFyQ2FsZW5kYXJdID0gTm9uZSk6DQogICAgICAgIHNlbGYuY2FsZW5kYXIgPSBjYWxlbmRhciBvciBMdW5hckNhbGVuZGFyKCkNCg0KICAgIGRlZiByZXNvbHZlKHNlbGYsIGV4cHJlc3Npb246IHN0ciwgYXNfb2Y6IGRhdGUpIC0+IE9wdGlvbmFsW1R1cGxlW2RhdGUsIGRhdGVdXToNCiAgICAgICAgIiIiDQogICAgICAgIFJlc29sdmUgYSBWaWV0bmFtZXNlIHRpbWUgZXhwcmVzc2lvbiB0byBhIChzdGFydF9kYXRlLCBlbmRfZGF0ZSkgcmFuZ2UuDQoNCiAgICAgICAgUGFyYW1ldGVycw0KICAgICAgICAtLS0tLS0tLS0tDQogICAgICAgIGV4cHJlc3Npb24gOiBzdHINCiAgICAgICAgICAgIFZpZXRuYW1lc2UgdGltZSBleHByZXNzaW9uIChlLmcuLCAidGjDoW5nIG7DoHkiLCAidHXhuqduIHLhu5NpIikuDQogICAgICAgIGFzX29mIDogZGF0ZQ0KICAgICAgICAgICAgVGhlIGFzLW9mIGRhdGUgZm9yIHJlc29sdmluZyByZWxhdGl2ZSBleHByZXNzaW9ucy4NCg0KICAgICAgICBSZXR1cm5zDQogICAgICAgIC0tLS0tLS0NCiAgICAgICAgT3B0aW9uYWxbVHVwbGVbZGF0ZSwgZGF0ZV1dDQogICAgICAgICAgICAoc3RhcnRfZGF0ZSwgZW5kX2RhdGUpIGluY2x1c2l2ZSwgb3IgTm9uZSBpZiBub3QgcmVjb2duaXplZC4NCiAgICAgICAgIiIiDQogICAgICAgIGV4cHIgPSBleHByZXNzaW9uLnN0cmlwKCkubG93ZXIoKQ0KDQogICAgICAgICMgLS0tIEV4YWN0IGRhdGUgcGF0dGVybnMgLS0tDQogICAgICAgICMgImjDtG0gbmF5Ig0KICAgICAgICBpZiBleHByIGluICgiaMO0bSBuYXkiLCAibmfDoHkgaMO0bSBuYXkiLCAidG9kYXkiKToNCiAgICAgICAgICAgIHJldHVybiAoYXNfb2YsIGFzX29mKQ0KDQogICAgICAgICMgImjDtG0gcXVhIg0KICAgICAgICBpZiBleHByIGluICgiaMO0bSBxdWEiLCAibmfDoHkgaMO0bSBxdWEiKToNCiAgICAgICAgICAgIHllc3RlcmRheSA9IGFzX29mIC0gdGltZWRlbHRhKGRheXM9MSkNCiAgICAgICAgICAgIHJldHVybiAoeWVzdGVyZGF5LCB5ZXN0ZXJkYXkpDQoNCiAgICAgICAgIyAtLS0gV2VlayBwYXR0ZXJucyAtLS0NCiAgICAgICAgaWYgcmUuc2VhcmNoKHIidHXhuqduXHMqKG7DoHl8bmF5KSIsIGV4cHIpOg0KICAgICAgICAgICAgbW9uZGF5ID0gX21vbmRheV9vZl93ZWVrKGFzX29mKQ0KICAgICAgICAgICAgcmV0dXJuIChtb25kYXksIGFzX29mKQ0KDQogICAgICAgIGlmIHJlLnNlYXJjaChyInR14bqnblxzKihy4buTaXx0csaw4bubY3xxdWF8duG7q2EgcuG7k2l8duG7q2EgcXVhKSIsIGV4cHIpOg0KICAgICAgICAgICAgcHJldl9tb25kYXkgPSBfbW9uZGF5X29mX3dlZWsoYXNfb2YpIC0gdGltZWRlbHRhKGRheXM9NykNCiAgICAgICAgICAgIHByZXZfc3VuZGF5ID0gcHJldl9tb25kYXkgKyB0aW1lZGVsdGEoZGF5cz02KQ0KICAgICAgICAgICAgcmV0dXJuIChwcmV2X21vbmRheSwgcHJldl9zdW5kYXkpDQoNCiAgICAgICAgIyAtLS0gTW9udGggcGF0dGVybnMgLS0tDQogICAgICAgIGlmIHJlLnNlYXJjaChyInRow6FuZ1xzKihuw6B5fG5heSkiLCBleHByKToNCiAgICAgICAgICAgIHJldHVybiBfbW9udGhfcmFuZ2UoYXNfb2YueWVhciwgYXNfb2YubW9udGgsIGFzX29mKQ0KDQogICAgICAgIGlmIHJlLnNlYXJjaChyInRow6FuZ1xzKihy4buTaXx0csaw4bubY3xxdWF8duG7q2EgcuG7k2l8duG7q2EgcXVhKSIsIGV4cHIpOg0KICAgICAgICAgICAgaWYgYXNfb2YubW9udGggPT0gMToNCiAgICAgICAgICAgICAgICByZXR1cm4gX21vbnRoX3JhbmdlKGFzX29mLnllYXIgLSAxLCAxMiwgYXNfb2YpDQogICAgICAgICAgICBlbHNlOg0KICAgICAgICAgICAgICAgIHJldHVybiBfbW9udGhfcmFuZ2UoYXNfb2YueWVhciwgYXNfb2YubW9udGggLSAxLCBhc19vZikNCg0KICAgICAgICAjICJ0aMOhbmcgTiIgb3IgInRow6FuZyA8bmFtZT4iDQogICAgICAgIG0gPSByZS5tYXRjaChyInRow6FuZ1xzKyhcZHsxLDJ9KSIsIGV4cHIpDQogICAgICAgIGlmIG06DQogICAgICAgICAgICBtb250aF9udW0gPSBpbnQobS5ncm91cCgxKSkNCiAgICAgICAgICAgIGlmIDEgPD0gbW9udGhfbnVtIDw9IDEyOg0KICAgICAgICAgICAgICAgIHJldHVybiBfbW9udGhfcmFuZ2UoYXNfb2YueWVhciwgbW9udGhfbnVtLCBhc19vZikNCg0KICAgICAgICBmb3IgbmFtZSwgbnVtIGluIF9NT05USF9OQU1FUy5pdGVtcygpOg0KICAgICAgICAgICAgaWYgZiJ0aMOhbmcge25hbWV9IiBpbiBleHByOg0KICAgICAgICAgICAgICAgIHJldHVybiBfbW9udGhfcmFuZ2UoYXNfb2YueWVhciwgbnVtLCBhc19vZikNCg0KICAgICAgICAjIC0tLSBRdWFydGVyIHBhdHRlcm5zIC0tLQ0KICAgICAgICBtID0gcmUubWF0Y2gociJxdcO9XHMrKFxkKSIsIGV4cHIpDQogICAgICAgIGlmIG06DQogICAgICAgICAgICBxID0gaW50KG0uZ3JvdXAoMSkpDQogICAgICAgICAgICBpZiAxIDw9IHEgPD0gNDoNCiAgICAgICAgICAgICAgICByZXR1cm4gX3F1YXJ0ZXJfcmFuZ2UoYXNfb2YueWVhciwgcSwgYXNfb2YpDQoNCiAgICAgICAgaWYgcmUuc2VhcmNoKHIicXXDvVxzKihuw6B5fG5heSkiLCBleHByKToNCiAgICAgICAgICAgIHEgPSAoYXNfb2YubW9udGggLSAxKSAvLyAzICsgMQ0KICAgICAgICAgICAgcmV0dXJuIF9xdWFydGVyX3JhbmdlKGFzX29mLnllYXIsIHEsIGFzX29mKQ0KDQogICAgICAgIGlmIHJlLnNlYXJjaChyInF1w71ccyoocuG7k2l8dHLGsOG7m2N8cXVhKSIsIGV4cHIpOg0KICAgICAgICAgICAgcSA9IChhc19vZi5tb250aCAtIDEpIC8vIDMgKyAxDQogICAgICAgICAgICBpZiBxID09IDE6DQogICAgICAgICAgICAgICAgcmV0dXJuIF9xdWFydGVyX3JhbmdlKGFzX29mLnllYXIgLSAxLCA0LCBhc19vZikNCiAgICAgICAgICAgIGVsc2U6DQogICAgICAgICAgICAgICAgcmV0dXJuIF9xdWFydGVyX3JhbmdlKGFzX29mLnllYXIsIHEgLSAxLCBhc19vZikNCg0KICAgICAgICAjIC0tLSAiY8O5bmcga+G7syBuxINtIG5nb8OhaSIgPSBzYW1lIHBlcmlvZCBsYXN0IHllYXIgLS0tDQogICAgICAgICMgKE11c3QgYmUgY2hlY2tlZCBiZWZvcmUgZ2VuZXJpYyAibsSDbSBuZ2/DoWkiIHBhdHRlcm4pDQogICAgICAgIGlmIHJlLnNlYXJjaChyImPDuW5nXHMqa+G7s1xzKihuxINtXHMqKT8obmdvw6FpfHRyxrDhu5tjfHLhu5NpKSIsIGV4cHIpOg0KICAgICAgICAgICAgIyBTYW1lIG1vbnRoLCBsYXN0IHllYXINCiAgICAgICAgICAgIHJldHVybiBfbW9udGhfcmFuZ2UoYXNfb2YueWVhciAtIDEsIGFzX29mLm1vbnRoLCBhc19vZikNCg0KICAgICAgICAjIC0tLSBZZWFyIHBhdHRlcm5zIC0tLQ0KICAgICAgICBpZiByZS5zZWFyY2gociJuxINtXHMqKG5heXxuw6B5KSIsIGV4cHIpOg0KICAgICAgICAgICAgcmV0dXJuIF95ZWFyX3JhbmdlKGFzX29mLnllYXIsIGFzX29mKQ0KDQogICAgICAgIGlmIHJlLnNlYXJjaChyIm7Eg21ccyoocuG7k2l8dHLGsOG7m2N8bmdvw6FpfHF1YSkiLCBleHByKToNCiAgICAgICAgICAgIHJldHVybiBfeWVhcl9yYW5nZShhc19vZi55ZWFyIC0gMSwgYXNfb2YpDQoNCiAgICAgICAgbSA9IHJlLm1hdGNoKHIibsSDbVxzKyhcZHs0fSkiLCBleHByKQ0KICAgICAgICBpZiBtOg0KICAgICAgICAgICAgeWVhciA9IGludChtLmdyb3VwKDEpKQ0KICAgICAgICAgICAgcmV0dXJuIF95ZWFyX3JhbmdlKHllYXIsIGFzX29mKQ0KDQogICAgICAgICMgLS0tIFThur90IC8gTHVuYXIgcGF0dGVybnMgLS0tDQogICAgICAgIGlmICJ04bq/dCIgaW4gZXhwcjoNCiAgICAgICAgICAgIHdpbmRvdyA9IHNlbGYuY2FsZW5kYXIuZmluZF90ZXRfd2luZG93KGFzX29mLnllYXIpDQogICAgICAgICAgICBpZiB3aW5kb3c6DQogICAgICAgICAgICAgICAgcmV0dXJuIHdpbmRvdw0KDQogICAgICAgICMgIm3DuW5nIE4iIChsdW5hciBuZXcgeWVhciBkYXkgTikNCiAgICAgICAgbSA9IHJlLnNlYXJjaChyIm3DuW5nXHMrKFxkezEsMn0pIiwgZXhwcikNCiAgICAgICAgaWYgbToNCiAgICAgICAgICAgIGRheSA9IGludChtLmdyb3VwKDEpKQ0KICAgICAgICAgICAgc29sYXIgPSBzZWxmLmNhbGVuZGFyLmdldF9zb2xhcl9kYXRlKGFzX29mLnllYXIsIDEsIGRheSkNCiAgICAgICAgICAgIGlmIHNvbGFyOg0KICAgICAgICAgICAgICAgIHJldHVybiAoc29sYXIsIHNvbGFyKQ0KDQogICAgICAgICMgLS0tIE4gZGF5cyBhZ28gLS0tDQogICAgICAgIG0gPSByZS5zZWFyY2gociIoXGQrKVxzKm5nw6B5XHMqKHRyxrDhu5tjfHF1YSkiLCBleHByKQ0KICAgICAgICBpZiBtOg0KICAgICAgICAgICAgbiA9IGludChtLmdyb3VwKDEpKQ0KICAgICAgICAgICAgc3RhcnQgPSBhc19vZiAtIHRpbWVkZWx0YShkYXlzPW4pDQogICAgICAgICAgICByZXR1cm4gKHN0YXJ0LCBhc19vZikNCg0KICAgICAgICAjIC0tLSBTcGVjaWZpYyBkYXRlIC0tLQ0KICAgICAgICBtID0gcmUuc2VhcmNoKHIibmfDoHlccysoXGR7MSwyfSlbL1wtLl0oXGR7MSwyfSkoPzpbL1wtLl0oXGR7Miw0fSkpPyIsIGV4cHIpDQogICAgICAgIGlmIG06DQogICAgICAgICAgICBkYXkgPSBpbnQobS5ncm91cCgxKSkNCiAgICAgICAgICAgIG1vbnRoID0gaW50KG0uZ3JvdXAoMikpDQogICAgICAgICAgICB5ZWFyID0gaW50KG0uZ3JvdXAoMykpIGlmIG0uZ3JvdXAoMykgZWxzZSBhc19vZi55ZWFyDQogICAgICAgICAgICBpZiB5ZWFyIDwgMTAwOg0KICAgICAgICAgICAgICAgIHllYXIgKz0gMjAwMA0KICAgICAgICAgICAgdHJ5Og0KICAgICAgICAgICAgICAgIGQgPSBkYXRlKHllYXIsIG1vbnRoLCBkYXkpDQogICAgICAgICAgICAgICAgcmV0dXJuIChkLCBkKQ0KICAgICAgICAgICAgZXhjZXB0IFZhbHVlRXJyb3I6DQogICAgICAgICAgICAgICAgcGFzcw0KDQogICAgICAgIGxvZ2dlci5kZWJ1ZygiQ291bGQgbm90IHJlc29sdmUgdGltZSBleHByZXNzaW9uOiAnJXMnIiwgZXhwcmVzc2lvbikNCiAgICAgICAgcmV0dXJuIE5vbmUNCg0KICAgIGRlZiBleHRyYWN0X2FuZF9yZXNvbHZlKHNlbGYsIHF1ZXN0aW9uOiBzdHIsIGFzX29mOiBkYXRlKSAtPiBMaXN0W1R1cGxlW3N0ciwgZGF0ZSwgZGF0ZV1dOg0KICAgICAgICAiIiINCiAgICAgICAgRXh0cmFjdCBhbGwgdGltZSBleHByZXNzaW9ucyBmcm9tIGEgcXVlc3Rpb24gYW5kIHJlc29sdmUgdGhlbS4NCg0KICAgICAgICBSZXR1cm5zIGEgbGlzdCBvZiAoZXhwcmVzc2lvbiwgc3RhcnRfZGF0ZSwgZW5kX2RhdGUpIHR1cGxlcy4NCiAgICAgICAgIiIiDQogICAgICAgIHJlc3VsdHMgPSBbXQ0KICAgICAgICBwYXR0ZXJucyA9IFsNCiAgICAgICAgICAgIHIiKHRow6FuZ1xzKyg/Om7DoHl8bmF5fHLhu5NpfHRyxrDhu5tjfHF1YXx24burYSBy4buTaXx24burYSBxdWF8XGR7MSwyfXxcdyspKSIsDQogICAgICAgICAgICByIih0deG6p25ccysoPzpuw6B5fG5heXxy4buTaXx0csaw4bubY3xxdWF8duG7q2EgcuG7k2l8duG7q2EgcXVhKSkiLA0KICAgICAgICAgICAgciIocXXDvVxzKyg/Om7DoHl8bmF5fHLhu5NpfHRyxrDhu5tjfHF1YXxcZCkpIiwNCiAgICAgICAgICAgIHIiKGPDuW5nXHMqa+G7s1xzKig/Om7Eg21ccyopPyg/Om5nb8OhaXx0csaw4bubY3xy4buTaSkpIiwNCiAgICAgICAgICAgIHIiKG7Eg21ccysoPzpuYXl8bsOgeXxy4buTaXx0csaw4bubY3xuZ2/DoWl8cXVhfFxkezR9KSkiLA0KICAgICAgICAgICAgciIoaMO0bVxzKyg/Om5heXxxdWEpKSIsDQogICAgICAgICAgICByIih04bq/dCkiLA0KICAgICAgICAgICAgciIoXGQrXHMqbmfDoHlccyooPzp0csaw4bubY3xxdWEpKSIsDQogICAgICAgIF0NCg0KICAgICAgICBmb3IgcGF0dGVybiBpbiBwYXR0ZXJuczoNCiAgICAgICAgICAgIGZvciBtIGluIHJlLmZpbmRpdGVyKHBhdHRlcm4sIHF1ZXN0aW9uLmxvd2VyKCkpOg0KICAgICAgICAgICAgICAgIGV4cHIgPSBtLmdyb3VwKDEpDQogICAgICAgICAgICAgICAgcmVzb2x2ZWQgPSBzZWxmLnJlc29sdmUoZXhwciwgYXNfb2YpDQogICAgICAgICAgICAgICAgaWYgcmVzb2x2ZWQ6DQogICAgICAgICAgICAgICAgICAgIHJlc3VsdHMuYXBwZW5kKChleHByLCByZXNvbHZlZFswXSwgcmVzb2x2ZWRbMV0pKQ0KDQogICAgICAgIHJldHVybiByZXN1bHRzDQo='
source_files['src/predict.py'] = '...'
source_files['prompts/templates/few_shot_examples.json'] = 'Ww0KICB7DQogICAgInF1ZXN0aW9uIjogIlThu5VuZyBkb2FuaCB0aHUgdGh14bqnbiB0cm9uZyB0aMOhbmcgbsOgeSB0w61uaCDEkeG6v24gdGjhu51pIMSRaeG7g20gaGnhu4duIHThuqFpIGzDoCBiYW8gbmhpw6p1PyIsDQogICAgImFzX29mX2RhdGUiOiAiMjAyNC0wNS0xOCIsDQogICAgImV4cGVjdGVkX3R5cGUiOiAibnVtYmVyIiwNCiAgICAiY29kZSI6ICIjIEzhu41jIMSRxqFuIGjDoG5nIGjhu6NwIGzhu4c6IGtow7RuZyBo4buneSwgdHJvbmcgdGjDoW5nIDUgdOG7qyBuZ8OgeSAxIMSR4bq/biBuZ8OgeSAxOFxub3JkZXJzX2RmWydvcmRlcl9kYXRlJ10gPSBwZC50b19kYXRldGltZShvcmRlcnNfZGZbJ29yZGVyX2RhdGUnXSlcbnZhbGlkX29yZGVycyA9IG9yZGVyc19kZltcbiAgICAob3JkZXJzX2RmWydzdGF0dXMnXSAhPSAnQ2FuY2VsbGVkJykgJlxuICAgIChvcmRlcnNfZGZbJ29yZGVyX2RhdGUnXSA+PSBwZC5UaW1lc3RhbXAoJzIwMjQtMDUtMDEnKSkgJlxuICAgIChvcmRlcnNfZGZbJ29yZGVyX2RhdGUnXSA8PSBwZC5UaW1lc3RhbXAoJzIwMjQtMDUtMTgnKSlcbl1cbiMgRG9hbmggdGh1IHRodeG6p24gPSBsaW5lX3RvdGFsIHNhdSBnaeG6o20gZ2nDoSwgbG/huqFpIHRy4burIHBow60gduG6rW4gY2h1eeG7g24gdsOgIHRodeG6v1xucmV2ZW51ZSA9IHZhbGlkX29yZGVyc1snc3VidG90YWwnXS5zdW0oKVxucmVzdWx0ID0gZmxvYXQocmV2ZW51ZSlcbmFuc3dlcl90eXBlID0gJ251bWJlciciLA0KICAgICJhbnN3ZXIiOiAxMjU0MDAwMDANCiAgfSwNCiAgew0KICAgICJxdWVzdGlvbiI6ICJT4bqjbiBwaOG6qW0gbsOgbyBiw6FuIGNo4bqheSBuaOG6pXQgdHJvbmcgcXXDvSAxIG7Eg20gMjAyND8iLA0KICAgICJhc19vZl9kYXRlIjogIjIwMjQtMDQtMDEiLA0KICAgICJleHBlY3RlZF90eXBlIjogInN0cmluZyIsDQogICAgImNvZGUiOiAiIyBCw6FuIGNo4bqheSBuaOG6pXQgPSB44bq/cCBo4bqhbmcgdGhlbyBz4buRIGzGsOG7o25nIChxdWFudGl0eSksIGtow7RuZyB0aGVvIGRvYW5oIHRodVxub3JkZXJzWydvcmRlcl9kYXRlJ10gPSBwZC50b19kYXRldGltZShvcmRlcnNbJ29yZGVyX2RhdGUnXSlcbnExX29yZGVycyA9IG9yZGVyc1tcbiAgICAob3JkZXJzWydvcmRlcl9kYXRlJ10gPj0gcGQuVGltZXN0YW1wKCcyMDI0LTAxLTAxJykpICZcbiAgICAob3JkZXJzWydvcmRlcl9kYXRlJ10gPD0gcGQuVGltZXN0YW1wKCcyMDI0LTAzLTMxJykpICZcbiAgICAob3JkZXJzWydzdGF0dXMnXSAhPSAnQ2FuY2VsbGVkJylcbl1cbm1lcmdlZCA9IHExX29yZGVycy5tZXJnZShvcmRlcl9pdGVtcywgb249J29yZGVyX2lkJylcbnRvcF9wcm9kID0gbWVyZ2VkLmdyb3VwYnkoJ3Byb2R1Y3RfbmFtZScpWydxdWFudGl0eSddLnN1bSgpLnJlc2V0X2luZGV4KClcbnRvcF9wcm9kID0gdG9wX3Byb2Quc29ydF92YWx1ZXMoYnk9J3F1YW50aXR5JywgYXNjZW5kaW5nPUZhbHNlKVxucmVzdWx0ID0gc3RyKHRvcF9wcm9kLmlsb2NbMF1bJ3Byb2R1Y3RfbmFtZSddKVxuYW5zd2VyX3R5cGUgPSAnc3RyaW5nJyIsDQogICAgImFuc3dlciI6ICJDw6AgcGjDqiBz4buvYSDEkcOhIg0KICB9LA0KICB7DQogICAgInF1ZXN0aW9uIjogIkRvYW5oIHRodSB0deG6p24gbsOgeSB0xINuZyBoYXkgZ2nhuqNtIGJhbyBuaGnDqnUgcGjhuqduIHRyxINtIHNvIHbhu5tpIHR14bqnbiB0csaw4bubYz8iLA0KICAgICJhc19vZl9kYXRlIjogIjIwMjQtMDMtMjQiLA0KICAgICJleHBlY3RlZF90eXBlIjogInBlcmNlbnQiLA0KICAgICJjb2RlIjogIiMgVHXhuqduIHTDrW5oIHThu6sgVGjhu6kgSGFpIMSR4bq/biBDaOG7pyBOaOG6rXQgdGhlbyBydWxlYm9va1xuIyBhc19vZl9kYXRlIDIwMjQtMDMtMjQgbMOgIENo4bunIE5o4bqtdFxub3JkZXJzWydvcmRlcl9kYXRlJ10gPSBwZC50b19kYXRldGltZShvcmRlcnNbJ29yZGVyX2RhdGUnXSlcbmN1cnJfd2VlayA9IG9yZGVyc1tcbiAgICAob3JkZXJzWydvcmRlcl9kYXRlJ10gPj0gcGQuVGltZXN0YW1wKCcyMDI0LTAzLTE4JykpICZcbiAgICAob3JkZXJzWydvcmRlcl9kYXRlJ10gPD0gcGQuVGltZXN0YW1wKCcyMDI0LTAzLTI0JykpICZcbiAgICAob3JkZXJzWydzdGF0dXMnXSAhPSAnQ2FuY2VsbGVkJylcbl1bJ3RvdGFsX2Ftb3VudCddLnN1bSgpXG5wcmV2X3dlZWsgPSBvcmRlcnNbXG4gICAgKG9yZGVyc1snb3JkZXJfZGF0ZSddID49IHBkLlRpbWVzdGFtcCgnMjAyNC0wMy0xMScpKSAmXG4gICAgKG9yZGVyc1snb3JkZXJfZGF0ZSddIDw9IHBkLlRpbWVzdGFtcCgnMjAyNC0wMy0xNycpKSAmXG4gICAgKG9yZGVyc1snc3RhdHVzJ10gIT0gJ0NhbmNlbGxlZCcpXG5dWyd0b3RhbF9hbW91bnQnXS5zdW0oKVxuaWYgcHJldl93ZWVrID09IDA6XG4gICAgcmVzdWx0ID0gMC4wXG5lbHNlOlxuICAgIHBjdF9jaGFuZ2UgPSAoKGN1cnJfd2VlayAtIHByZXZfd2VlaykgLyBwcmV2X3dlZWspICogMTAwLjBcbiAgICByZXN1bHQgPSByb3VuZChmbG9hdChwY3RfY2hhbmdlKSwgMilcbmFuc3dlcl90eXBlID0gJ3BlcmNlbnQnIiwNCiAgICAiYW5zd2VyIjogMTUuMzUNCiAgfSwNCiAgew0KICAgICJxdWVzdGlvbiI6ICJOZ8OgeSBuw6BvIGPDsyBkb2FuaCB0aHUgY2FvIG5o4bqldCB0cm9uZyB0aMOhbmcgMiBuxINtIDIwMjQ/IiwNCiAgICAiYXNfb2ZfZGF0ZSI6ICIyMDI0LTAzLTAxIiwNCiAgICAiZXhwZWN0ZWRfdHlwZSI6ICJkYXRlIiwNCiAgICAiY29kZSI6ICJvcmRlcnNbJ29yZGVyX2RhdGUnXSA9IHBkLnRvX2RhdGV0aW1lKG9yZGVyc1snb3JkZXJfZGF0ZSddKVxuZmViX29yZGVycyA9IG9yZGVyc1tcbiAgICAob3JkZXJzWydvcmRlcl9kYXRlJ10gPj0gcGQuVGltZXN0YW1wKCcyMDI0LTAyLTAxJykpICZcbiAgICAob3JkZXJzWydvcmRlcl9kYXRlJ10gPD0gcGQuVGltZXN0YW1wKCcyMDI0LTAyLTI5JykpICZcbiAgICAob3JkZXJzWydzdGF0dXMnXSAhPSAnQ2FuY2VsbGVkJylcbl1cbmRhaWx5X3JldiA9IGZlYl9vcmRlcnMuZ3JvdXBieSgnb3JkZXJfZGF0ZScpWydhbW91bnQnXS5zdW0oKS5yZXNldF9pbmRleCgpXG5iZXN0X2RheSA9IGRhaWx5X3Jldi5zb3J0X3ZhbHVlcyhieT0nYW1vdW50JywgYXNjZW5kaW5nPUZhbHNlKS5pbG9jWzBdWydvcmRlcl9kYXRlJ11cbnJlc3VsdCA9IGJlc3RfZGF5Lmlzb2Zvcm1hdCgpIGlmIGhhc2F0dHIoYmVzdF9kYXksICdpc29mb3JtYXQnKSBlbHNlIHN0cihiZXN0X2RheSlcbmFuc3dlcl90eXBlID0gJ2RhdGUnIiwNCiAgICAiYW5zd2VyIjogIjIwMjQtMDItMTQiDQogIH0sDQogIHsNCiAgICAicXVlc3Rpb24iOiAiQ8OzIGtow6FjaCBow6BuZyBuw6BvIG11YSBow6BuZyB0csOqbiA1MCB0cmnhu4d1IHRyb25nIHRow6FuZyBuw6B5IGtow7RuZz8iLA0KICAgICJhc19vZl9kYXRlIjogIjIwMjQtMDYtMjAiLA0KICAgICJleHBlY3RlZF90eXBlIjogImJvb2xlYW4iLA0KICAgICJjb2RlIjogIm9yZGVyc1snb3JkZXJfZGF0ZSddID0gcGQudG9fZGF0ZXRpbWUob3JkZXJzWydvcmRlcl9kYXRlJ10pXG5tb250aF9vcmRlcnMgPSBvcmRlcnNbXG4gICAgKG9yZGVyc1snb3JkZXJfZGF0ZSddID49IHBkLlRpbWVzdGFtcCgnMjAyNC0wNi0wMScpKSAmXG4gICAgKG9yZGVyc1snb3JkZXJfZGF0ZSddIDw9IHBkLlRpbWVzdGFtcCgnMjAyNC0wNi0yMCcpKSAmXG4gICAgKG9yZGVyc1snc3RhdHVzJ10gIT0gJ0NhbmNlbGxlZCcpXG5dXG5jdXN0X3NwZW5kaW5nID0gbW9udGhfb3JkZXJzLmdyb3VwYnkoJ2N1c3RvbWVyX2lkJylbJ3RvdGFsX2Ftb3VudCddLnN1bSgpXG5yZXN1bHQgPSBib29sKChjdXN0X3NwZW5kaW5nID4gNTAwMDAwMDApLmFueSgpKVxuYW5zd2VyX3R5cGUgPSAnYm9vbGVhbiciLA0KICAgICJhbnN3ZXIiOiB0cnVlDQogIH0sDQogIHsNCiAgICAicXVlc3Rpb24iOiAiTGnhu4d0IGvDqiBkYW5oIHPDoWNoIHRvcCAzIGNoaSBuaMOhbmggY8OzIGzGsOG7o25nIMSRxqFuIGjDoG5nIG5oaeG7gXUgbmjhuqV0IHRyb25nIG7Eg20gMjAyMy4iLA0KICAgICJhc19vZl9kYXRlIjogIjIwMjQtMDEtMDEiLA0KICAgICJleHBlY3RlZF90eXBlIjogImxpc3QiLA0KICAgICJjb2RlIjogIm9yZGVyc1snb3JkZXJfZGF0ZSddID0gcGQudG9fZGF0ZXRpbWUob3JkZXJzWydvcmRlcl9kYXRlJ10pXG55MjAyM19vcmRlcnMgPSBvcmRlcnNbXG4gICAgKG9yZGVyc1snb3JkZXJfZGF0ZSddID49IHBkLlRpbWVzdGFtcCgnMjAyMy0wMS0wMScpKSAmXG4gICAgKG9yZGVyc1snb3JkZXJfZGF0ZSddIDw9IHBkLlRpbWVzdGFtcCgnMjAyMy0xMi0zMScpKSAmXG4gICAgKG9yZGVyc1snc3RhdHVzJ10gIT0gJ0NhbmNlbGxlZCcpXG5dXG50b3BfYnJhbmNoZXMgPSB5MjAyM19vcmRlcnMuZ3JvdXBieSgnYnJhbmNoX25hbWUnKVsnb3JkZXJfaWQnXS5udW5pcXVlKCkucmVzZXRfaW5kZXgoKVxudG9wX2JyYW5jaGVzID0gdG9wX2JyYW5jaGVzLnNvcnRfdmFsdWVzKGJ5PSdvcmRlcl9pZCcsIGFzY2VuZGluZz1GYWxzZSkuaGVhZCgzKVxucmVzdWx0ID0gdG9wX2JyYW5jaGVzWydicmFuY2hfbmFtZSddLnRvbGlzdCgpXG5hbnN3ZXJfdHlwZSA9ICdsaXN0JyIsDQogICAgImFuc3dlciI6IFsNCiAgICAgICJDaGkgbmjDoW5oIFF14bqtbiAxIiwNCiAgICAgICJDaGkgbmjDoW5oIEJhIMSQw6xuaCIsDQogICAgICAiQ2hpIG5ow6FuaCBI4bqjaSBDaMOidSINCiAgICBdDQogIH0sDQogIHsNCiAgICAicXVlc3Rpb24iOiAiQuG6o25nIHThu5VuZyBo4bujcCBkb2FuaCB0aHUgdsOgIHPhu5EgxJHGoW4gaMOgbmcgdGhlbyB04burbmcgdGjDoW5nIHRyb25nIG7Eg20gMjAyMy4iLA0KICAgICJhc19vZl9kYXRlIjogIjIwMjQtMDEtMDEiLA0KICAgICJleHBlY3RlZF90eXBlIjogInRhYmxlIiwNCiAgICAiY29kZSI6ICJvcmRlcnNbJ29yZGVyX2RhdGUnXSA9IHBkLnRvX2RhdGV0aW1lKG9yZGVyc1snb3JkZXJfZGF0ZSddKVxueTIwMjNfb3JkZXJzID0gb3JkZXJzW1xuICAgIChvcmRlcnNbJ29yZGVyX2RhdGUnXS5kdC55ZWFyID09IDIwMjMpICZcbiAgICAob3JkZXJzWydzdGF0dXMnXSAhPSAnQ2FuY2VsbGVkJylcbl0uY29weSgpXG55MjAyM19vcmRlcnNbJ21vbnRoJ10gPSB5MjAyM19vcmRlcnNbJ29yZGVyX2RhdGUnXS5kdC5zdHJmdGltZSgnJVktJW0nKVxuc3VtbWFyeSA9IHkyMDIzX29yZGVycy5ncm91cGJ5KCdtb250aCcpLmFnZyhcbiAgICBkb2FuaF90aHU9KCdhbW91bnQnLCAnc3VtJyksXG4gICAgc29fZG9uPSgnb3JkZXJfaWQnLCAnY291bnQnKVxuKS5yZXNldF9pbmRleCgpXG5yZXN1bHQgPSB7XG4gICAgJ2NvbHVtbnMnOiBbJ21vbnRoJywgJ2RvYW5oX3RodScsICdzb19kb24nXSxcbiAgICAncm93cyc6IHN1bW1hcnkudmFsdWVzLnRvbGlzdCgpXG59XG5hbnN3ZXJfdHlwZSA9ICd0YWJsZSciLA0KICAgICJhbnN3ZXIiOiB7DQogICAgICAiY29sdW1ucyI6IFsNCiAgICAgICAgIm1vbnRoIiwNCiAgICAgICAgImRvYW5oX3RodSIsDQogICAgICAgICJzb19kb24iDQogICAgICBdLA0KICAgICAgInJvd3MiOiBbDQogICAgICAgIFsNCiAgICAgICAgICAiMjAyMy0wMSIsDQogICAgICAgICAgMTAwMDAwMDAsDQogICAgICAgICAgNTANCiAgICAgICAgXSwNCiAgICAgICAgWw0KICAgICAgICAgICIyMDIzLTAyIiwNCiAgICAgICAgICAxMjAwMDAwMCwNCiAgICAgICAgICA2MA0KICAgICAgICBdDQogICAgICBdDQogICAgfQ0KICB9LA0KICB7DQogICAgInF1ZXN0aW9uIjogIsSQ4buZIHR14buVaSB0cnVuZyBiw6xuaCBj4bunYSBraMOhY2ggaMOgbmcgxJHEg25nIGvDvSB0aMOgbmggdmnDqm4gbMOgIGJhbyBuaGnDqnU/IiwNCiAgICAiYXNfb2ZfZGF0ZSI6ICIyMDI0LTA1LTAxIiwNCiAgICAiZXhwZWN0ZWRfdHlwZSI6ICJ1bmFuc3dlcmFibGUiLA0KICAgICJjb2RlIjogIiMgS2nhu4NtIHRyYSBjw6FjIGLhuqNuZyBk4buvIGxp4buHdToga2jDtG5nIGPDsyBj4buZdCB0deG7lWkgKGFnZSkgaGF5IG5nw6B5IHNpbmggKGJpcnRoZGF5L2RvYilcbnJlc3VsdCA9IE5vbmVcbmFuc3dlcl90eXBlID0gJ3VuYW5zd2VyYWJsZSciLA0KICAgICJhbnN3ZXIiOiBudWxsDQogIH0NCl0NCg=='
source_files['prompts/templates/system_prompt.txt'] = 'WW91IGFyZSBhbiBleHBlcnQgVmlldG5hbWVzZSBkYXRhIGFuYWx5c3QgYW5kIFB5dGhvbiBwcm9ncmFtbWVyLg0KWW91ciB0YXNrIGlzIHRvIGFuc3dlciBhIGJ1c2luZXNzIHF1ZXN0aW9uIGJ5IHdyaXRpbmcgZXhlY3V0YWJsZSBQeXRob24gcGFuZGFzIGNvZGUgb3ZlciBzdHJ1Y3R1cmVkIGRhdGEuDQoNCklNUE9SVEFOVCBFWEVDVVRJT04gUlVMRVM6DQoxLiBUaGUgYnVzaW5lc3MgZGF0YSB0YWJsZXMgYXJlIGFscmVhZHkgbG9hZGVkIGludG8gcGFuZGFzIERhdGFGcmFtZXMgaW4geW91ciBuYW1lc3BhY2UuIFVzZSB0aGUgZXhhY3QgRGF0YUZyYW1lIHZhcmlhYmxlcyBwcm92aWRlZCBpbiB0aGUgcHJvbXB0Lg0KMi4gV3JpdGUgY29uY2lzZSwgY29ycmVjdCBQeXRob24gY29kZSB0aGF0IGNvbXB1dGVzIHRoZSBhbnN3ZXIgYW5kIGFzc2lnbnMgdGhlIGZpbmFsIHZhbHVlIHRvIGByZXN1bHRgLg0KMy4gU2V0IGBhbnN3ZXJfdHlwZWAgdG8gb25lIG9mIHRoZSBmb2xsb3dpbmcgZXhhY3QgdHlwZXM6DQogICAtICJudW1iZXIiOiBTaW5nbGUgaW50ZWdlciBvciBmbG9hdCAoZS5nLiwgMTUwMDAwMCBvciAxMi4zNCkuDQogICAtICJwZXJjZW50IjogU2luZ2xlIGZsb2F0IChlLmcuLCAyNS41IGZvciAyNS41JSwgbm90IDAuMjU1KS4NCiAgIC0gInN0cmluZyI6IEV4YWN0IHRleHQgb3IgbmFtZSBhcyBpdCBhcHBlYXJzIGluIHRoZSBkYXRhLCBwcmVzZXJ2ZWQgaW4gVW5pY29kZSBORkMuDQogICAtICJkYXRlIjogSVNPIGZvcm1hdCBzdHJpbmcgIllZWVktTU0tREQiIG9yICJZWVlZLU1NIi4NCiAgIC0gImJvb2xlYW4iOiBQeXRob24gVHJ1ZSBvciBGYWxzZS4NCiAgIC0gImxpc3QiOiBQeXRob24gbGlzdCBvZiBpdGVtcyBpbiB0aGUgcmVxdWVzdGVkIG9yZGVyIChlLmcuLCBbIlPhuqNuIHBo4bqpbSBBIiwgIlPhuqNuIHBo4bqpbSBCIl0pLg0KICAgLSAidGFibGUiOiBEaWN0IHdpdGgga2V5cyB7ImNvbHVtbnMiOiBbLi4uXSwgInJvd3MiOiBbWy4uLl0sIC4uLl19Lg0KICAgLSAidW5hbnN3ZXJhYmxlIjogVXNlIE9OTFkgd2hlbiB0aGUgcXVlc3Rpb24gYXNrcyBhYm91dCBpbmZvcm1hdGlvbiBjb21wbGV0ZWx5IGFic2VudCBmcm9tIHRoZSBkYXRhLg0KNC4gQnVzaW5lc3MgUnVsZWJvb2sgY29tcGxpYW5jZToNCiAgIC0gUmV2ZW51ZSA9IHN1bSBsaW5lIGFtb3VudHMgYWZ0ZXIgZGlzY291bnRzLCBFWENMVURJTkcgVkFULCBkZWxpdmVyeSBmZWVzLCBhbmQgQ0FOQ0VMTEVEIG9yZGVycy4NCiAgIC0gIkLDoW4gY2jhuqF5IG5o4bqldCIgLyAiVG9wIHPhuqNuIHBo4bqpbSIgPSByYW5rZWQgYnkgdG90YWwgUVVBTlRJVFkgc29sZCwgbm90IHJldmVudWUsIHVubGVzcyBzcGVjaWZpY2FsbHkgc3BlY2lmaWVkLg0KICAgLSBXZWVrID0gTW9uZGF5IHRvIFN1bmRheS4NCiAgIC0gIlRow6FuZyBuw6B5IiAvICJOxINtIG5heSIgPSBmcm9tIHN0YXJ0IG9mIG1vbnRoL3llYXIgdXAgdG8gYW5kIGluY2x1ZGluZyBgYXNfb2ZfZGF0ZWAuDQogICAtIFNhbWUgcGVyaW9kIGxhc3QgeWVhciAoImPDuW5nIGvhu7MgbsSDbSBuZ2/DoWkiKSA9IGV4YWN0IG1hdGNoaW5nIGRhdGUgaW50ZXJ2YWwgc2hpZnRlZCBiYWNrIDEgeWVhci4NCjUuIERlZmVuc2l2ZSBjb2Rpbmc6DQogICAtIENoZWNrIGlmIGNvbHVtbnMgZXhpc3Q7IGlmIG1pc3Npbmcgb3IgYW1iaWd1b3VzLCB2ZXJpZnkgc2NoZW1hIGZpcnN0Lg0KICAgLSBIYW5kbGUgcG9zc2libGUgTmFOL251bGwgdmFsdWVzIHdpdGggYC5maWxsbmEoKWAgb3IgYC5kcm9wbmEoKWAuDQogICAtIFByZXZlbnQgWmVyb0RpdmlzaW9uRXJyb3IgYnkgY2hlY2tpbmcgZGVub21pbmF0b3JzIGJlZm9yZSBkaXZpc2lvbi4NCiAgIC0gRm9yIHN0cmluZyBzZWFyY2hlcyBvciBmaWx0ZXJzLCB1c2UgY2FzZS1pbnNlbnNpdGl2ZSBtYXRjaGluZyBvciBub3JtYWxpemVkIGNvbXBhcmlzb24gd2hlcmUgYXBwcm9wcmlhdGUuDQo2LiBEbyBOT1QgcHJpbnQgdGhlIGFuc3dlciB3aXRoIGBwcmludCgpYC4gSnVzdCBhc3NpZ24gYHJlc3VsdCA9IC4uLmAgYW5kIGBhbnN3ZXJfdHlwZSA9IC4uLmAuDQo3LiBEbyBOT1Qgd3JpdGUgYW55IGltcG9ydCBzdGF0ZW1lbnRzLiBgcGRgLCBgbnBgLCBgbWF0aGAsIGByZWAsIGBkYXRldGltZWAsIGBkdGAsIGBkYXRlYCBhbmQgYHRpbWVkZWx0YWAgYXJlIGFscmVhZHkgYXZhaWxhYmxlLg0KOC4gRGVmaW5lIGV2ZXJ5IHZhcmlhYmxlIGJlZm9yZSB1c2luZyBpdDsgdGhlIGNvZGUgbXVzdCBiZSBjb21wbGV0ZSBhbmQgc2VsZi1jb250YWluZWQuDQo='

for relative_path, encoded in source_files.items():
    destination = SOURCE_ROOT / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(base64.b64decode(encoded))

sys.path.insert(0, str(SOURCE_ROOT))
os.environ['VLLM_TP'] = '2'
print('Embedded source files:', len(source_files))
print('Source root:', SOURCE_ROOT)
print('VLLM_TP:', os.environ['VLLM_TP'])


In [ ]:
from pathlib import Path
import os

INPUT_ROOT = Path('/kaggle/input')
SPLIT = 'public_test'
LIMIT = 3
MODEL_DIR = 'Qwen/Qwen2.5-Coder-3B-Instruct'
TENSOR_PARALLEL_SIZE = 2
OUTPUT_DIR = Path('/kaggle/working/result')
DATA_DIR = None

def find_data(root):
    candidates = []
    for current_root, dirs, files in os.walk(root):
        dirs[:] = [name for name in dirs if not name.startswith('.')]
        current = Path(current_root)
        if {'questions.jsonl', 'calendar.csv', 'rulebook.md'}.issubset(files) and (current / 'data').is_dir():
            candidates.append(current)
    if DATA_DIR is not None:
        return Path(DATA_DIR).resolve()
    preferred = [path for path in candidates if path.name == SPLIT or SPLIT in path.parts]
    if len(preferred) != 1:
        raise RuntimeError(f'Found {len(preferred)} {SPLIT} data directories; set DATA_DIR manually: {candidates}')
    return preferred[0].resolve()

data_path = find_data(INPUT_ROOT)
print('Data:', data_path)
print('Output:', OUTPUT_DIR)


In [ ]:
import json
import torch
from vllm import LLM, SamplingParams
from src.predict import DataMatePipeline

questions = [json.loads(line) for line in (data_path / 'questions.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()]
questions.sort(key=lambda question: question['business_id'])
if LIMIT is not None:
    questions = questions[:LIMIT]
if not questions:
    raise ValueError('No questions selected')
if torch.cuda.device_count() < TENSOR_PARALLEL_SIZE:
    raise RuntimeError(f'Need {TENSOR_PARALLEL_SIZE} GPUs, found {torch.cuda.device_count()}')

llm = LLM(model=MODEL_DIR, tensor_parallel_size=TENSOR_PARALLEL_SIZE, trust_remote_code=True, max_model_len=8192, gpu_memory_utilization=0.85, dtype='float16')
tokenizer = llm.get_tokenizer()
sampling_params = SamplingParams(temperature=0.0, max_tokens=1024, stop=['```\n', '\n\n\n'], include_stop_str_in_output=True)

def generate(prompt):
    if getattr(tokenizer, 'chat_template', None):
        prompt = tokenizer.apply_chat_template([{'role': 'user', 'content': prompt}], tokenize=False, add_generation_prompt=True)
    return llm.generate([prompt], sampling_params, use_tqdm=False)[0].outputs[0].text

pipeline = DataMatePipeline(data_dir=str(data_path), output_dir=str(OUTPUT_DIR), model_dir=MODEL_DIR, max_retries=2, confidence_threshold=0.3, confidence_runs=4, llm_generate_fn=generate)
for index, question in enumerate(questions, 1):
    answer = pipeline.process_question(question)
    print(f'[{index}/{len(questions)}] {question["id"]}:', json.dumps(answer, ensure_ascii=False))
print('Submission:', pipeline.submission_path)
